# Part 1 — Load Dataset

Loads the real **SMARD German day-ahead electricity price** CSV dataset.

---

### Key Data Handling Considerations

The file requires specific preprocessing due to three key format quirks:

1. **Delimiter:** Uses `;` as the separator instead of standard `,`.
2. **Header:** Contains a title row at the top that must be skipped.
3. **Number Formatting:** Uses European formatting where `,` is the thousands separator (e.g., `"45,471.50"` $\rightarrow$ `45471.50`).

---

### Excluded Columns

We explicitly exclude the 4 balancing-market columns:
* `Volume(+)`
* `Volume(-)`
* `Balancing Price`
* `Net Income`

> **Note on Data Leakage:** These columns are not available for day-ahead forecasting. Furthermore, $\text{Net Income} = \text{Volume} \times \text{Price}$, creating a circular relationship with the target variable.


# Setup — Libraries, Configuration & Data Path

This cell consolidates **all libraries** used throughout the notebook, sets up the
Plotly renderer for inline display, and auto-detects the data location so the
notebook runs **both locally and on Kaggle** without edits.

Run this cell first, then run the parts in order.

In [12]:
# ═══════════════════════════════════════════════════════════════════════════
#  CONSOLIDATED IMPORTS — run this cell first
# ═══════════════════════════════════════════════════════════════════════════
import os
import json
import warnings
import numpy as np
import pandas as pd

# Scientific / ML
from IPython.display import display
from scipy import stats
from scipy.special import gammaln
from scipy.stats import pearsonr, spearmanr
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import ElasticNet, Lasso, LinearRegression
from sklearn.feature_selection import mutual_info_regression
from sklearn.inspection import permutation_importance
import webbrowser
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.feature_selection import (
    RFE,
    SequentialFeatureSelector,
    f_regression,
    mutual_info_regression,
)
from sklearn.inspection import permutation_importance
from sklearn.linear_model import ElasticNet, Lasso, LinearRegression
from sklearn.preprocessing import StandardScaler
import plotly.io as pio
pio.renderers.default = "iframe"
# -- Paths -------------------------------------------------------------------
# One writable working directory holds every generated file (.pkl / .npy /
# .json and the Plotly .html dashboards), because this pipeline reads and
# writes through the same DATA_DIR / DOCS_DIR names. Read-only sources are
# linked into it, so os.path.join(DATA_DIR, ...) resolves for BOTH.
#
#   Kaggle : /kaggle/working, with the attached datasets linked in
#   local  : $EPF_OUT, else <repo>/outputs, with data/ files/ figures/ src/ linked in
import sys

ON_KAGGLE = os.path.isdir('/kaggle/input')

def _repo_root():
    here = os.path.abspath(os.getcwd())
    while True:
        if os.path.isdir(os.path.join(here, '.git')):
            return here
        up = os.path.dirname(here)
        if up == here:
            return os.path.abspath(os.getcwd())
        here = up

if ON_KAGGLE:
    WORK_DIR = '/kaggle/working'
    # Kaggle's mount layout has changed over time: attached datasets appear at
    # /kaggle/input/<slug> on older images and /kaggle/input/datasets/<owner>/<slug>
    # on current ones, so discover the dirs that hold files instead of guessing.
    SOURCE_DIRS = sorted({r for r, _sub, f in os.walk('/kaggle/input') if f})
else:
    _ROOT = _repo_root()
    WORK_DIR = os.environ.get('EPF_OUT') or os.path.join(_ROOT, 'outputs')
    SOURCE_DIRS = [os.path.join(_ROOT, d) for d in ('data', 'files', 'figures', 'src')]
    SOURCE_DIRS = [d for d in SOURCE_DIRS if os.path.isdir(d)]

os.makedirs(WORK_DIR, exist_ok=True)
for _d in SOURCE_DIRS:
    if _d not in sys.path:
        sys.path.append(_d)
    for _f in os.listdir(_d):
        _s, _dst = os.path.join(_d, _f), os.path.join(WORK_DIR, _f)
        if os.path.isfile(_s) and not os.path.exists(_dst):
            os.symlink(_s, _dst)

DATA_DIR = DOCS_DIR = WORK_DIR
from german_epf_research import (
    BSSM,
    DDNN,
    VIDDNN,
    ConformalWrapper,
    EvDNN,
    compute_all_metrics,
    inv_signed_log,
    signed_log,
)


# Hyperparameter optimization
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

# Plotting (Plotly) — auto-pick a renderer that works on the current platform
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
# ── Renderer control ────────────────────────────────────────────────────────
# Set GITHUB_STATIC = True and re-run all cells BEFORE committing, so plots save
# as static images that GitHub displays. Keep it False for interactive plots
# locally / on Kaggle. (Static mode needs: pip install kaleido)
GITHUB_STATIC = False
def _pick_renderer():
    if GITHUB_STATIC:
        return 'svg'             # static images GitHub can render in the notebook
    if 'KAGGLE_KERNEL_RUN_TYPE' in os.environ:
        return 'kaggle'          # Kaggle notebooks
    try:
        import google.colab      # noqa
        return 'colab'
    except Exception:
        pass
    return 'notebook'            # classic Jupyter / most local setups
pio.renderers.default = _pick_renderer()

import sys
for _p in SOURCE_DIRS + ['.']:
    if _p not in sys.path:
        sys.path.append(_p)

# Project toolbox (models + helpers)
from german_epf_research import (
    DDNN, EvDNN, VIDDNN, BSSM,
    StudentTDNN_Adam, interval_metrics,
    signed_log, inv_signed_log, compute_all_metrics,
)

warnings.filterwarnings('ignore')
np.random.seed(42)

# -- Data location -----------------------------------------------------------
def data_dir():
    """Everything lives in the one working directory (see Paths above)."""
    return DATA_DIR

def dpath(fname):
    """Build a path to a data file inside the working directory."""
    p = os.path.join(DATA_DIR, fname)
    return p if os.path.exists(p) else fname

print(f'Setup complete. Working directory: {WORK_DIR}')
print(f'  sources linked in: {SOURCE_DIRS}')


Setup complete. Data directory: <working dir>


In [6]:
def parse_european_number(s):
    """
    Convert a European-format number string to a float.
    "45,471.50" -> 45471.50   (comma removed = thousands separator)
    "-19.00"    -> -19.00
    """
    if pd.isna(s):
        return np.nan
    return pd.to_numeric(str(s).replace(',', ''), errors='coerce')


def load_dataset(path):
    """Load and clean the SMARD CSV. Returns a tidy DataFrame."""
    print("Loading SMARD dataset...")

    # sep=';'      -> semicolon separated
    # skiprows=1   -> skip the title row at the very top
    df = pd.read_csv(path, sep=';', skiprows=1, encoding='utf-8')

    # --- Rename the long German column names to short, easy names ---
    rename_map = {
        'Germany/Luxembourg [€/MWh] Calculated resolutions':  'price',
        'grid load [MWh] Calculated resolutions':             'load_forecast',
        'Residual load [MWh] Calculated resolutions':         'residual_load',
        'Total [MWh] Calculated resolutions':                 'total_gen',
        'Photovoltaics and wind [MWh] Calculated resolutions':'pv_wind',
        'Wind offshore [MWh] Calculated resolutions':         'wind_offshore',
        'Wind onshore [MWh] Calculated resolutions':          'wind_onshore',
        'Photovoltaics [MWh] Calculated resolutions':         'solar',
        'Other [MWh] Calculated resolutions':                 'other_gen',
    }
    df = df.rename(columns=rename_map)

    # --- Drop the 4 balancing-market columns (user instruction) ---
    balancing_cols = [
        'Volume (+) [MWh] Calculated resolutions',
        'Volume (-) [MWh] Calculated resolutions',
        'Price [€/MWh] Calculated resolutions',
        'Net income [€] Calculated resolutions',
    ]
    df = df.drop(columns=[c for c in balancing_cols if c in df.columns])
    print(f"  Dropped balancing columns: Volume(+), Volume(-), Price, Net income")

    # --- Convert European number format to real numbers ---
    numeric_cols = ['price', 'load_forecast', 'residual_load', 'total_gen',
                    'pv_wind', 'wind_offshore', 'wind_onshore', 'solar', 'other_gen']
    for col in numeric_cols:
        if col in df.columns:
            df[col] = df[col].apply(parse_european_number)

    # --- Parse the datetime column ---
    # Format example: "Jan 1, 2020 12:00 AM"
    df['datetime'] = pd.to_datetime(df['Start date'],
                                    format='%b %d, %Y %I:%M %p',
                                    errors='coerce')

    # Drop rows where datetime failed, sort chronologically
    df = df.dropna(subset=['datetime']).sort_values('datetime').reset_index(drop=True)
    df = df.drop(columns=['Start date', 'End date'])

    print(f"  Loaded {len(df):,} hourly rows")
    print(f"  Date range: {df['datetime'].iloc[0].date()} to {df['datetime'].iloc[-1].date()}")
    print(f"  Price range: €{df['price'].min():.1f} to €{df['price'].max():.1f}")
    return df


# ── Run this part ──────────────────────────────────────────────────────────
if __name__ == "__main__":
    PATH = os.path.join(DATA_DIR, "Day-ahead_prices_202001010000_202605010000_Hour.csv")
    df = load_dataset(PATH)

    # Show the first few rows so you can see what you've got
    print("\nFirst 5 rows:")
    print(df[['datetime', 'price', 'load_forecast', 'wind_onshore', 'solar']].head())

    # Save for the next part
    df.to_pickle(os.path.join(DATA_DIR, "data_part1.pkl"))
    print("\n✓ Saved to data_part1.pkl — ready for Part 2")

Loading SMARD dataset...
  Dropped balancing columns: Volume(+), Volume(-), Price, Net income
  Loaded 55,487 hourly rows
  Date range: 2020-01-01 to 2026-04-30
  Price range: €-500.0 to €936.3

First 5 rows:
             datetime  price  load_forecast  wind_onshore  solar
0 2020-01-01 00:00:00  41.88       45471.50       5781.00    0.0
1 2020-01-01 01:00:00  38.60       43471.50       5859.00    0.0
2 2020-01-01 02:00:00  36.55       42555.50       5987.25    0.0
3 2020-01-01 03:00:00  32.32       42448.25       6058.75    0.0
4 2020-01-01 04:00:00  30.85       42567.75       5973.25    0.0

✓ Saved to data_part1.pkl — ready for Part 2


# Part 2 — Preprocessing & Feature Engineering

Transforms raw columns into predictive features for the model.

---

### Preprocessing & Feature Generation Plan

Generates 35 candidate features to capture temporal dependencies, market momentum, supply-demand dynamics, and seasonal patterns across multiple time horizons.

---

### Candidate Feature Categories

* **Lags & Price Deltas:** Past price values across 8 discrete lag intervals ($t-1, t-2, \dots$) plus 1-hour price momentum ($\Delta p_t = p_t - p_{t-1}$).
* **Rolling Statistics:** Moving window averages and volatility metrics, including 24-hour mean, 24-hour standard deviation, and 168-hour (weekly) mean.
* **Exponential Moving Averages (EMA):** Short-, medium-, and long-term trend indicators capturing price velocity over 6-hour, 24-hour, and 72-hour windows.
* **Supply & Demand Metrics:** Operational features including solar and wind generation ramp rates ($\Delta \text{Generation}_t$) and the demand-to-renewable generation ratio ($\frac{\text{Demand}}{\text{Renewables}}$).
* **Cyclical Calendar Encodings:** Sine and cosine trigonometric transformations applied to hour of day (0–23), day of week (0–6), and month of year (1–12) to model periodicities continuously.
---

### Prevention of Data Leakage

> **Critical Rule:** We only utilize values known prior to the forecast horizon—specifically forecasts and historical prices. The actual target price for the target hour is never included as an input.

In [7]:
"""
═══════════════════════════════════════════════════════════════════════════
 PART 2 — PREPROCESSING & ADVANCED FEATURE ENGINEERING (35 FEATURES)
═══════════════════════════════════════════════════════════════════════════
 Generates 35 candidate features:
   1. Lags & Price Deltas (8 Lags + 1-hour momentum)
   2. Rolling Statistics (24h Mean/Std, 168h Mean)
   3. Exponential Moving Averages (EMA 6h, 24h, 72h)
   4. Supply / Demand Ratios & Ramp Rates (Solar/Wind Ramps, Demand-Renewable Ratio)
   5. Cyclic Calendar Encodings (Hour, Day, Month sin/cos)
═══════════════════════════════════════════════════════════════════════════
"""



def engineer_features(df):
    """Creates all 35 model candidate features."""
    print("Engineering advanced features...")

    # Fill missing values
    num_cols = df.select_dtypes(include=np.number).columns
    df[num_cols] = df[num_cols].ffill().bfill().fillna(0)

    # ── 1. LAG FEATURES ──────────────────────────────────────────────────────
    for lag in [1, 2, 3, 6, 12, 24, 48, 168]:
        df[f"lag_{lag}"] = df["price"].shift(lag)

    # ── 2. ROLLING STATISTICS ────────────────────────────────────────────────
    df["roll_mean_24"] = df["price"].shift(1).rolling(24).mean()
    df["roll_std_24"] = df["price"].shift(1).rolling(24).std()
    df["roll_mean_168"] = df["price"].shift(1).rolling(168).mean()
    df["price_change"] = df["price"].shift(1) - df["price"].shift(25)

    # ── 3. EXPONENTIAL MOVING AVERAGES (EMA) ─────────────────────────────────
    df["ema_6h"] = df["price"].shift(1).ewm(span=6).mean()
    df["ema_24h"] = df["price"].shift(1).ewm(span=24).mean()
    df["ema_72h"] = df["price"].shift(1).ewm(span=72).mean()

    # ── 4. SUPPLY/DEMAND RATIOS & RAMP RATES ─────────────────────────────────
    df["total_renewable"] = (
        df["wind_offshore"] + df["wind_onshore"] + df["solar"]
    )
    df["ren_share"] = df["total_renewable"] / (df["load_forecast"] + 1e-6)
    df["demand_ren_ratio"] = df["load_forecast"] / (
        df["total_renewable"] + 1e-6
    )

    df["solar_ramp_1h"] = df["solar"] - df["solar"].shift(1)
    df["solar_ramp_3h"] = df["solar"] - df["solar"].shift(3)
    df["wind_ramp_1h"] = df["total_renewable"] - df["total_renewable"].shift(1)
    df["load_ramp_1h"] = df["load_forecast"] - df["load_forecast"].shift(1)

    # ── 5. CALENDAR FEATURES (CYCLIC) ─────────────────────────────────────────
    df["hour"] = df["datetime"].dt.hour
    df["dow"] = df["datetime"].dt.dayofweek
    df["month"] = df["datetime"].dt.month
    df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
    df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)
    df["dow_sin"] = np.sin(2 * np.pi * df["dow"] / 7)
    df["dow_cos"] = np.cos(2 * np.pi * df["dow"] / 7)
    df["month_sin"] = np.sin(2 * np.pi * (df["month"] - 1) / 12)
    df["month_cos"] = np.cos(2 * np.pi * (df["month"] - 1) / 12)

    # Drop NaNs created by lags/rolling windows
    df = df.dropna().reset_index(drop=True)

    feature_names = [
        "lag_1",
        "lag_2",
        "lag_3",
        "lag_6",
        "lag_12",
        "lag_24",
        "lag_48",
        "lag_168",
        "roll_mean_24",
        "roll_std_24",
        "roll_mean_168",
        "price_change",
        "ema_6h",
        "ema_24h",
        "ema_72h",
        "load_forecast",
        "residual_load",
        "total_gen",
        "pv_wind",
        "ren_share",
        "demand_ren_ratio",
        "solar_ramp_1h",
        "solar_ramp_3h",
        "wind_ramp_1h",
        "load_ramp_1h",
        "wind_offshore",
        "wind_onshore",
        "solar",
        "other_gen",
        "hour_sin",
        "hour_cos",
        "dow_sin",
        "dow_cos",
        "month_sin",
        "month_cos",
    ]
    feature_names = [f for f in feature_names if f in df.columns]

    print(f"  ✓ Created {len(feature_names)} candidate features")
    print(f"  ✓ Rows after cleaning: {len(df):,}")
    return df, feature_names


def split_train_val_test(df):
    train = df[df["datetime"] <= "2024-04-30 23:00"].copy()
    val = df[
        (df["datetime"] >= "2024-05-01") & (df["datetime"] <= "2025-04-30 23:00")
    ].copy()
    test = df[df["datetime"] >= "2025-05-01"].copy()
    return train, val, test


if __name__ == "__main__":
    df = pd.read_pickle(os.path.join(DATA_DIR, "data_part1.pkl"))
    df, features = engineer_features(df)
    train, val, test = split_train_val_test(df)

    df.to_pickle(os.path.join(DATA_DIR, "data_part2.pkl"))
    with open(os.path.join(DATA_DIR, "features_part2.json"), "w") as f:
        json.dump(features, f)

    print("\n✓ Saved to data_part2.pkl — Ready for Part 4 feature selection!")

Engineering advanced features...
  ✓ Created 35 candidate features
  ✓ Rows after cleaning: 55,319

✓ Saved to data_part2.pkl — Ready for Part 4 feature selection!


# Part 3 — Visualize the Data

Explores key market dynamics through six complementary plots prior to modeling.

---

### Visualization Plan

1. **Full Price Timeline (2020–2026):** Macro view capturing long-term trends and the impact of the 2022 energy crisis.
2. **Price Distribution Histogram:** Highlights structural market characteristics, including negative prices and heavy right-tails.
3. **Average Price by Hour of Day:** Reveals intraday demand patterns and peak pricing windows.
4. **Average Price by Month:** Captures seasonal supply and demand shifts across the calendar year.
5. **Price vs. Renewable Generation:** Visualizes the inverse relationship between renewable output and electricity prices.
6. **Example Week Zoom-In:** Granular inspection showing the continuous hourly shape and weekly profile.

In [9]:
"""
═══════════════════════════════════════════════════════════════════════════
 PART 3 — EXPLORATORY DATA ANALYSIS (EDA) & VISUALIZATION (PLOTLY)
═══════════════════════════════════════════════════════════════════════════
"""

def visualize_data_plotly(df, save_path=os.path.join(DOCS_DIR, "part3_data_overview.html")):
    print("Creating data visualization with Plotly...")

    # Create 3x2 grid of subplots
    fig = make_subplots(
        rows=3,
        cols=2,
        subplot_titles=(
            "1. Price over time — the 2022 energy crisis is obvious",
            "2. Price distribution — note negative prices & long right tail",
            "3. Average price by hour — cheap at night, peaks morning & evening",
            "4. Average price by month — higher in winter",
            "5. Price vs renewable output — more renewables, lower price",
            "6. One example week (Jun 2023) — daily up-and-down rhythm",
        ),
        vertical_spacing=0.08,
        horizontal_spacing=0.08,
    )

    # ── Plot 1: Full price timeline ──────────────────────────────────────────
    fig.add_trace(
        go.Scatter(
            x=df["datetime"],
            y=df["price"],
            mode="lines",
            line=dict(width=0.8, color="#378ADD"),
            name="Price (€/MWh)",
        ),
        row=1,
        col=1,
    )
    fig.add_hline(
        y=0, line_dash="dot", line_color="red", line_width=1, row=1, col=1
    )

    # ── Plot 2: Price distribution ───────────────────────────────────────────
    fig.add_trace(
        go.Histogram(
            x=df["price"],
            nbinsx=120,
            marker_color="#1D9E75",
            opacity=0.8,
            name="Distribution",
        ),
        row=1,
        col=2,
    )
    fig.add_vline(
        x=0,
        line_dash="dash",
        line_color="red",
        line_width=1.5,
        annotation_text="zero",
        row=1,
        col=2,
    )
    fig.update_yaxes(type="log", row=1, col=2)  # Log scale for y-axis

    # ── Plot 3: Average price by hour of day ─────────────────────────────────
    hourly = df.groupby("hour")["price"].mean().reset_index()
    fig.add_trace(
        go.Scatter(
            x=hourly["hour"],
            y=hourly["price"],
            mode="lines+markers",
            line=dict(width=2, color="#BA7517"),
            fill="tozeroy",
            fillcolor="rgba(186, 117, 23, 0.2)",
            name="Avg Price",
        ),
        row=2,
        col=1,
    )

    # ── Plot 4: Average price by month ───────────────────────────────────────
    monthly = df.groupby("month")["price"].mean().reset_index()
    months = [
        "Jan", "Feb", "Mar", "Apr", "May", "Jun",
        "Jul", "Aug", "Sep", "Oct", "Nov", "Dec",
    ]
    fig.add_trace(
        go.Bar(
            x=months,
            y=monthly["price"],
            marker_color="#7F77DD",
            opacity=0.8,
            name="Avg Price",
        ),
        row=2,
        col=2,
    )

    # ── Plot 5: Price vs renewable generation ────────────────────────────────
    renew = df["wind_offshore"] + df["wind_onshore"] + df["solar"]
    sample_size = min(4000, len(df))
    idx = np.random.choice(len(df), sample_size, replace=False)
    fig.add_trace(
        go.Scatter(
            x=renew.iloc[idx] / 1000,
            y=df["price"].iloc[idx],
            mode="markers",
            marker=dict(size=4, opacity=0.3, color="#D85A30"),
            name="Renewables vs Price",
        ),
        row=3,
        col=1,
    )
    fig.add_hline(
        y=0, line_dash="dot", line_color="red", line_width=1, row=3, col=1
    )

    # ── Plot 6: One example week ─────────────────────────────────────────────
    week = df[
        (df["datetime"] >= "2023-06-05") & (df["datetime"] < "2023-06-12")
    ]
    fig.add_trace(
        go.Scatter(
            x=week["datetime"],
            y=week["price"],
            mode="lines+markers",
            line=dict(width=1.2, color="#0C447C"),
            fill="tozeroy",
            fillcolor="rgba(12, 68, 124, 0.15)",
            name="Example Week",
        ),
        row=3,
        col=2,
    )

    # ── Axis Labels ──────────────────────────────────────────────────────────
    fig.update_xaxes(title_text="Date", row=1, col=1)
    fig.update_yaxes(title_text="Price (€/MWh)", row=1, col=1)

    fig.update_xaxes(title_text="Price (€/MWh)", row=1, col=2)
    fig.update_yaxes(title_text="Number of hours (Log)", row=1, col=2)

    fig.update_xaxes(title_text="Hour of day", dtick=2, row=2, col=1)
    fig.update_yaxes(title_text="Avg price (€/MWh)", row=2, col=1)

    fig.update_xaxes(title_text="Month", row=2, col=2)
    fig.update_yaxes(title_text="Avg price (€/MWh)", row=2, col=2)

    fig.update_xaxes(title_text="Renewable generation (GW)", row=3, col=1)
    fig.update_yaxes(title_text="Price (€/MWh)", row=3, col=1)

    fig.update_xaxes(title_text="Date", tickformat="%a\n%b %d", row=3, col=2)
    fig.update_yaxes(title_text="Price (€/MWh)", row=3, col=2)

    # ── Styling & Background Color ───────────────────────────────────────────
    fig.update_layout(
        title_text="German Day-Ahead Electricity Prices — Data Overview (2020–2026)",
        title_font=dict(size=18),
        paper_bgcolor="#f7f6f3",
        plot_bgcolor="#ffffff",
        height=1100,
        width=1300,
        showlegend=False,
    )
    fig.update_xaxes(showgrid=True, gridwidth=1, gridcolor="rgba(0,0,0,0.1)")
    fig.update_yaxes(showgrid=True, gridwidth=1, gridcolor="rgba(0,0,0,0.1)")

    # Save to interactive HTML file
    fig.write_html(save_path)
    print(f"  ✓ Saved interactive HTML dashboard to {save_path}")
    
    # Safely display or open in browser
    try:
        fig.show(renderer="iframe")
    except Exception:
        webbrowser.open('file://' + os.path.abspath(save_path))


# ── Run script ─────────────────────────────────────────────────────────────
if __name__ == "__main__":
    df_path = os.path.join(DOCS_DIR, "data_part2.pkl")
    print(f"Loading data from {df_path}...")
    df = pd.read_pickle(df_path)
    visualize_data_plotly(df)
    print("\n✓ Done — interactive dashboard saved and launched successfully!")

Loading data from data_part2.pkl...
Creating data visualization with Plotly...
  ✓ Saved interactive HTML dashboard to part3_data_overview.html



✓ Done — interactive dashboard saved and launched successfully!


# Part 4 — Feature Selection (13 Methods + Voting)

Evaluates all candidate features across 13 feature selection techniques. Each method casts a vote for features it identifies as important, and features reaching a consensus threshold ($\ge 5\text{ votes}$) are retained.

---

### Selection Methods by Category

* **Filter:** Pearson Correlation, Spearman Rank Correlation, Mutual Information, F-statistic, VIF
* **Wrapper:** Recursive Feature Elimination (RFE), Sequential Feature Selection (SFS)
* **Embedded:** LASSO Regression, Elastic Net, Random Forest Importance, Permutation Importance
* **Advanced:** Granger Causality, SHAP (SHapley Additive exPlanations)

---

### Workflow Priority

> **Variance Inflation Factor (VIF) Pre-Filtering:** VIF runs first as a preprocessing step to remove highly collinear and redundant features before the remaining methods execute their voting process.

In [13]:
# ── Helper: VIF (pure numpy, no statsmodels needed) ────────────────────────
def compute_vif(X, j):
    """VIF for feature j = 1/(1-R²) when regressing feature j on all others."""
    y_j = X[:, j]
    X_others = np.column_stack([np.ones(len(y_j)), np.delete(X, j, axis=1)])
    beta = np.linalg.lstsq(X_others, y_j, rcond=None)[0]
    r2 = 1 - np.sum((y_j - X_others @ beta) ** 2) / (
        np.sum((y_j - y_j.mean()) ** 2) + 1e-10
    )
    return 1.0 / (1.0 - min(max(r2, 0), 0.9999))


# ── Helper: Granger causality (pure numpy) ──────────────────────────────────
def granger_pvalue(y, x, maxlag=8):
    """Min p-value: does past x help predict y beyond past y alone?"""
    n = len(y)
    min_p = 1.0
    for lag in range(1, maxlag + 1):
        if n <= 2 * lag + 5:
            continue
        ne = n - lag
        Y = y[lag:]
        Yr = np.column_stack(
            [np.ones(ne)] + [y[lag - k - 1 : n - k - 1] for k in range(lag)]
        )
        Yu = np.column_stack([Yr] + [x[lag - k - 1 : n - k - 1] for k in range(lag)])
        try:
            br = np.linalg.lstsq(Yr, Y, rcond=None)[0]
            bu = np.linalg.lstsq(Yu, Y, rcond=None)[0]
            rr = np.sum((Y - Yr @ br) ** 2)
            ru = np.sum((Y - Yu @ bu) ** 2)
            denom = ne - 2 * lag - 1
            if denom <= 0 or ru <= 0:
                continue
            F = ((rr - ru) / lag) / (ru / denom)
            min_p = min(min_p, float(1 - stats.f.cdf(F, lag, denom)))
        except Exception:
            pass
    return min_p


def run_feature_selection(X, y, X_test, y_test, df_train, features):
    """Run all 13 methods.

    Returns (selected_features, scores, votes, vif_removed).
    """
    print("Running 13 feature-selection methods...\n")
    votes = {f: 0 for f in features}
    scores = {}

    # ── STEP 1: VIF pre-filter (remove collinear features first) ─────────────
    print("  [VIF] removing multicollinear features...")
    remaining = list(features)
    removed = []
    while True:
        Xs = np.column_stack([X[:, features.index(f)] for f in remaining])
        vifs = {f: compute_vif(Xs, i) for i, f in enumerate(remaining)}
        worst = max(vifs, key=vifs.get)
        if vifs[worst] > 10 and len(remaining) > 5:
            remaining.remove(worst)
            removed.append(worst)
        else:
            scores["VIF"] = vifs
            break
    print(f"        removed {len(removed)}: {removed}")

    # Only VIF-survivors get to vote
    keep_mask = np.array([f in remaining for f in features])
    Xv = X[:, keep_mask]
    Xtv = X_test[:, keep_mask]
    fv = [f for f in features if f in remaining]

    # ── FILTER METHODS ───────────────────────────────────────────────────────
    # 1. Pearson — linear correlation
    sc = {f: abs(pearsonr(Xv[:, i], y)[0]) for i, f in enumerate(fv)}
    scores["Pearson"] = sc
    for f in [f for f, v in sc.items() if v > 0.15]:
        votes[f] += 1
    print("  [1] Pearson done")

    # 2. Spearman — rank correlation (nonlinear monotonic)
    sc = {f: abs(spearmanr(Xv[:, i], y)[0]) for i, f in enumerate(fv)}
    scores["Spearman"] = sc
    for f in [f for f, v in sc.items() if v > 0.15]:
        votes[f] += 1
    print("  [2] Spearman done")

    # 3. Mutual Information — any dependency
    mi = mutual_info_regression(Xv, y, random_state=42)
    sc = dict(zip(fv, mi))
    scores["MutualInfo"] = sc
    for f in [f for f, v in sc.items() if v > 0.05]:
        votes[f] += 1
    print("  [3] Mutual Information done")

    # 4. F-statistic — variance explained
    fvals, pvals = f_regression(Xv, y)
    scores["F-stat"] = dict(zip(fv, fvals))
    for f in [f for f, p in zip(fv, pvals) if p < 0.05]:
        votes[f] += 1
    print("  [4] F-statistic done")

    # ── WRAPPER METHODS ──────────────────────────────────────────────────────
    # 5. RFE — recursive feature elimination
    rfe = RFE(LinearRegression(), n_features_to_select=min(15, len(fv)))
    rfe.fit(Xv, y)
    scores["RFE"] = {f: 1 / r for f, r in zip(fv, rfe.ranking_)}
    for f in [f for f, s in zip(fv, rfe.support_) if s]:
        votes[f] += 1
    print("  [5] RFE done")

    # 6. Sequential Feature Selection (forward)
    n_sub = min(4000, len(y))
    idx = np.random.choice(len(y), n_sub, replace=False)
    sfs = SequentialFeatureSelector(
        LinearRegression(),
        n_features_to_select=min(15, len(fv)),
        direction="forward",
        cv=3,
        n_jobs=-1,
    )
    sfs.fit(Xv[idx], y[idx])
    sel_sfs = set(np.array(fv)[sfs.get_support()])
    scores["SFS"] = {f: (1 if f in sel_sfs else 0) for f in fv}
    for f in sel_sfs:
        votes[f] += 1
    print("  [6] Sequential Selection done")

    # ── EMBEDDED METHODS ─────────────────────────────────────────────────────
    # 7. LASSO
    best_mae, best_coef = np.inf, None
    for a in [0.001, 0.01, 0.05, 0.1, 0.5, 1.0]:
        m = Lasso(alpha=a, max_iter=5000).fit(Xv, y)
        mae = np.mean(np.abs(y - m.predict(Xv)))
        if mae < best_mae and np.sum(m.coef_ != 0) >= 5:
            best_mae, best_coef = mae, m.coef_.copy()
    sc = {f: abs(c) for f, c in zip(fv, best_coef)}
    scores["LASSO"] = sc
    for f in [f for f, v in sc.items() if v > 0]:
        votes[f] += 1
    print("  [7] LASSO done")

    # 8. Elastic Net
    best_mae, best_coef = np.inf, None
    for a in [0.01, 0.1, 0.5]:
        for l1 in [0.3, 0.5, 0.7]:
            m = ElasticNet(alpha=a, l1_ratio=l1, max_iter=5000).fit(Xv, y)
            mae = np.mean(np.abs(y - m.predict(Xv)))
            if mae < best_mae and np.sum(m.coef_ != 0) >= 5:
                best_mae, best_coef = mae, m.coef_.copy()
    sc = {f: abs(c) for f, c in zip(fv, best_coef)}
    scores["ElasticNet"] = sc
    for f in [f for f, v in sc.items() if v > 0]:
        votes[f] += 1
    print("  [8] Elastic Net done")

    # 9. Random Forest importance
    rf = RandomForestRegressor(
        n_estimators=100,
        max_depth=8,
        min_samples_leaf=20,
        random_state=42,
        n_jobs=-1,
    )
    rf.fit(Xv, y)
    sc = dict(zip(fv, rf.feature_importances_))
    scores["RF"] = sc
    for f in [f for f, v in sc.items() if v > 0.01]:
        votes[f] += 1
    print("  [9] Random Forest done")

    # 10. Permutation importance
    n_p = min(2000, len(X_test))
    idx_p = np.random.choice(len(X_test), n_p, replace=False)
    perm = permutation_importance(
        rf, Xtv[idx_p], y_test[idx_p], n_repeats=5, random_state=42
    )
    sc = dict(zip(fv, perm.importances_mean))
    scores["Permutation"] = sc
    for f in [f for f, v in sc.items() if v > 0.001]:
        votes[f] += 1
    print("  [10] Permutation done")

    # ── ADVANCED METHODS ─────────────────────────────────────────────────────
    # 11. Granger causality
    sc = {}
    for f in fv:
        if f in [
            "hour_sin",
            "hour_cos",
            "dow_sin",
            "dow_cos",
            "month_sin",
            "month_cos",
        ]:
            sc[f] = 0.5
        else:
            x_arr = (
                df_train[f].values
                if f in df_train.columns
                else Xv[:, fv.index(f)]
            )
            sc[f] = 1.0 - granger_pvalue(df_train["price"].values, x_arr)
    scores["Granger"] = sc
    for f in [f for f, v in sc.items() if v > 0.95]:
        votes[f] += 1
    print("  [11] Granger causality done")

    # 12. SHAP (permutation-based approximation using the RF)
    base = rf.predict(Xtv[:500])
    shap = np.zeros(len(fv))
    for j in range(len(fv)):
        Xp = Xtv[:500].copy()
        Xp[:, j] = np.random.permutation(Xp[:, j])
        shap[j] = np.mean(np.abs(base - rf.predict(Xp)))
    sc = dict(zip(fv, shap))
    scores["SHAP"] = sc
    mx = max(sc.values()) or 1
    for f in [f for f, v in sc.items() if v > 0.1 * mx]:
        votes[f] += 1
    print("  [12] SHAP done")

    # VIF-removed features get 0 votes
    for f in removed:
        votes[f] = 0

    # ── FINAL SELECTION ──────────────────────────────────────────────────────
    selected = [f for f, v in votes.items() if v >= 5]
    print(f"\n  RESULT: {len(selected)} features selected (>= 5 votes)")
    print(f"  {'Feature':<22}{'Votes':>6}  Decision")
    print("  " + "-" * 42)
    for f, v in sorted(votes.items(), key=lambda x: -x[1]):
        print(f"  {f:<22}{v:>6}  {'SELECT' if v >= 5 else 'drop'}")

    return selected, scores, votes, removed


# ── Run this part ──────────────────────────────────────────────────────────
if __name__ == "__main__":
    df = pd.read_pickle(os.path.join(DATA_DIR, "data_part2.pkl"))
    with open(os.path.join(DATA_DIR, "features_part2.json")) as f:
        features = json.load(f)

    # Updated Split standard
    train = df[df["datetime"] <= "2024-04-30 23:00"].copy()
    val = df[
        (df["datetime"] >= "2024-05-01") & (df["datetime"] <= "2025-04-30 23:00")
    ].copy()
    test = df[df["datetime"] >= "2025-05-01"].copy()

    # Fit Scaler solely on Train set
    scaler_x = StandardScaler()
    scaler_y = StandardScaler()

    X_tr = scaler_x.fit_transform(train[features].values)
    y_tr = scaler_y.fit_transform(train["price"].values.reshape(-1, 1)).flatten()

    X_val = scaler_x.transform(val[features].values)
    y_val = val["price"].values

    X_te = scaler_x.transform(test[features].values)
    y_te = test["price"].values

    # Run feature selection using train and test sets
    selected, scores, votes, removed = run_feature_selection(
        X_tr, y_tr, X_te, y_te, train, features
    )

    # Save outputs for down-stream modeling steps
    np.save(
        os.path.join(DATA_DIR, "fs_part4.npy"),
        {
            "selected": selected,
            "scores": scores,
            "votes": votes,
            "removed": removed,
            "features": features,
        },
        allow_pickle=True,
    )
    with open(os.path.join(DATA_DIR, "selected_part4.json"), "w") as f:
        json.dump(selected, f)

    print("\n✓ Saved feature selection — ready for Part 5")

Running 13 feature-selection methods...

  [VIF] removing multicollinear features...
        removed 15: ['load_forecast', 'total_gen', 'pv_wind', 'ema_24h', 'ema_6h', 'ema_72h', 'lag_2', 'roll_mean_24', 'lag_1', 'ren_share', 'lag_24', 'solar_ramp_1h', 'roll_mean_168', 'residual_load', 'lag_3']
  [1] Pearson done
  [2] Spearman done
  [3] Mutual Information done
  [4] F-statistic done
  [5] RFE done


/opt/anaconda3/lib/python3.13/site-packages/pandas/core/computation/expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/opt/anaconda3/lib/python3.13/site-packages/pandas/core/computation/expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/opt/anaconda3/lib/python3.13/site-packages/pandas/core/computation/expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/opt/anaconda3/lib/python3.13/site-packages/pandas/core/computation/expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.che

  [6] Sequential Selection done
  [7] LASSO done
  [8] Elastic Net done
  [9] Random Forest done
  [10] Permutation done
  [11] Granger causality done
  [12] SHAP done

  RESULT: 15 features selected (>= 5 votes)
  Feature                Votes  Decision
  ------------------------------------------
  lag_12                    12  SELECT
  lag_48                    12  SELECT
  lag_168                   12  SELECT
  other_gen                 12  SELECT
  lag_6                     11  SELECT
  price_change              10  SELECT
  demand_ren_ratio          10  SELECT
  roll_std_24                9  SELECT
  wind_offshore              9  SELECT
  wind_onshore               9  SELECT
  month_sin                  8  SELECT
  solar                      6  SELECT
  month_cos                  6  SELECT
  solar_ramp_3h              5  SELECT
  hour_sin                   5  SELECT
  wind_ramp_1h               4  drop
  load_ramp_1h               4  drop
  hour_cos                   4  drop
  dow

# Part 5 — Visualize Feature Selection

Evaluates the outcomes of the feature selection process through four diagnostic plots.

---

### Visualization Plan

1. **Pearson Correlation Bars:** Ranks individual features based on linear relationship strength with the target price.
2. **Random Forest Importance Bars:** Highlights feature contributions based on non-linear impurity reduction.
3. **Selection Consensus Heatmap:** Cross-tabulates features against all 13 selection methods to illustrate method agreement.
4. **Final Vote Distribution:** Displays total vote counts per feature alongside the threshold decision line ($\ge 5\text{ votes}$).

In [17]:
pio.renderers.default = "iframe"
def visualize_feature_selection_plotly(data):
    features = data["features"]
    votes = data["votes"]
    scores = data["scores"]
    selected = data["selected"]
    removed = data["removed"]

    # Sort features by votes (most voted on top)
    order = sorted(features, key=lambda f: votes[f])
    n = len(order)

    def get_color(f):
        if f in removed:
            return "#D85A30"  # Orange = removed by VIF
        if f in selected:
            return "#1D9E75"  # Green = selected
        return "#C0BDB4"  # Grey = dropped

    colors = [get_color(f) for f in order]

    fig = make_subplots(
        rows=2,
        cols=2,
        subplot_titles=(
            "1. Pearson correlation with price",
            "2. Random Forest importance",
            "3. All 12 methods (brighter = more important)",
            "4. Final vote count — the decision",
        ),
        horizontal_spacing=0.18,
        vertical_spacing=0.12,
    )

    # ── Plot 1: Pearson correlation ──────────────────────────────────────────
    vals1 = [scores["Pearson"].get(f, 0) for f in order]
    fig.add_trace(
        go.Bar(
            x=vals1,
            y=order,
            orientation="h",
            marker_color=colors,
            name="Pearson",
            showlegend=False,
            hovertemplate="Feature: %{y}<br>Pearson: %{x:.4f}<extra></extra>",
        ),
        row=1,
        col=1,
    )
    fig.add_vline(
        x=0.15,
        line_dash="dash",
        line_color="red",
        line_width=1.5,
        annotation_text="threshold 0.15",
        annotation_position="bottom right",
        row=1,
        col=1,
    )

    # ── Plot 2: Random Forest importance ─────────────────────────────────────
    vals2 = [scores["RF"].get(f, 0) for f in order]
    fig.add_trace(
        go.Bar(
            x=vals2,
            y=order,
            orientation="h",
            marker_color=colors,
            name="RF",
            showlegend=False,
            hovertemplate="Feature: %{y}<br>Importance: %{x:.4f}<extra></extra>",
        ),
        row=1,
        col=2,
    )

    # ── Plot 3: Heatmap of all methods ───────────────────────────────────────
    methods = [
        "Pearson",
        "Spearman",
        "MutualInfo",
        "F-stat",
        "RFE",
        "SFS",
        "LASSO",
        "ElasticNet",
        "RF",
        "Permutation",
        "Granger",
        "SHAP",
    ]
    M = np.zeros((n, len(methods)))
    for j, m in enumerate(methods):
        raw = np.array([scores[m].get(f, 0) for f in order])
        M[:, j] = raw / (raw.max() + 1e-9)  # normalize each column to 0-1

    fig.add_trace(
        go.Heatmap(
            z=M,
            x=methods,
            y=order,
            colorscale="YlOrRd",
            zmin=0,
            zmax=1,
            showscale=True,
            colorbar=dict(
                title="Normalized",
                len=0.42,
                y=0.2,
                x=0.42,
                thickness=15,
            ),
            hovertemplate="Feature: %{y}<br>Method: %{x}<br>Score: %{z:.2f}<extra></extra>",
        ),
        row=2,
        col=1,
    )

    # ── Plot 4: Final vote count ─────────────────────────────────────────────
    vals4 = [votes[f] for f in order]
    fig.add_trace(
        go.Bar(
            x=vals4,
            y=order,
            orientation="h",
            marker_color=colors,
            text=[str(v) for v in vals4],
            textposition="outside",
            name="Votes",
            showlegend=False,
            hovertemplate="Feature: %{y}<br>Votes: %{x}<extra></extra>",
        ),
        row=2,
        col=2,
    )
    fig.add_vline(
        x=5,
        line_dash="dash",
        line_color="red",
        line_width=1.5,
        annotation_text="threshold (>=5)",
        annotation_position="bottom right",
        row=2,
        col=2,
    )

    # ── Legend Traces ────────────────────────────────────────────────────────
    fig.add_trace(
        go.Bar(
            x=[None],
            y=[None],
            marker_color="#1D9E75",
            name="Selected",
            showlegend=True,
        ),
        row=1,
        col=1,
    )
    fig.add_trace(
        go.Bar(
            x=[None],
            y=[None],
            marker_color="#C0BDB4",
            name="Dropped (too few votes)",
            showlegend=True,
        ),
        row=1,
        col=1,
    )
    fig.add_trace(
        go.Bar(
            x=[None],
            y=[None],
            marker_color="#D85A30",
            name="Removed by VIF (redundant)",
            showlegend=True,
        ),
        row=1,
        col=1,
    )

    # ── Axis Labels & Layout Config ──────────────────────────────────────────
    fig.update_xaxes(title_text="|correlation|", row=1, col=1)
    fig.update_xaxes(title_text="importance", row=1, col=2)
    fig.update_xaxes(title_text="number of methods voting yes", range=[0, 14], row=2, col=2)

    fig.update_layout(
        title=dict(
            text=f"Feature Selection — {len(selected)} of {len(features)} features selected (13 methods, real SMARD data)",
            font=dict(size=16),
            x=0.5,
            xanchor="center",
        ),
        paper_bgcolor="#f7f6f3",
        plot_bgcolor="#ffffff",
        height=1200,
        width=1400,
        legend=dict(
            orientation="h",
            yanchor="bottom",
            y=1.02,
            xanchor="center",
            x=0.5,
            font=dict(size=11),
        ),
    )

    fig.update_xaxes(showgrid=True, gridwidth=1, gridcolor="rgba(0,0,0,0.1)")
    fig.update_yaxes(showgrid=True, gridwidth=1, gridcolor="rgba(0,0,0,0.1)")

    # Display directly in Jupyter Notebook
    fig.show()


# ── Run directly in a Jupyter Cell ──────────────────────────────────────────
data = np.load(
    os.path.join(DATA_DIR, "fs_part4.npy"), allow_pickle=True
).item()
visualize_feature_selection_plotly(data)

# Part 6 — Signed-Log Transform (Production Ready)

Addresses wide value ranges and negative values in electricity price modeling through symmetrical logarithmic scaling.

---

### Motivation & Formulation

Electricity prices exhibit extreme price spikes (up to €936) and negative price events (down to -€500). To stabilize model training on this skewed range, we apply the **Signed-Log transform**:

$$z = \text{sign}(p) \cdot \ln(1 + |p|)$$

To invert predictions back to original Euro units:

$$p = \text{sign}(z) \cdot \left(\exp(|z|) - 1\right)$$

---

### Uncertainty & Prediction Interval Rules

> **Critical Note on Intervals:** Do **not** use the Delta Method ($\sigma_{\text{EUR}} = \sigma_z \cdot \exp|z|$) to transform variance for interval estimation. Instead, construct upper and lower prediction bounds in log-space first, then invert both bounds individually:
>
> $$\text{Bounds}_{\text{EUR}} = \text{InverseSignedLog}\left([z - q, z + q]\right)$$

In [18]:
"""
═══════════════════════════════════════════════════════════════════════════
 PART 6 — SIGNED-LOG TRANSFORM (PRODUCTION READY)
═══════════════════════════════════════════════════════════════════════════
 Problem: Electricity prices have extreme spikes (up to €936) and go
 negative (down to -€500). This wide, skewed range makes models struggle.

 Solution: SIGNED-LOG transform (works for negative numbers):
     z = sign(p) * log(1 + |p|)
 Inversion to get Euros back:
     p = sign(z) * (exp(|z|) - 1)

 CRITICAL NOTE ON UNCERTAINTY:
   Do NOT use the delta-method (sigma_eur = sigma_z * exp|z|) for interval
   generation! Construct upper/lower bounds [z - q, z + q] in log-space FIRST,
   then invert both bounds using inverse_signed_log().
═══════════════════════════════════════════════════════════════════════════
"""


pio.renderers.default = "iframe"
# ── Transform Functions ──────────────────────────────────────────────────
def signed_log(p):
    """Forward transform: handles negative, zero, and positive prices."""
    return np.sign(p) * np.log1p(np.abs(p))  # log1p(x) = log(1+x)


def inverse_signed_log(z, price_clip=(-600.0, 1000.0)):
    """Inverse transform: converts signed-log to Euros with safety guardrails."""
    z_min = (
        -np.log1p(np.abs(price_clip[0]))
        if price_clip[0] < 0
        else np.log1p(price_clip[0])
    )
    z_max = np.log1p(price_clip[1])

    z_safe = np.clip(z, z_min, z_max)
    p = np.sign(z_safe) * np.expm1(np.abs(z_safe))
    return np.clip(p, price_clip[0], price_clip[1])


# ── Plotly Visualization Function ─────────────────────────────────────────
def visualize_transform_plotly(price):
    print("Visualizing signed-log transform with Plotly...")
    z = signed_log(price)

    fig = make_subplots(
        rows=2,
        cols=2,
        subplot_titles=(
            "1. RAW price — wide range, extreme spikes",
            "2. SIGNED-LOG price — compact, easier to model",
            "3. The transform curve — squashes big values, keeps small ones",
            "4. Round-trip check — transform → inverse = original",
        ),
        horizontal_spacing=0.1,
        vertical_spacing=0.12,
    )

    # ── Plot 1: Raw price distribution ───────────────────────────────────────
    fig.add_trace(
        go.Histogram(
            x=price,
            nbinsx=100,
            marker_color="#D85A30",
            opacity=0.8,
            name="Raw Price",
            hovertemplate="Price (€/MWh): %{x}<br>Count: %{y}<extra></extra>",
        ),
        row=1,
        col=1,
    )
    fig.add_vline(
        x=0, line_dash="dot", line_color="black", line_width=1, row=1, col=1
    )
    fig.add_annotation(
        text=f"range: {price.max()-price.min():.0f}<br>std: {price.std():.0f}",
        xref="x domain",
        yref="y domain",
        x=0.95,
        y=0.9,
        showarrow=False,
        bgcolor="white",
        bordercolor="#ccc",
        borderwidth=1,
        row=1,
        col=1,
    )

    # ── Plot 2: Signed-log distribution ──────────────────────────────────────
    fig.add_trace(
        go.Histogram(
            x=z,
            nbinsx=100,
            marker_color="#1D9E75",
            opacity=0.8,
            name="Signed-Log",
            hovertemplate="Signed-log: %{x:.2f}<br>Count: %{y}<extra></extra>",
        ),
        row=1,
        col=2,
    )
    fig.add_vline(
        x=0, line_dash="dot", line_color="black", line_width=1, row=1, col=2
    )
    fig.add_annotation(
        text=f"range: {z.max()-z.min():.1f}<br>std: {z.std():.2f}",
        xref="x domain",
        yref="y domain",
        x=0.95,
        y=0.9,
        showarrow=False,
        bgcolor="white",
        bordercolor="#ccc",
        borderwidth=1,
        row=1,
        col=2,
    )

    # ── Plot 3: The transform curve ──────────────────────────────────────────
    p_range = np.linspace(-500, 936, 500)
    z_range = signed_log(p_range)
    fig.add_trace(
        go.Scatter(
            x=p_range,
            y=z_range,
            mode="lines",
            line=dict(color="#0C447C", width=2.5),
            name="Transform Curve",
            hovertemplate="Price: €%{x:.1f}<br>Signed-log: %{y:.3f}<extra></extra>",
        ),
        row=2,
        col=1,
    )

    # Highlight specific points along the curve
    pts = [-500, -50, 0, 50, 500, 936]
    z_pts = [signed_log(p) for p in pts]
    fig.add_trace(
        go.Scatter(
            x=pts,
            y=z_pts,
            mode="markers+text",
            marker=dict(color="#D85A30", size=8),
            text=[f"€{p}" for p in pts],
            textposition="top left",
            name="Sample Points",
            showlegend=False,
        ),
        row=2,
        col=1,
    )

    # ── Plot 4: Round-trip check ─────────────────────────────────────────────
    recovered = inverse_signed_log(z)
    sample_size = min(3000, len(price))
    idx = np.random.choice(len(price), sample_size, replace=False)

    fig.add_trace(
        go.Scatter(
            x=price[idx],
            y=recovered[idx],
            mode="markers",
            marker=dict(color="#7F77DD", size=4, opacity=0.3),
            name="Recovered Points",
            hovertemplate="Original: €%{x:.2f}<br>Recovered: €%{y:.2f}<extra></extra>",
        ),
        row=2,
        col=2,
    )

    # Identity reference line y = x
    fig.add_trace(
        go.Scatter(
            x=[price.min(), price.max()],
            y=[price.min(), price.max()],
            mode="lines",
            line=dict(color="red", dash="dash", width=1.5),
            name="Perfect Recovery",
        ),
        row=2,
        col=2,
    )

    err = np.abs(price - recovered).max()
    fig.add_annotation(
        text=f"max error: {err:.2e}<br>(perfectly invertible)",
        xref="x domain",
        yref="y domain",
        x=0.05,
        y=0.9,
        showarrow=False,
        bgcolor="white",
        bordercolor="#ccc",
        borderwidth=1,
        row=2,
        col=2,
    )

    # ── Axis Titles & Formatting ─────────────────────────────────────────────
    fig.update_xaxes(title_text="Price (€/MWh)", row=1, col=1)
    fig.update_yaxes(title_text="Count", row=1, col=1)

    fig.update_xaxes(title_text="signed-log price", row=1, col=2)
    fig.update_yaxes(title_text="Count", row=1, col=2)

    fig.update_xaxes(title_text="Price (€/MWh)", row=2, col=1)
    fig.update_yaxes(title_text="signed-log value", row=2, col=1)

    fig.update_xaxes(title_text="Original price", row=2, col=2)
    fig.update_yaxes(title_text="Recovered price", row=2, col=2)

    # Global Layout
    fig.update_layout(
        title=dict(
            text="Signed-Log Transform — Taming Negative Prices & Extreme Spikes",
            font=dict(size=16),
            x=0.5,
            xanchor="center",
        ),
        paper_bgcolor="#f7f6f3",
        plot_bgcolor="#ffffff",
        height=900,
        width=1200,
        showlegend=False,
    )

    fig.update_xaxes(showgrid=True, gridwidth=1, gridcolor="rgba(0,0,0,0.1)")
    fig.update_yaxes(showgrid=True, gridwidth=1, gridcolor="rgba(0,0,0,0.1)")

    # Render directly in Jupyter Notebook
    fig.show()


# ── Run Script ─────────────────────────────────────────────────────────────
if __name__ == "__main__":
    df = pd.read_pickle(os.path.join(DATA_DIR, "data_part2.pkl"))
    price = df["price"].values

    print("\nExample transformations:")
    print(f"  {'price':>8} {'signed_log':>12} {'recovered':>10}")
    for p in [-500, -50, -1, 0, 1, 50, 500, 936]:
        z = signed_log(p)
        print(f"  {p:>8.1f} {z:>12.3f} {inverse_signed_log(z):>10.1f}")

    visualize_transform_plotly(price)


Example transformations:
     price   signed_log  recovered
    -500.0       -6.217     -500.0
     -50.0       -3.932      -50.0
      -1.0       -0.693       -1.0
       0.0        0.000        0.0
       1.0        0.693        1.0
      50.0        3.932       50.0
     500.0        6.217      500.0
     936.0        6.843      936.0
Visualizing signed-log transform with Plotly...


# Part 7 — Regime Detection (Leakage-Free)

Identifies distinct market states ("moods") automatically from price dynamics without manual labeling.

---

### Defined Market Regimes

* **Normal:** Baseline market behavior with typical supply-demand balance and price stability.
* **Elevated:** Increased price volatility or sustained higher price levels.
* **Crisis:** Severe structural shocks, extreme price spikes, or acute supply constraints.

---

### Methodological Fixes

1. **Strict Train-Only Normalization:** Min/Max stress normalization parameters are calculated exclusively on training data (pre-2024) to prevent look-ahead bias and data leakage into testing periods.
2. **Dynamic Plotting Scalability:** Y-axis limits adapt dynamically to dataset extremes to maintain clear visualization across distinct market periods.

In [19]:
# ── Regime Detection Function (Leakage-Free) ─────────────────────────────────

pio.renderers.default = "iframe"
def detect_regimes(df):
    """Add a 'regime' column (0=Normal, 1=Elevated, 2=Crisis) and 'stress'."""
    print("Detecting market regimes...")
    price = df["price"].values
    dates = df["datetime"]

    # Grid-stress index = 60% price level + 40% price volatility
    roll_level = (
        pd.Series(price)
        .rolling(720, min_periods=48)
        .mean()
        .bfill()
        .values  # 30-day
    )
    roll_vol = (
        pd.Series(price)
        .rolling(168, min_periods=24)
        .std()
        .bfill()
        .values  # 7-day
    )

    # Fit scaling bounds on training period (pre-2024) to avoid data leakage
    tr_mask = dates < "2024-01-01"
    min_lvl, max_lvl = roll_level[tr_mask].min(), roll_level[tr_mask].max()
    min_vol, max_vol = roll_vol[tr_mask].min(), roll_vol[tr_mask].max()

    norm_level = np.clip(
        (roll_level - min_lvl) / (max_lvl - min_lvl + 1e-9), 0, 1
    )
    norm_vol = np.clip((roll_vol - min_vol) / (max_vol - min_vol + 1e-9), 0, 1)

    stress = 0.6 * norm_level + 0.4 * norm_vol
    stress = (
        pd.Series(stress).rolling(168, min_periods=24).mean().bfill().values
    )

    # Classify into 3 regimes using training quantiles
    q_elev, q_crisis = np.quantile(stress[tr_mask], [0.55, 0.85])
    regime = np.where(stress > q_crisis, 2, np.where(stress > q_elev, 1, 0))

    df["stress"] = stress
    df["regime"] = regime

    names = {0: "Normal", 1: "Elevated", 2: "Crisis"}
    for r in [0, 1, 2]:
        m = regime == r
        yr = pd.Series(df["datetime"][m]).dt.year.mode()
        print(
            f"  {names[r]:9s}: {m.sum():6,} hrs  avg €{price[m].mean():6.1f}  "
            f"(mostly {yr.iloc[0] if len(yr) else '?'})"
        )

    crisis_2022 = ((regime == 2) & (df["datetime"].dt.year == 2022)).sum()
    crisis_total = (regime == 2).sum()
    print(
        f"  >> {crisis_2022:,} of {crisis_total:,} crisis hours are in 2022 "
        f"({crisis_2022/crisis_total*100:.0f}%) — matches the real energy crisis!"
    )
    return df, q_elev, q_crisis


# ── Plotly Visualization Function ─────────────────────────────────────────
def visualize_regimes_plotly(df, q_elev, q_crisis):
    print("Visualizing regimes with Plotly...")
    df["datetime"] = pd.to_datetime(df["datetime"])  # Ensure datetime type
    dates = df["datetime"]
    price = df["price"].values
    stress = df["stress"].values
    regime = df["regime"].values

    rcol = {0: "#1D9E75", 1: "#BA7517", 2: "#D85A30"}
    names = {0: "Normal", 1: "Elevated", 2: "Crisis"}

    fig = make_subplots(
        rows=3,
        cols=1,
        subplot_titles=(
            "Price timeline — the model paints 2022 RED automatically (the crisis)",
            "The grid-stress signal — crosses the red line during the crisis",
            "Regime classification of every hour",
        ),
        row_heights=[0.55, 0.30, 0.15],
        vertical_spacing=0.08,
        shared_xaxes=True,
    )

    # ── Plot 1: Price Timeline ───────────────────────────────────────────────
    fig.add_trace(
        go.Scatter(
            x=dates,
            y=price,
            mode="lines",
            line=dict(color="#222222", width=0.8),
            name="Price (€/MWh)",
            hovertemplate="Date: %{x}<br>Price: €%{y:.2f}<extra></extra>",
        ),
        row=1,
        col=1,
    )

    # Collect rectangular background shapes for regimes
    shapes = []
    for r in [0, 1, 2]:
        inreg = (regime == r).astype(int)
        edges = np.where(np.diff(np.concatenate([[0], inreg, [0]])) != 0)[0]
        for i in range(0, len(edges), 2):
            s, e = edges[i], min(edges[i + 1] - 1, len(dates) - 1)
            shapes.append(
                dict(
                    type="rect",
                    xref="x",
                    yref="y",
                    x0=dates.iloc[s],
                    x1=dates.iloc[e],
                    y0=price.min() - 50,
                    y1=price.max() + 100,
                    fillcolor=rcol[r],
                    opacity=0.15,
                    layer="below",
                    line_width=0,
                )
            )

    # Dynamic annotation for 2022 Crisis
    fig.add_annotation(
        x=pd.Timestamp("2022-08-01"),
        y=450,
        ax=pd.Timestamp("2023-06-01"),
        ay=600,
        xref="x",
        yref="y",
        axref="x",
        ayref="y",
        text="2022 crisis<br>auto-detected",
        showarrow=True,
        arrowhead=2,
        arrowsize=1,
        arrowwidth=1.5,
        arrowcolor="#791F1F",
        font=dict(size=12, color="#791F1F", family="sans-serif"),
        bgcolor="white",
        bordercolor="#791F1F",
        borderpad=4,
        row=1,
        col=1,
    )

    # ── Plot 2: Stress Index ─────────────────────────────────────────────────
    fig.add_trace(
        go.Scatter(
            x=dates,
            y=stress,
            mode="lines",
            line=dict(color="#185FA5", width=1.2),
            fill="tozeroy",
            fillcolor="rgba(55, 138, 221, 0.25)",
            name="Stress Index",
            hovertemplate="Date: %{x}<br>Stress: %{y:.3f}<extra></extra>",
        ),
        row=2,
        col=1,
    )

    fig.add_hline(
        y=q_elev,
        line_dash="dash",
        line_color="#BA7517",
        line_width=1.5,
        annotation_text="Elevated threshold",
        annotation_position="top right",
        row=2,
        col=1,
    )
    fig.add_hline(
        y=q_crisis,
        line_dash="dash",
        line_color="#D85A30",
        line_width=1.5,
        annotation_text="Crisis threshold",
        annotation_position="top right",
        row=2,
        col=1,
    )

    # ── Plot 3: Regime Classification Ribbon ─────────────────────────────────
    for r in [0, 1, 2]:
        inreg = (regime == r).astype(int)
        edges = np.where(np.diff(np.concatenate([[0], inreg, [0]])) != 0)[0]
        for i in range(0, len(edges), 2):
            s, e = edges[i], min(edges[i + 1] - 1, len(dates) - 1)
            shapes.append(
                dict(
                    type="rect",
                    xref="x3",
                    yref="y3",
                    x0=dates.iloc[s],
                    x1=dates.iloc[e],
                    y0=0,
                    y1=1,
                    fillcolor=rcol[r],
                    opacity=0.85,
                    layer="below",
                    line_width=0,
                )
            )

    # ── Legend Dummy Markers ─────────────────────────────────────────────────
    for r in [0, 1, 2]:
        fig.add_trace(
            go.Scatter(
                x=[None],
                y=[None],
                mode="markers",
                marker=dict(size=12, color=rcol[r], symbol="square"),
                name=names[r],
                showlegend=True,
            ),
            row=1,
            col=1,
        )

    # ── Axis Formatting ──────────────────────────────────────────────────────
    fig.update_yaxes(
        title_text="Price (€/MWh)",
        range=[price.min() - 50, price.max() + 100],
        row=1,
        col=1,
    )
    fig.update_yaxes(title_text="Stress index", range=[0, 1.05], row=2, col=1)
    fig.update_yaxes(
        showticklabels=False, showgrid=False, range=[0, 1], row=3, col=1
    )
    fig.update_xaxes(title_text="Date", row=3, col=1)

    # ── Global Layout ────────────────────────────────────────────────────────
    fig.update_layout(
        shapes=shapes,
        title=dict(
            text="Automatic Regime Detection in German Electricity Prices",
            font=dict(size=18),
            x=0.5,
            xanchor="center",
        ),
        paper_bgcolor="#f7f6f3",
        plot_bgcolor="#ffffff",
        height=1000,
        width=1300,
        legend=dict(
            orientation="h",
            yanchor="bottom",
            y=1.02,
            xanchor="center",
            x=0.5,
            font=dict(size=12),
        ),
    )
    # Force Date tick formatting on the bottom shared axis
    fig.update_xaxes(
    title_text="Date",
    type="date",
    tickformat="%Y",  # Formats ticks to show years (e.g. 2020, 2021, 2022)
    dtick="M12",  # Shows tick every 12 months (every year)
    row=3,
    col=1,
    )
    fig.update_xaxes(showgrid=True, gridwidth=1, gridcolor="rgba(0,0,0,0.1)")
    fig.update_yaxes(
        showgrid=True, gridwidth=1, gridcolor="rgba(0,0,0,0.1)", row=1, col=1
    )
    fig.update_yaxes(
        showgrid=True, gridwidth=1, gridcolor="rgba(0,0,0,0.1)", row=2, col=1
    )

    # Display directly in Jupyter Notebook
    fig.show()


# ── Execution ────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    df = pd.read_pickle(os.path.join(DATA_DIR, "data_part2.pkl"))
    df, q_elev, q_crisis = detect_regimes(df)
    visualize_regimes_plotly(df, q_elev, q_crisis)
    df.to_pickle(os.path.join(DATA_DIR, "data_part7_regimes.pkl"))
    print("\n✓ Saved regimes — ready for Parts 9 & 11")

Detecting market regimes...
  Normal   : 30,641 hrs  avg €  64.2  (mostly 2020)
  Elevated : 19,321 hrs  avg € 122.2  (mostly 2025)
  Crisis   :  5,357 hrs  avg € 266.3  (mostly 2022)
  >> 4,567 of 5,357 crisis hours are in 2022 (85%) — matches the real energy crisis!
Visualizing regimes with Plotly...



✓ Saved regimes — ready for Parts 9 & 11


# Part 8 — Apply the Models (Clean & Fixed)

Trains four uncertainty-aware architectures to output both price forecasts and associated predictive uncertainty.

---

### Model Architecture Overview

1. **DDNN (Baseline):** Distributional Neural Network outputting both mean and variance ($\mu, \sigma^2$).
2. **EvDNN:** Evidential Deep Neural Network capturing epistemic and aleatoric uncertainty in a single forward pass.
3. **VI-DDNN:** Bayesian Neural Network using Variational Inference over weight distributions.
4. **BSSM:** Bayesian State-Space Model relying on time-series dynamic estimation via Kalman Filtering.
5. **VI+CP:** Variational Inference DDNN augmented with Conformal Prediction to guarantee empirical coverage calibration.

### Model Training, Optuna Tuning & Feature Set Comparison

* Reads preprocessed data (Part 2) and selected features (Part 4).
* Tunes DDNN, EvDNN, and VI-DDNN via Optuna on the Validation Set.
* Evaluates models on both **Selected Features** and **All Features**.
* Saves best performing results to `results_part8.npy` for Part 9.
---

### Key Numerical Stability Fix

> **Log-Space Interval Construction:** Target variables are trained and evaluated in **Signed-Log Space**. Upper and lower prediction bounds $[z - q, z + q]$ are constructed in log-space *before* applying the inverse transformation back to Euros. This prevents variance explosions caused by delta-method approximations on large price spikes.

## Methodological Framework & Model Formulations

### 1. Distributional Deep Neural Network (DDNN)

The Distributional Deep Neural Network serves as the parametric distributional baseline. Instead of predicting a single point forecast $\hat{y}_t$, the network models the conditionally Gaussian probability density function $p(y_t \mid \mathbf{x}_t) = \mathcal{N}(\mu(\mathbf{x}_t; \boldsymbol{\theta}), \sigma^2(\mathbf{x}_t; \boldsymbol{\theta}))$, where $\mathbf{x}_t$ represents the input feature vector.  

The hidden representation $\mathbf{h}_L$ at layer $L$ is obtained via stacked non-linear transformations:

$$\mathbf{h}_1 = \text{ReLU}\left(\mathbf{W}_1 \mathbf{x}_t + \mathbf{b}_1\right), \quad \mathbf{h}_l = \text{ReLU}\left(\mathbf{W}_l \mathbf{h}_{l-1} + \mathbf{b}_l\right) \quad \text{for } l = 2, \dots, L$$

The output layer splits into two distinct parameter heads:  

* **Location Head (Mean $\mu$):**

$$\mu(\mathbf{x}_t) = \mathbf{W}_\mu \mathbf{h}_L + b_\mu$$

* **Scale Head (Log-Variance $s = \ln\sigma^2$):**

$$s(\mathbf{x}_t) = \text{clip}\left(\mathbf{W}_s \mathbf{h}_L + b_s, s_{\min}, s_{\max}\right) \implies \sigma^2(\mathbf{x}_t) = \exp(s(\mathbf{x}_t)) + \epsilon$$

Predicting $s_t = \ln\sigma_t^2$ ensures strictly positive variance without requiring constrained optimization. The model is trained by minimizing the Gaussian Negative Log-Likelihood (NLL) loss over $N$ samples:  

$$\mathcal{L}_{\text{NLL}}(\boldsymbol{\theta}) = \frac{1}{2N} \sum_{i=1}^N \left[ \ln\sigma^2(\mathbf{x}_i) + \frac{\left(y_i - \mu(\mathbf{x}_i)\right)^2}{\sigma^2(\mathbf{x}_i)} \right] + \text{const}$$

**Uncertainty Characterization:**  
DDNN quantifies purely aleatoric uncertainty ($\sigma_a^2 = \sigma^2(\mathbf{x}_t)$). Epistemic uncertainty is not captured ($\sigma_e^2 = 0$).  

---

### 2. Evidential Deep Neural Network (EvDNN)

EvDNN parameterizes a higher-order Normal-Inverse-Gamma (NIG) conjugate prior distribution over the unknown parameters $(\mu, \sigma^2)$ of the target distribution $y_t \sim \mathcal{N}(\mu, \sigma^2)$ in a single forward pass.  

The network outputs four evidential parameters $\boldsymbol{\psi} = (\gamma, \nu, \alpha, \beta)$:  

$$\gamma \in \mathbb{R} \quad \text{(Mean)}, \quad \nu > 0 \quad \text{(Evidence Strength)}, \quad \alpha > 1 \quad \text{(Shape)}, \quad \beta > 0 \quad \text{(Scale)}$$

$$\nu = \text{Softplus}\left(\mathbf{W}_\nu \mathbf{h}_L + b_\nu\right) + \epsilon, \quad \alpha = \text{Softplus}\left(\mathbf{W}_\alpha \mathbf{h}_L + b_\alpha\right) + 1 + \epsilon, \quad \beta = \text{Softplus}\left(\mathbf{W}_\beta \mathbf{h}_L + b_\beta\right) + \epsilon$$

The marginal predictive distribution following integration over $(\mu, \sigma^2)$ is a heavy-tailed Student-$t$ distribution $\text{St}\left(y; \gamma, \frac{\beta(1+\nu)}{\nu\alpha}, 2\alpha\right)$. The loss function combines the marginal negative log-likelihood with an evidential regularizer penalized by hyperparameter $\lambda_{\text{DER}}$:  

$$\mathcal{L}_{\text{Evidential}}(\boldsymbol{\theta}) = \mathcal{L}_{\text{NLL}}^{\text{NIG}}(\boldsymbol{\theta}) + \lambda_{\text{DER}} \mathcal{L}_{\text{REG}}(\boldsymbol{\theta})$$

$$\mathcal{L}_{\text{NLL}}^{\text{NIG}} = \frac{1}{2}\ln\left(\frac{\pi}{\nu}\right) - \alpha\ln(\Omega) + (\alpha + 0.5)\ln\left(\nu(y - \gamma)^2 + \Omega\right) + \ln\left(\frac{\Gamma(\alpha)}{\Gamma(\alpha + 0.5)}\right)$$

$$\text{where } \Omega = 2\beta(1 + \nu), \quad \mathcal{L}_{\text{REG}} = \vert{}y - \gamma\vert{} \cdot (2\nu + \alpha)$$

**Uncertainty Decomposition:**  

* **Aleatoric Uncertainty (Data Noise):**

$$\mathbb{E}[\sigma^2] = \frac{\beta}{\alpha - 1}$$

* **Epistemic Uncertainty (Model Doubt):**

$$\text{Var}[\mu] = \frac{\beta}{\nu(\alpha - 1)}$$

* **Total Predictive Variance:**

$$\sigma_{\text{total}}^2 = \frac{\beta(1 + \nu)}{\nu(\alpha - 1)}$$

---

### 3. Variational Inference Deep Neural Network (VI-DDNN)

VI-DDNN is a true Bayesian Neural Network (BNN) where static scalar weights $\mathbf{W}$ are replaced by probability distributions $q_\boldsymbol{\phi}(\mathbf{W}) = \mathcal{N}(\boldsymbol{\mu}_w, \boldsymbol{\sigma}_w^2)$ parameterised by variational parameters $\boldsymbol{\phi} = \{\boldsymbol{\mu}_w, \boldsymbol{\rho}_w\}$ with $\boldsymbol{\sigma}_w = \text{softplus}(\boldsymbol{\rho}_w)$.  

During forward passes, weights are sampled using the reparameterization trick to allow backpropagation:  

$$\mathbf{W} = \boldsymbol{\mu}_w + \boldsymbol{\sigma}_w \odot \boldsymbol{\epsilon}, \quad \boldsymbol{\epsilon} \sim \mathcal{N}(\mathbf{0}, \mathbf{I})$$

Training minimizes the Negative Evidence Lower Bound (ELBO), balancing data fidelity against a Gaussian prior $p(\mathbf{W}) = \mathcal{N}(\mathbf{0}, \sigma_{\text{prior}}^2 \mathbf{I})$ via Kullback-Leibler (KL) divergence:  

$$\mathcal{L}_{\text{ELBO}}(\boldsymbol{\phi}) = \mathbb{E}_{q_\boldsymbol{\phi}(\mathbf{W})}\left[ -\ln p(\mathbf{y} \mid \mathbf{X}, \mathbf{W}) \right] + \mathbb{D}_{\text{KL}}\left(q_\boldsymbol{\phi}(\mathbf{W}) \parallel p(\mathbf{W})\right)$$

$$\mathbb{D}_{\text{KL}}\left(q_\boldsymbol{\phi}(\mathbf{W}) \parallel p(\mathbf{W})\right) = \frac{1}{2} \sum_{j=1}^{\vert{}\mathbf{W}\vert{}} \left[ \frac{\mu_{w,j}^2 + \sigma_{w,j}^2}{\sigma_{\text{prior}}^2} - 1 - \ln\left(\frac{\sigma_{w,j}^2}{\sigma_{\text{prior}}^2}\right) \right]$$

**Bayesian Monte Carlo Inference:**  
At test time, the model executes $T$ Monte Carlo forward passes with independently sampled weight vectors $\{\mathbf{W}^{(t)}\}_{t=1}^T$, producing predictions $\{\hat{\mu}^{(t)}, \hat{\sigma}^{(t)}\}_{t=1}^T$:  

$$\hat{\mu}_{\text{final}} = \frac{1}{T}\sum_{t=1}^T \hat{\mu}^{(t)}$$

* **Epistemic Uncertainty ($\sigma_e^2$):** $\sigma_e^2 = \frac{1}{T}\sum_{t=1}^T \left(\hat{\mu}^{(t)} - \hat{\mu}_{\text{final}}\right)^2$ (Disagreement across weights)
* **Aleatoric Uncertainty ($\sigma_a^2$):** $\sigma_a^2 = \frac{1}{T}\sum_{t=1}^T \left(\hat{\sigma}^{(t)}\right)^2$ (Average expected data noise)
* **Total Uncertainty ($\sigma_{\text{total}}$):** $\sigma_{\text{total}} = \sqrt{\sigma_e^2 + \sigma_a^2}$

---

### 4. VI-DDNN with Log-Space Split Conformal Prediction (VI+CP)

Split Conformal Prediction provides distribution-free, finite-sample $1-\alpha$ coverage guarantees. To avoid exponential variance explosion when mapping back to Euros, residual non-conformity scores $s_i$ are computed strictly in signed-log transform space:  

$$s_i = \left\vert{} z_{\text{true}, i} - z_{\text{pred}, i} \right\vert{}, \quad \text{where } z = \operatorname{sign}(p)\ln(1 + \vert{}p\vert{})$$

For each hour of the day $h \in \{0, \dots, 23\}$, the empirical $1-\alpha$ quantile $q_h$ is calculated over the calibration set $\mathcal{D}_{\text{val}}$:  

$$q_h = \text{Quantile}_{\frac{\lceil(N_h+1)(1-\alpha)\rceil}{N_h}} \left( \left\{ s_i \;\middle\vert{}\; i \in \mathcal{D}_{\text{val}}, \, \text{hour}(i) = h \right\} \right)$$

The conformal prediction bounds are constructed in transform space and mapped back via inverse transformation $\mathcal{T}^{-1}(z) = \operatorname{sign}(z)\left(e^{\vert{}z\vert{}} - 1\right)$:  

$$\mathbf{C}_{\text{Euro}}(\mathbf{x}_{t, h}) = \left[ \mathcal{T}^{-1}\left(z_{\text{pred}, t} - q_h\right), \, \mathcal{T}^{-1}\left(z_{\text{pred}, t} + q_h\right) \right]$$

The implied effective standard deviation in Euro space for evaluation metric computation is defined as:

$$\sigma_{\text{effective}} = \frac{\mathcal{T}^{-1}\left(z_{\text{pred}, t} + q_h\right) - \mathcal{T}^{-1}\left(z_{\text{pred}, t} - q_h\right)}{2 \cdot z_{1 - \alpha/2}}$$

---

### 5. Bayesian State-Space Model (BSSM / Kalman Filter)

The Bayesian State-Space Model decomposes time-series prices into an unobserved dynamic market state $x_t$ (true equilibrium price) governed by a Gaussian random walk, and an observed spot price $y_t$:  

$$\text{State Equation:} \quad x_t = x_{t-1} + w_t, \quad w_t \sim \mathcal{N}(0, Q)$$

$$\text{Observation Equation:} \quad y_t = x_t + v_t, \quad v_t \sim \mathcal{N}(0, R)$$

**Recursive Kalman Filter Updates:**  

* **Prediction Step:**

$$\hat{x}_{t \mid t-1} = \hat{x}_{t-1 \mid t-1}, \quad P_{t \mid t-1} = P_{t-1 \mid t-1} + Q$$

* **Measurement Update Step:**

$$K_t = \frac{P_{t \mid t-1}}{P_{t \mid t-1} + R} \quad \text{(Kalman Gain)}$$

$$\hat{x}_{t \mid t} = \hat{x}_{t \mid t-1} + K_t \left(y_t - \hat{x}_{t \mid t-1}\right), \quad P_{t \mid t} = (1 - K_t) P_{t \mid t-1}$$

**Expectation-Maximization (EM) Tuning:**  
Parameters $\mathbf{\Theta} = \{Q, R\}$ are learned iteratively by maximizing the expected log-likelihood via the EM algorithm:  

$$\hat{R}^{(k+1)} = \frac{1}{N}\sum_{t=1}^N \left(y_t - \hat{x}_{t \mid N}\right)^2, \quad \hat{Q}^{(k+1)} = \frac{1}{N-1}\sum_{t=2}^N \left(\hat{x}_{t \mid N} - \hat{x}_{t-1 \mid N}\right)^2$$

One-step-ahead probabilistic forecasts yield $\mu_{t+1} = \hat{x}_{t+1 \mid t}$ with predictive scale $\sigma_{t+1} = \sqrt{P_{t+1 \mid t}}$.

## Mathematical Formulation of Optuna Hyperparameter Optimization

Optuna employs a state-of-the-art Bayesian Optimization framework known as **Tree-structured Parzen Estimator (TPE)**. Instead of searching the hyperparameter space randomly or exhaustively (grid search), Optuna builds a probabilistic model of the objective function to sequentially select promising hyperparameter configurations.

---

### 1. The Optimization Objective

Let $\lambda \in \Lambda$ represent the vector of hyperparameters to be optimized (e.g., learning rate $lr$, batch size, number of epochs, regularization coefficient $\lambda_{\text{lam}}$, or prior variance $\sigma^2$).

In your EPF pipeline, the objective function $f(\lambda)$ maps a hyperparameter configuration $\lambda$ to a validation loss value—specifically, the Continuous Ranked Probability Score (CRPS) evaluated on the validation dataset:

$$f(\lambda) = \text{CRPS}\left(y_{\text{val}}, \hat{\mu}_{\text{val}}(\lambda), \hat{\sigma}_{\text{val}}(\lambda)\right)$$

The optimization goal is to find the optimal hyperparameter configuration $\lambda^*$ that minimizes this validation score over a bounded domain $\Lambda$:

$$\lambda^* = \arg\min_{\lambda \in \Lambda} f(\lambda)$$

---

### 2. Tree-structured Parzen Estimator (TPE) Algorithm

Standard Bayesian optimization models the posterior probability $P(\text{score} \mid \lambda)$ using Gaussian Processes. In contrast, TPE uses Parzen Estimators (Kernel Density Estimation) to model two separate density functions:

* $l(\lambda)$: The density distribution of hyperparameter configurations that yielded good validation scores (scoring below a specific quantile threshold $\alpha$, where $f(\lambda) < y^*$).
* $g(\lambda)$: The density distribution of hyperparameter configurations that yielded poor validation scores ($f(\lambda) \ge y^*$).

Mathematically, the relationship is defined as:

$$P(\lambda \mid \text{score}) = \begin{cases} l(\lambda) & \text{if } f(\lambda) < y^* \\ g(\lambda) & \text{if } f(\lambda) \ge y^* \end{cases}$$

---

### 3. Expected Improvement (EI) Acquisition Function

To determine the next hyperparameter set $\lambda$ to evaluate, Optuna maximizes an acquisition function called Expected Improvement (EI). EI calculates the expectation of how much better a new configuration can perform compared to a threshold score $y^*$:

$$\text{EI}_{y^*}(\lambda) = \int_{-\infty}^{y^*} (y^* - y) P(y \mid \lambda) \, dy$$

Using Bayes' rule, TPE reformulates the EI acquisition function to show that maximizing Expected Improvement is proportional to the ratio of the "good" density $l(\lambda)$ to the "bad" density $g(\lambda)$:

$$\text{EI}_{y^*}(\lambda) = \frac{\gamma l(\lambda)}{\gamma l(\lambda) + (1 - \gamma) g(\lambda)}$$

* $\gamma$: The quantile threshold representing the fraction of trials considered "successful" (e.g., the top 20% of trials).
* $l(\lambda)$: Built from historical trials that minimized validation CRPS.
* $g(\lambda)$: Built from historical trials with higher CRPS values.

At each iteration step, Optuna samples candidate configurations from $l(\lambda)$, evaluates their ratio $\frac{l(\lambda)}{g(\lambda)}$, and selects the hyperparameter vector $\lambda$ that maximizes this ratio for the next trial.

---

### 4. Sequential Execution in Your Pipeline

The mathematical loop executes iteratively as follows:

1. **Initialization:** Optuna runs initial random trials to populate a history buffer of pairs $\{(\lambda^{(1)}, y^{(1)}), (\lambda^{(2)}, y^{(2)}), \dots, (\lambda^{(t)}, y^{(t)})\}$ where $y^{(i)} = f(\lambda^{(i)})$.
2. **Density Estimation:** It splits the history at the $\gamma$-quantile into $l(\lambda)$ and $g(\lambda)$ distributions.
3. **Candidate Generation:** It maximizes $\frac{l(\lambda)}{g(\lambda)}$ to propose the next optimal parameter set $\lambda^{(t+1)}$.
4. **Evaluation:** The model trains on the training set using $\lambda^{(t+1)}$ and calculates $y^{(t+1)} = f(\lambda^{(t+1)})$ on the validation split.
5. **Convergence:** After $N$ trials (e.g., $N=10$ or $15$), Optuna outputs the absolute minimizer:

$$\lambda^* = \arg\min_{\lambda \in \{\lambda^{(1)}, \dots, \lambda^{(N)}\}} f(\lambda)$$

In [20]:
# Import core research components


compute_metrics = compute_all_metrics


# ── Helper: Log-Space Interval Prediction ─────────────────────────────────
def predict_log_space_interval(model, X_input, scaler_y, confidence=0.90):
    mu_s, sig_s = model.predict(X_input)
    mu_z = scaler_y.inverse_transform(mu_s.reshape(-1, 1)).flatten()
    sig_z = np.abs(sig_s) * scaler_y.scale_[0]

    z_quant = stats.norm.ppf((1 + confidence) / 2)
    z_lower, z_upper = mu_z - z_quant * sig_z, mu_z + z_quant * sig_z

    mu_eur = inv_signed_log(mu_z)
    lower_eur, upper_eur = inv_signed_log(z_lower), inv_signed_log(z_upper)
    sig_eur = (upper_eur - lower_eur) / (2 * z_quant)
    return mu_eur, np.clip(sig_eur, 1e-3, 500.0), mu_z


# ══════════════════════════════════════════════════════════════════════════════
# SECTION 6: OPTUNA HYPERPARAMETER TUNING
# ══════════════════════════════════════════════════════════════════════════════
def tune_hyperparameters(X_tr, y_tr, X_val, y_val, y_val_raw, scaler_y, n_trials=20):
    """
    Tune EACH neural model SEPARATELY so every model gets hyperparameters
    optimized for its own architecture (not the winner's applied to all).
 
    Returns a dict:  {"DDNN": {...}, "EvDNN": {...}, "VI-DDNN": {...}}
    (BSSM and VI+CP are excluded: BSSM tunes itself via EM; VI+CP reuses VI-DDNN.)
    """
    print("\n  Running Optuna — tuning each neural model SEPARATELY...")
 
    def make_objective(model_name):
        def objective(trial):
            lr         = trial.suggest_float("lr", 1e-4, 2e-2, log=True)
            batch_size = trial.suggest_categorical("batch_size", [64, 128, 256])
            epochs     = trial.suggest_int("epochs", 100, 300, step=50)
 
            if model_name == "DDNN":
                model = DDNN(input_dim=X_tr.shape[1], hidden=[128, 64, 32],
                             lr=lr, epochs=epochs, batch_size=batch_size)
            elif model_name == "EvDNN":
                lam = trial.suggest_float("lam", 0.001, 0.05, log=True)
                model = EvDNN(input_dim=X_tr.shape[1], hidden=[128, 64, 32],
                              lr=lr, epochs=epochs, lam=lam, batch_size=batch_size)
            else:  # VI-DDNN
                prior_sig = trial.suggest_float("prior_sigma", 0.5, 2.0)
                model = VIDDNN(input_dim=X_tr.shape[1], hidden=[128, 64],
                               lr=lr, epochs=epochs, n_samples=15,
                               prior_sigma=prior_sig, batch_size=batch_size)
 
            model.fit(X_tr, y_tr, verbose=False)
            mu_s, sig_s = model.predict(X_val)
            mu_z = scaler_y.inverse_transform(mu_s.reshape(-1, 1)).flatten()
            mu_eur = inv_signed_log(mu_z)
            sig_eur = np.abs(sig_s) * scaler_y.scale_[0] * np.exp(np.abs(mu_z))
            return compute_metrics(y_val_raw, mu_eur, sig_eur)["CRPS"]
        return objective
 
    best_by_model = {}
    for model_name in ["DDNN", "EvDNN", "VI-DDNN"]:
        study = optuna.create_study(direction="minimize")
        study.optimize(make_objective(model_name), n_trials=n_trials)
        best_by_model[model_name] = study.best_params
        print(f"    ✓ {model_name:<8} best CRPS={study.best_value:.4f}  params={study.best_params}")
 
    print(f"\n  ✓ Tuned all 3 neural models separately ({n_trials} trials each).")
    return best_by_model
 
 
def train_and_evaluate_pipeline(feature_set, train, val, test, best_by_model, exp_name):
    print(f"\n  --- Training Models on {exp_name} ({len(feature_set)} Features) ---")
    
    y_tr_model = signed_log(train["price"].values)
    y_val_model = signed_log(val["price"].values)
    y_te = test["price"].values
 
    scaler_X, scaler_y = StandardScaler(), StandardScaler()
    X_tr = scaler_X.fit_transform(train[feature_set].values)
    X_val = scaler_X.transform(val[feature_set].values)
    X_te = scaler_X.transform(test[feature_set].values)
    y_tr_s = scaler_y.fit_transform(y_tr_model.reshape(-1, 1)).flatten()
    y_val_s = scaler_y.transform(y_val_model.reshape(-1, 1)).flatten()
 
    results = {}
    def P(model):  # per-model tuned params with safe fallback
        return best_by_model.get(model, {}) if isinstance(best_by_model, dict) else {}
    dd_p, ev_p, vi_p = P("DDNN"), P("EvDNN"), P("VI-DDNN")
 
    # 1. DDNN
    ddnn = DDNN(len(feature_set), hidden=[128, 64, 32], lr=dd_p.get("lr", 0.005), epochs=dd_p.get("epochs", 300), batch_size=dd_p.get("batch_size", 128))
    ddnn.fit(X_tr, y_tr_s, X_val, y_val_s)
    mu_eur, sig_eur, _ = predict_log_space_interval(ddnn, X_te, scaler_y)
    results["DDNN"] = compute_metrics(y_te, mu_eur, sig_eur, label=f"DDNN ({exp_name})")
 
    # 2. EvDNN
    ev = EvDNN(len(feature_set), hidden=[128, 64, 32], lr=ev_p.get("lr", 0.005), epochs=ev_p.get("epochs", 300), lam=ev_p.get("lam", 0.02), batch_size=ev_p.get("batch_size", 128))
    ev.fit(X_tr, y_tr_s, X_val, y_val_s)
    mu_eur, sig_eur, _ = predict_log_space_interval(ev, X_te, scaler_y)
    results["EvDNN"] = compute_metrics(y_te, mu_eur, sig_eur, label=f"EvDNN ({exp_name})")
 
    # 3. VI-DDNN
    vi = VIDDNN(len(feature_set), hidden=[128, 64], lr=vi_p.get("lr", 0.002), epochs=vi_p.get("epochs", 300), n_samples=20, prior_sigma=vi_p.get("prior_sigma", 1.0), batch_size=vi_p.get("batch_size", 128))
    vi.fit(X_tr, y_tr_s, X_val, y_val_s)
    mu_eur, sig_eur, mu_z = predict_log_space_interval(vi, X_te, scaler_y)
    results["VI-DDNN"] = compute_metrics(y_te, mu_eur, sig_eur, label=f"VI-DDNN ({exp_name})")
 
    # 3b. VI+CP
    mu_val_s, _ = vi.predict(X_val)
    mu_val_z = scaler_y.inverse_transform(mu_val_s.reshape(-1, 1)).flatten()
    mu_val_eur = inv_signed_log(mu_val_z)
    cp = ConformalWrapper(coverage=0.90)
    cp.fit(val["price"].values, mu_val_eur, val["hour"].values)
    mu_cp, sig_cp = cp.predict(mu_eur, test["hour"].values)
    results["VI+CP"] = compute_metrics(y_te, mu_cp, sig_cp, label=f"VI+CP ({exp_name})")
 
    # 4. BSSM
    bssm = BSSM(Q_init=50, R_init=300, em_iters=15)
    bssm.fit(y_tr_model)
    mu_z_bssm, sig_z_bssm = bssm.predict_test(y_tr_model, signed_log(test["price"].values))
    z90 = stats.norm.ppf(0.95)
    lower_bssm = inv_signed_log(mu_z_bssm - z90 * sig_z_bssm)
    upper_bssm = inv_signed_log(mu_z_bssm + z90 * sig_z_bssm)
    mu_bssm = inv_signed_log(mu_z_bssm)
    sig_bssm = (upper_bssm - lower_bssm) / (2 * z90)
    results["BSSM"] = compute_metrics(y_te, mu_bssm, sig_bssm, label=f"BSSM ({exp_name})")
 
    return results


# ── Main Pipeline Execution ────────────────────────────────────────────────
def main():
    print("\n" + "█" * 65)
    print("  PART 8 — MODEL TRAINING & OPTUNA TUNING COMPARISON")
    print("█" * 65)
 
    # 1. Load Datasets
    df = pd.read_pickle(os.path.join(DATA_DIR, "data_part2.pkl"))
    with open(os.path.join(DATA_DIR, "features_part2.json")) as f:
        all_features = json.load(f)
    with open(os.path.join(DATA_DIR, "selected_part4.json")) as f:
        selected_features = json.load(f)
 
    # 2. Chronological Split
    train = df[df["datetime"] <= "2024-04-30 23:00"].copy()
    val = df[(df["datetime"] >= "2024-05-01") & (df["datetime"] <= "2025-04-30 23:00")].copy()
    test = df[df["datetime"] >= "2025-05-01"].copy()
 
    scaler_X, scaler_y = StandardScaler(), StandardScaler()
    X_tr_sel = scaler_X.fit_transform(train[selected_features].values)
    y_tr_s = scaler_y.fit_transform(signed_log(train["price"].values).reshape(-1, 1)).flatten()
    X_val_sel = scaler_X.transform(val[selected_features].values)
    y_val_s = scaler_y.transform(signed_log(val["price"].values).reshape(-1, 1)).flatten()
 
    # 3. Hyperparameter Tuning using Optuna
    best_by_model = tune_hyperparameters(
        X_tr_sel,
        y_tr_s,
        X_val_sel,
        y_val_s,
        val["price"].values,
        scaler_y,
        n_trials=20          # 20 trials PER MODEL (3 models = 60 total)
    )
 
    # 4. Run Both Pipeline Configurations
    sel_name = f"Selected ({len(selected_features)})"
    all_name = f"All Candidate ({len(all_features)})"
 
    results_selected = train_and_evaluate_pipeline(selected_features, train, val, test, best_by_model, sel_name)
    results_all = train_and_evaluate_pipeline(all_features, train, val, test, best_by_model, all_name)
 
    # 5. Display Comparison Table
    print("\n" + "=" * 80)
    print("  FEATURE SET COMPARISON TABLE (Test Set Results)")
    print("=" * 80)
    print(f"  {'Model / Feature Set':<28}{'MAE':>8}{'RMSE':>8}{'CRPS':>8}{'PICP_90':>9}{'MPIW_90':>9}{'MAACE':>8}")
    print("  " + "-" * 78)
 
    for name in results_selected.keys():
        r_sel = results_selected[name]
        r_all = results_all[name]
        print(f"  {name + ' (' + sel_name + ')':<28}{r_sel['MAE']:>8.2f}{r_sel['RMSE']:>8.2f}{r_sel['CRPS']:>8.2f}{r_sel['PICP_90']:>9.1%}{r_sel['MPIW_90']:>9.2f}{r_sel['MAACE']:>7.2f}%")
        print(f"  {name + ' (' + all_name + ')':<28}{r_all['MAE']:>8.2f}{r_all['RMSE']:>8.2f}{r_all['CRPS']:>8.2f}{r_all['PICP_90']:>9.1%}{r_all['MPIW_90']:>9.2f}{r_all['MAACE']:>7.2f}%")
        print("  " + "-" * 78)

    # FIXED: Save BOTH Selected and All Candidate results into results_part8.npy
    np.save(
        os.path.join(DATA_DIR, "results_part8.npy"),
        {
            "results_selected": results_selected,
            "results_all": results_all,
            "y_test": test["price"].values,
            "test_dates": test["datetime"].values,
        },
        allow_pickle=True,
    )
    print("\n  ✓ Saved both feature set results to results_part8.npy — Ready for Part 9!")

    with open(os.path.join(DATA_DIR, "results_part8.json"), "w") as f:
        json.dump(best_by_model, f)
    print("  ✓ Saved per-model tuned hyperparameters to results_part8.json")
 
 
if __name__ == "__main__":
    main()


█████████████████████████████████████████████████████████████████
  PART 8 — MODEL TRAINING & OPTUNA TUNING COMPARISON
█████████████████████████████████████████████████████████████████

  Running Optuna — tuning each neural model SEPARATELY...
    ✓ DDNN     best CRPS=12.3051  params={'lr': 0.0008274803923368282, 'batch_size': 128, 'epochs': 100}
    ✓ EvDNN    best CRPS=16.8888  params={'lr': 0.002006906273275508, 'batch_size': 128, 'epochs': 250, 'lam': 0.003851460995733125}
    ✓ VI-DDNN  best CRPS=15.6046  params={'lr': 0.001696657436973614, 'batch_size': 256, 'epochs': 300, 'prior_sigma': 1.9114387304106248}

  ✓ Tuned all 3 neural models separately (20 trials each).

  --- Training Models on Selected (15) (15 Features) ---

done. Final NLL=-1.6772.. 

done. Final loss=-1.2156.. 

done. Final ELBO=-0.8755amples/pass)... 

done.  Q=0.01  R=0.57M iterations)... 

  --- Training Models on All Candidate (35) (35 Features) ---

done. Final NLL=-1.3699.. 

done. Final loss=-1.5643.. 



# Part 9 — Show Results (Interactive Plotly Dashboard)

Displays the model metrics comparison table and generates an interactive Plotly dashboard to evaluate all five forecasts against actual market prices.

---

### 1. MAE (Mean Absolute Error)
* **What it measures:** The average magnitude of absolute errors between the predicted point forecast $\hat{y}_t$ and the true market price $y_t$. It treats all errors equally without heavily penalizing extreme spikes.
* **Mathematical Equation:**

$$\text{MAE} = \frac{1}{N} \sum_{t=1}^N \vert y_t - \hat{y}_t \vert$$

---

### 2. RMSE (Root Mean Squared Error)
* **What it measures:** The square root of the average squared errors. Because errors are squared before being averaged, RMSE gives a much higher weight to large errors and extreme price spikes, making it sensitive to outliers.
* **Mathematical Equation:**

$$\text{RMSE} = \sqrt{\frac{1}{N} \sum_{t=1}^N (y_t - \hat{y}_t)^2}$$

---

### 3. CRPS (Continuous Ranked Probability Score)
* **What it measures:** A strict generalization of the absolute error for probabilistic forecasts. It evaluates the entire predictive cumulative distribution function $F_t$ against the true observation $y_t$, rewarding both sharpness and calibration simultaneously.
* **Mathematical Equation:**

$$\text{CRPS}(F_t, y_t) = \int_{-\infty}^{\infty} \left( F_t(x) - \mathbb{I}(x \ge y_t) \right)^2 dx$$

*(For a Gaussian distribution $\mathcal{N}(\mu, \sigma^2)$, this simplifies analytically to:)*

$$\text{CRPS} = \sigma \left[ z \left(2\Phi(z) - 1\right) + 2\phi(z) - \frac{1}{\sqrt{\pi}} \right], \quad \text{where } z = \frac{y_t - \mu}{\sigma}$$

---

### 4. PICP_90 (Prediction Interval Coverage Probability at 90%)
* **What it measures:** The empirical frequency with which the true price falls inside the model's 90% predicted lower and upper bounds ($[L_t, U_t]$). For a well-calibrated model, this should closely approach $0.90$ (90%).
* **Mathematical Equation:**

$$\text{PICP}_{90} = \frac{1}{N} \sum_{t=1}^N \mathbb{I}\left( y_t \in [L_t, U_t] \right)$$

---

### 5. MPIW_90 (Mean Prediction Interval Width at 90%)
* **What it measures:** The average span or width of the 90% prediction interval. Narrower intervals (lower MPIW) indicate sharper, more informative forecasts, provided the PICP target is still met.
* **Mathematical Equation:**

$$\text{MPIW}_{90} = \frac{1}{N} \sum_{t=1}^N \left( U_t - L_t \right)$$

---

### 6. MAACE (Mean Absolute Marginal Calibration Error)
* **What it measures:** The average deviation between empirical coverage and nominal confidence levels across multiple probability thresholds (e.g., from 10% to 90%). It quantifies overall calibration performance across the entire distribution.
* **Mathematical Equation:**

$$\text{MAACE} = \frac{1}{M} \sum_{m=1}^M \left| \text{Nominal Level}_m - \text{Empirical Coverage}_m \right|$$

In [21]:
DOCS_DIR = DATA_DIR

pio.renderers.default = "iframe"
def show_results_tables(results_selected, results_all):
    print("\n" + "="*84)
    print("  RESULTS TABLE — SELECTED FEATURES TEST SET (May 2025 – End)")
    print("="*84)
    print(f"  {'Model':<12}{'MAE':>9}{'RMSE':>9}{'CRPS':>9}"
          f"{'PICP_90':>10}{'MPIW_90':>10}{'MAACE':>9}")
    print("  " + "-"*76)
    for name, m in results_selected.items():
        print(f"  {name:<12}{m['MAE']:>9.2f}{m['RMSE']:>9.2f}{m['CRPS']:>9.2f}"
              f"{m['PICP_90']:>9.1%}{m['MPIW_90']:>10.2f}{m['MAACE']:>8.2f}%")
    print("="*84)

    print("\n" + "="*84)
    print("  RESULTS TABLE — ALL CANDIDATE FEATURES TEST SET (May 2025 – End)")
    print("="*84)
    print(f"  {'Model':<12}{'MAE':>9}{'RMSE':>9}{'CRPS':>9}"
          f"{'PICP_90':>10}{'MPIW_90':>10}{'MAACE':>9}")
    print("  " + "-"*76)
    for name, m in results_all.items():
        print(f"  {name:<12}{m['MAE']:>9.2f}{m['RMSE']:>9.2f}{m['CRPS']:>9.2f}"
              f"{m['PICP_90']:>9.1%}{m['MPIW_90']:>10.2f}{m['MAACE']:>8.2f}%")
    print("="*84)


def plot_predictions_plotly(results_selected, results_all, y_test, test_dates=None, save_path=os.path.join(DOCS_DIR, 'part9_predictions.html')):
    print("\nGenerating high-performance WebGL Plotly dashboard for both feature sets...")
    
    combined_results = {}
    for name, m in results_selected.items():
        combined_results[f"{name} (Selected Features)"] = m
    for name, m in results_all.items():
        combined_results[f"{name} (All Candidate Features)"] = m

    models = list(combined_results.keys())
    n_show = len(y_test)

    colors = {
        'DDNN (Selected Features)': '#378ADD',
        'EvDNN (Selected Features)': '#D85A30',
        'VI-DDNN (Selected Features)': '#1D9E75',
        'VI+CP (Selected Features)': '#7F77DD',
        'BSSM (Selected Features)': '#BA7517',
        'DDNN (All Candidate Features)': '#1B4F72',
        'EvDNN (All Candidate Features)': '#B03A2E',
        'VI-DDNN (All Candidate Features)': '#117A65',
        'VI+CP (All Candidate Features)': '#512E5F',
        'BSSM (All Candidate Features)': '#7E5109'
    }

    x_vals = test_dates[:n_show] if test_dates is not None else list(range(n_show))

    subplot_titles = [
        f"<b>{name}</b> — MAE: €{m['MAE']:.1f} | CRPS: {m['CRPS']:.2f} | MAACE: {m['MAACE']:.1f}%"
        for name, m in combined_results.items()
    ]
    
    fig = make_subplots(
        rows=len(models),
        cols=1,
        shared_xaxes=True,
        vertical_spacing=0.03,
        subplot_titles=subplot_titles
    )

    for i, name in enumerate(models, start=1):
        m = combined_results[name]
        mu = m['mu'][:n_show]
        
        # Fallback handling for confidence bounds
        if 'lower' in m and 'upper' in m:
            lo = m['lower'][:n_show]
            hi = m['upper'][:n_show]
        else:
            sig = m.get('sigma', np.abs(m['mu'] - y_test[:n_show]))[:n_show]
            lo = mu - 1.645 * sig
            hi = mu + 1.645 * sig

        c = colors.get(name, '#378ADD')

        rgb_tuple = tuple(int(c.lstrip('#')[j:j+2], 16) for j in (0, 2, 4))
        fill_color = f"rgba({rgb_tuple[0]}, {rgb_tuple[1]}, {rgb_tuple[2]}, 0.22)"

        # 1. Upper Bound
        fig.add_trace(
            go.Scattergl(
                x=x_vals, y=hi,
                mode='lines',
                line=dict(width=0),
                showlegend=False,
                hoverinfo='skip'
            ),
            row=i, col=1
        )

        # 2. Lower Bound + Shaded Fill
        fig.add_trace(
            go.Scattergl(
                x=x_vals, y=lo,
                mode='lines',
                line=dict(width=0),
                fill='tonexty',
                fillcolor=fill_color,
                name='90% Band',
                showlegend=(i == 1)
            ),
            row=i, col=1
        )

        # 3. Actual Price Line
        fig.add_trace(
            go.Scattergl(
                x=x_vals, y=y_test[:n_show],
                mode='lines',
                line=dict(color='#222222', width=1.1),
                name='Actual Price',
                showlegend=(i == 1)
            ),
            row=i, col=1
        )

        # 4. Forecast Mean Line
        fig.add_trace(
            go.Scattergl(
                x=x_vals, y=mu,
                mode='lines',
                line=dict(color=c, width=1.8),
                name=f'{name} Forecast',
                showlegend=False
            ),
            row=i, col=1
        )

        fig.update_yaxes(title_text="€/MWh", row=i, col=1, gridcolor='#E8E8E8')

    fig.update_layout(
        height=280 * len(models),
        width=1300,
        title_text="<b>Model Predictions vs. Actual Prices: Selected vs. All Candidate Features</b>",
        title_x=0.5,
        template="plotly_white",
        legend=dict(orientation="h", yanchor="bottom", y=1.01, xanchor="right", x=1),
        hovermode="x unified"
    )

    fig.update_xaxes(gridcolor='#E8E8E8')
    fig.update_xaxes(title_text="Time / Hour", row=len(models), col=1)

    fig.write_html(save_path)
    print(f"  ✓ Saved interactive Plotly dashboard to -> {save_path}")
    
    try:
        fig.show()
    except Exception:
        print(f"  (Note: Open '{save_path}' directly in your web browser to view the interactive plot)")


if __name__ == "__main__":
    results_path = os.path.join(DOCS_DIR, 'results_part8.npy')
    if not os.path.exists(results_path):
        results_path = 'results_part8.npy'
        
    data = np.load(results_path, allow_pickle=True).item()
    
    # Safely load both feature sets from the saved numpy dictionary
    results_selected = data.get('results_selected', data.get('results', {}))
    results_all = data.get('results_all', {})
    y_test = data['y_test']
    test_dates = data.get('test_dates', None)

    show_results_tables(results_selected, results_all)
    plot_predictions_plotly(results_selected, results_all, y_test, test_dates)
    print("\n✓ Done — view dashboard above or open part9_predictions.html in your browser")


  RESULTS TABLE — SELECTED FEATURES TEST SET (May 2025 – End)
  Model             MAE     RMSE     CRPS   PICP_90   MPIW_90    MAACE
  ----------------------------------------------------------------------------
  DDNN            15.81    28.69    12.43    78.9%     61.06    5.46%
  EvDNN           16.97    37.34    37.14    99.9%    486.24   37.29%
  VI-DDNN         19.66    27.09    22.99    98.4%    260.97   23.80%
  VI+CP           19.66    27.09    14.02    90.0%     76.92    1.27%
  BSSM            33.14    47.86    28.10    59.8%     71.12   13.03%

  RESULTS TABLE — ALL CANDIDATE FEATURES TEST SET (May 2025 – End)
  Model             MAE     RMSE     CRPS   PICP_90   MPIW_90    MAACE
  ----------------------------------------------------------------------------
  DDNN            14.18    27.06    12.55    75.5%     80.23    7.31%
  EvDNN            8.56    28.50    28.92   100.0%    391.53   43.52%
  VI-DDNN         15.84    22.70    16.69    98.3%    175.85   17.12%
  VI+CP  


✓ Done — view dashboard above or open part9_predictions.html in your browser


# Part 10 — Result Analysis (Plotly Interactive Dashboard)

Interactive visualizations for exploring model performance and uncertainty calibration.

---

### Interactive Plot Specifications

1. **Side-by-Side Metric Bar Charts:** Compares MAE, CRPS, and MAACE across all models, complete with hover details for granular inspection.
2. **Interactive Calibration Curve Diagnostic:** Plots empirical coverage against nominal confidence levels, featuring a reference diagonal line for instant visual assessment of under- or over-confidence.

In [22]:

pio.renderers.default = "iframe"
def analyze_results_plotly(results_selected, results_all, save_path=os.path.join(DATA_DIR, "part10_analysis.html")):
    print("Analyzing results for 16 and 35 features with Plotly...")
    models = ['DDNN', 'EvDNN', 'VI-DDNN', 'VI+CP', 'BSSM']
    
    colors = {
        'DDNN': '#378ADD',
        'EvDNN': '#D85A30',
        'VI-DDNN': '#1D9E75',
        'VI+CP': '#7F77DD',
        'BSSM': '#BA7517'
    }

    fig = make_subplots(
        rows=2, cols=3,
        subplot_titles=(
            "1. MAE (€/MWh) — Accuracy",
            "2. CRPS — Distribution Quality",
            "3. MAACE (%) — Calibration Error",
            "4. Calibration Curve (Selected vs. All Candidate Features)"
        ),
        specs=[
            [{"type": "xy"}, {"type": "xy"}, {"type": "xy"}],
            [{"colspan": 3, "type": "xy"}, None, None]
        ],
        vertical_spacing=0.18,
        horizontal_spacing=0.08
    )

    # ── 1. Three Metric Bar Charts (Grouped by Feature Set) ───────────────────
    metrics_config = [
        ('MAE', 1, 1, 'MAE (€/MWh)', True),
        ('CRPS', 1, 2, 'CRPS', True),
        ('MAACE', 1, 3, 'MAACE (%)', True)
    ]

    for metric, row, col, label, lower_better in metrics_config:
        # Selected Features Bar Group
        vals_sel = [results_selected[m][metric] for m in models if m in results_selected]
        fig.add_trace(
            go.Bar(
                x=models,
                y=vals_sel,
                name='Selected (16 Features)',
                marker_color=[colors.get(m, '#888') for m in models],
                text=[f"{v:.1f}" for v in vals_sel],
                textposition='outside',
                legendgroup='sel',
                showlegend=(row == 1 and col == 1),
                hovertemplate="Model: %{x}<br>Selected: %{y:.2f}<extra></extra>"
            ),
            row=row, col=col
        )

        # All Candidate Features Bar Group (Distinguished via pattern shape)
        vals_all = [results_all[m][metric] for m in models if m in results_all]
        fig.add_trace(
            go.Bar(
                x=models,
                y=vals_all,
                name='All Candidate (35 Features)',
                marker_color=[colors.get(m, '#888') for m in models],
                marker_pattern_shape="x",
                text=[f"{v:.1f}" for v in vals_all],
                textposition='outside',
                legendgroup='all',
                showlegend=(row == 1 and col == 1),
                hovertemplate="Model: %{x}<br>All Candidate: %{y:.2f}<extra></extra>"
            ),
            row=row, col=col
        )
        
        fig.update_xaxes(title_text="", row=row, col=col)
        fig.update_yaxes(title_text=label, row=row, col=col)

    fig.update_layout(barmode='group')

    # ── 2. Interactive Calibration Curves (Bottom Row Spanning All Columns) ───
    feature_sets = [
        ('Selected (16 Features)', results_selected, 'solid'),
        ('All Candidate (35 Features)', results_all, 'dash')
    ]
    
    for fs_label, res_dict, dash_style in feature_sets:
        for name in models:
            if name not in res_dict:
                continue
            m = res_dict[name]
            levels = np.array(m['levels']) * 100
            picps = np.array(m['picps']) * 100
            col = colors.get(name, '#888')

            fig.add_trace(
                go.Scatter(
                    x=levels,
                    y=picps,
                    mode='lines+markers',
                    line=dict(color=col, width=2, dash=dash_style),
                    marker=dict(size=5),
                    name=f"{name} ({fs_label})",
                    hovertemplate=f"<b>%{{fullData.name}}</b><br>Claimed Confidence: %{{x}}%<br>Actual Coverage: %{{y:.1f}}%<extra></extra>"
                ),
                row=2, col=1
            )

    # Ideal calibration reference line (y = x)
    fig.add_trace(
        go.Scatter(
            x=[10, 90],
            y=[10, 90],
            mode='lines',
            line=dict(color="black", dash="dash", width=1.5),
            name="Perfect calibration",
            hoverinfo="skip"
        ),
        row=2, col=1
    )

    # Shaded acceptable calibration zone
    fig.add_trace(
        go.Scatter(
            x=[10, 90, 90, 10],
            y=[5, 85, 95, 15],
            fill='toself',
            fillcolor='rgba(128, 128, 128, 0.08)',
            line=dict(color='rgba(255,255,255,0)'),
            name="Acceptable Zone",
            hoverinfo="skip"
        ),
        row=2, col=1
    )

    # Layout configurations with spacing fixes for legend and title overlap
    fig.update_layout(
        title=dict(
            text="<b>Result Analysis — Accuracy and Uncertainty Calibration Dashboard (16 vs 35 Features)</b>",
            font=dict(size=16),
            x=0.5,
            xanchor="center",
            y=0.96
        ),
        paper_bgcolor="#f7f6f3",
        plot_bgcolor="#ffffff",
        height=1050,
        width=1350,
        margin=dict(t=220, l=60, r=60, b=60),
        legend=dict(
            orientation="h",
            yanchor="bottom",
            y=1.06,
            xanchor="center",
            x=0.5,
            font=dict(size=10)
        )
    )

    fig.update_xaxes(title_text="Confidence the model claims (%)", row=2, col=1)
    fig.update_yaxes(title_text="How often the truth was actually in the band (%)", row=2, col=1)

    fig.update_xaxes(showgrid=True, gridwidth=1, gridcolor="rgba(0,0,0,0.1)")
    fig.update_yaxes(showgrid=True, gridwidth=1, gridcolor="rgba(0,0,0,0.1)")

    # Save to HTML and render
    fig.write_html(save_path)
    print(f"  ✓ Saved interactive report to {save_path}")
    fig.show()


def print_analysis(results_selected, results_all):
    print("\n  PLAIN-LANGUAGE ANALYSIS (16 vs 35 Features):")
    best_mae_sel = min(results_selected, key=lambda m: results_selected[m]['MAE'])
    best_mae_all = min(results_all, key=lambda m: results_all[m]['MAE'])
    print(f"  • Best MAE (Selected 16): {best_mae_sel} at €{results_selected[best_mae_sel]['MAE']:.1f}")
    print(f"  • Best MAE (All 35): {best_mae_all} at €{results_all[best_mae_all]['MAE']:.1f}")
    
    best_cal_sel = min(results_selected, key=lambda m: results_selected[m]['MAACE'])
    best_cal_all = min(results_all, key=lambda m: results_all[m]['MAACE'])
    print(f"  • Best Calibration MAACE (Selected 16): {best_cal_sel} at {results_selected[best_cal_sel]['MAACE']:.1f}%")
    print(f"  • Best Calibration MAACE (All 35): {best_cal_all} at {results_all[best_cal_all]['MAACE']:.1f}%")


# ── Run this part ──────────────────────────────────────────────────────────
if __name__ == "__main__":
    data = np.load(os.path.join(DATA_DIR, "results_part8.npy"), allow_pickle=True).item()
    results_selected = data.get('results_selected', data.get('results', {}))
    results_all = data.get('results_all', {})
    
    analyze_results_plotly(results_selected, results_all, os.path.join(DATA_DIR, "part10_analysis.html"))
    print_analysis(results_selected, results_all)
    print("\n✓ Done — interactive dashboard saved to part10_analysis.html")

Analyzing results for 16 and 35 features with Plotly...
  ✓ Saved interactive report to part10_analysis.html



  PLAIN-LANGUAGE ANALYSIS (16 vs 35 Features):
  • Best MAE (Selected 16): DDNN at €15.8
  • Best MAE (All 35): EvDNN at €8.6
  • Best Calibration MAACE (Selected 16): VI+CP at 1.3%
  • Best Calibration MAACE (All 35): VI+CP at 5.4%

✓ Done — interactive dashboard saved to part10_analysis.html


## Mathematical Formulation of Part 11: Per-Regime Feature Analysis

To rigorously analyze how market drivers change across different economic conditions, Part 11 formalizes feature importance through a multi-model ensemble conditioned on market regimes.  

---

### 1. Regime Partitioning

Let the time-indexed dataset be denoted by $\mathcal{D} = \{(\mathbf{x}_t, y_t)\}_{t=1}^T$, where $\mathbf{x}_t \in \mathbb{R}^d$ represents the feature vector at hour $t$, and $y_t$ is the day-ahead electricity price.

The historical data is partitioned into $K=3$ discrete market regimes ($r \in \{0, \text{Normal}; 1, \text{Elevated}; 2, \text{Crisis}\}$) based on price thresholds:

$$\mathcal{D}^{(r)} = \{(\mathbf{x}_t, y_t) \in \mathcal{D} \mid y_t \in \mathcal{I}^{(r)}\}$$

where $\mathcal{I}^{(r)}$ represents the price interval defining regime $r$ (e.g., normal pricing bounds, elevated pricing bands, and extreme crisis pricing peaks).

---

### 2. Feature Grouping Mapping

The total set of candidate features is partitioned into $M=3$ distinct economic categories: $\mathcal{G}_1$ (Conventional Generation), $\mathcal{G}_2$ (Renewables), and $\mathcal{G}_3$ (Demand).

Let $\mathcal{F}_g \subset \{1, 2, \dots, d\}$ denote the index subset of features belonging to economic group $g \in \{1, 2, 3\}$.

---

### 3. Dual-Metric Feature Importance Score

Within each regime $\mathcal{D}^{(r)}$, the marginal importance of feature $j$ (where $j \in \{1, \dots, d\}$) is evaluated using a normalized blend of a non-linear information-theoretic measure and a tree-based ensemble importance score.

#### A. Mutual Information (MI)
Mutual Information measures general non-linear statistical dependencies between feature $X_j^{(r)}$ and the target price $Y^{(r)}$:

$$\text{MI}(X_j^{(r)}; Y^{(r)}) = \iint p(x, y) \log \frac{p(x, y)}{p(x)p(y)} \, dx \, dy$$

The MI scores across all features are min-max normalized within regime $r$:

$$\text{MI}_{\text{norm}}(j) = \frac{\text{MI}(X_j^{(r)}; Y^{(r)})}{\max_{k} \text{MI}(X_k^{(r)}; Y^{(r)}) + \epsilon}$$

where $\epsilon = 10^{-9}$ prevents division by zero.

#### B. Random Forest Feature Importance (RFI)
A Random Forest Regressor is trained independently on regime data $\mathcal{D}^{(r)}$. The importance $\text{RFI}(j)$ is derived from the normalized total reduction of mean squared error (impurity) brought about by feature $j$ across all ensemble decision trees $T$:

$$\text{RFI}(j) = \frac{1}{\vert{}T\vert{}} \sum_{T} \sum_{\text{node} \in T} \Delta \text{Impurity}(\text{node}, j)$$

This is similarly normalized:

$$\text{RFI}_{\text{norm}}(j) = \frac{\text{RFI}(j)}{\max_{k} \text{RFI}(k) + \epsilon}$$

#### C. Hybrid Feature Importance Score
The final importance score $I_j^{(r)}$ for feature $j$ under regime $r$ is defined as the arithmetic mean of the normalized metrics:

$$I_j^{(r)} = \frac{\text{MI}_{\text{norm}}(j) + \text{RFI}_{\text{norm}}(j)}{2}$$

---

### 4. Economic Group Aggregation & Share Calculation

To evaluate macro-level shifts, the individual feature importances are aggregated into their respective economic groups $\mathcal{G}_g$:

$$\Phi_g^{(r)} = \sum_{j \in \mathcal{F}_g} I_j^{(r)}$$

Finally, the percentage share $\text{Share}_g^{(r)}$ of total market importance belonging to group $g$ within regime $r$ is expressed as:

$$\text{Share}_g^{(r)} = \frac{\Phi_g^{(r)}}{\sum_{m=1}^3 \Phi_m^{(r)}} \times 100\%$$

---

### Mathematical Implication of the Findings

When transitioning from the Normal Regime ($r=0$) to the Crisis Regime ($r=2$), empirical execution reveals:

$$\text{Share}_{\text{Renewable}}^{(2)} \gg \text{Share}_{\text{Renewable}}^{(0)}$$

Mathematically, this demonstrates that the sensitivity of wholesale market clearing prices with respect to zero-marginal-cost renewable penetration is non-linear and amplifies dramatically during periods of fossil-fuel scarcity.

In [23]:
def per_regime_analysis(df):
    print("Analyzing feature importance per regime...")

    # Group features into 3 economic buckets
    groups = {
        'Conventional':  ['residual_load', 'other_gen'],
        'Renewable':     ['wind_offshore', 'wind_onshore', 'solar', 'pv_wind'],
        'Demand':        ['load_forecast', 'total_gen'],
    }
    all_feats = [f for g in groups.values() for f in g if f in df.columns]
    names = {0: 'Normal', 1: 'Elevated', 2: 'Crisis'}
    group_share = {}

    for r in [0, 1, 2]:
        sub = df[df['regime'] == r]
        X = StandardScaler().fit_transform(sub[all_feats].fillna(0).values)
        y = sub['price'].values
        
        # Importance = average of mutual information + random forest
        mi = mutual_info_regression(X, y, random_state=42)
        mi = mi / (mi.max() + 1e-9)
        rf = RandomForestRegressor(n_estimators=80, max_depth=7,
                                   min_samples_leaf=30, random_state=42, n_jobs=-1)
        rf.fit(X, y)
        rfi = rf.feature_importances_ / (rf.feature_importances_.max() + 1e-9)
        imp = dict(zip(all_feats, (mi + rfi) / 2))

        # Sum importance by group, normalise to shares
        gs = {g: sum(imp[f] for f in fl if f in imp) for g, fl in groups.items()}
        tot = sum(gs.values())
        group_share[r] = {g: v / tot * 100 for g, v in gs.items()}

        print(f"\n  {names[r]} regime (avg €{y.mean():.0f}):")
        for g, v in group_share[r].items():
            print(f"    {g:14s}: {v:5.1f}%")

    return group_share, names


def visualize_per_regime_plotly(group_share, names, save_path=os.path.join(DATA_DIR, "part11_per_regime.html")):
    print("\nVisualizing per-regime importance with Plotly...")
    regimes = [0, 1, 2]
    
    conv = [group_share[r]['Conventional'] for r in regimes]
    ren  = [group_share[r]['Renewable'] for r in regimes]
    dem  = [group_share[r]['Demand'] for r in regimes]

    regime_labels = [f"{names[r]}<br>(avg €{[64, 111, 236][r]})" for r in regimes]

    fig = go.Figure()

    # Add Stacked Bars
    fig.add_trace(go.Bar(
        x=regime_labels,
        y=conv,
        name='Conventional (gas/coal need)',
        marker_color='#5F5E5A',
        hovertemplate="<b>%{x}</b><br>Conventional Share: %{y:.1f}%<extra></extra>"
    ))

    fig.add_trace(go.Bar(
        x=regime_labels,
        y=ren,
        name='Renewable (wind/solar)',
        marker_color='#639922',
        hovertemplate="<b>%{x}</b><br>Renewable Share: %{y:.1f}%<extra></extra>"
    ))

    fig.add_trace(go.Bar(
        x=regime_labels,
        y=dem,
        name='Demand',
        marker_color='#85B7EB',
        hovertemplate="<b>%{x}</b><br>Demand Share: %{y:.1f}%<extra></extra>"
    ))

    # Layout configuration
    fig.update_layout(
        barmode='stack',
        title=dict(
            text=f"<b>What drives price changes by regime</b><br><sup>Renewables jump from {ren[0]:.0f}% (normal) to {ren[2]:.0f}% (crisis)</sup>",
            font=dict(size=16),
            x=0.5,
            xanchor="center"
        ),
        xaxis_title="<b>Market Regime & Average Price</b>",
        yaxis_title="<b>Share of feature importance (%)</b>",
        yaxis=dict(range=[0, 100]),
        paper_bgcolor="#f7f6f3",
        plot_bgcolor="#ffffff",
        height=750,
        width=1100,
        legend=dict(
            orientation="h",
            yanchor="bottom",
            y=-0.25,
            xanchor="center",
            x=0.5
        )
    )

    fig.update_xaxes(showgrid=False)
    fig.update_yaxes(showgrid=True, gridwidth=1, gridcolor="rgba(0,0,0,0.1)")

    # Save to HTML and display
    fig.write_html(save_path)
    print(f"  ✓ Saved interactive report to {save_path}")
    fig.show()


# ── Run this part ──────────────────────────────────────────────────────────
if __name__ == "__main__":
    df = pd.read_pickle(os.path.join(DATA_DIR, "data_part7_regimes.pkl"))
    group_share, names = per_regime_analysis(df)
    visualize_per_regime_plotly(group_share, names, os.path.join(DATA_DIR, "part11_per_regime.html"))
    print("\n✓ Done — interactive dashboard saved to part11_per_regime.html")

Analyzing feature importance per regime...

  Normal regime (avg €64):
    Conventional  :  64.4%
    Renewable     :  25.6%
    Demand        :  10.0%

  Elevated regime (avg €122):
    Conventional  :  70.5%
    Renewable     :  23.5%
    Demand        :   6.0%

  Crisis regime (avg €266):
    Conventional  :  39.5%
    Renewable     :  53.0%
    Demand        :   7.5%

Visualizing per-regime importance with Plotly...
  ✓ Saved interactive report to part11_per_regime.html



✓ Done — interactive dashboard saved to part11_per_regime.html


## Mathematical Formulation of Part 12: Battery Profit & Uncertainty-Aware Trading

Part 12 models the economic value of uncertainty quantification (UQ) within a real-world energy storage application—specifically, lithium-ion battery arbitrage in the German day-ahead electricity market. It contrasts a deterministic trading strategy (Trader A) with an uncertainty-aware risk-mitigating strategy (Trader B).

---

### 1. Battery Arbitrage Optimization Framework

Consider a battery energy storage system operating over a 24-hour day $d$. Let the true day-ahead wholesale electricity prices be denoted by vector $\mathbf{y}_{\text{true}}^{(d)} = [y_{t,1}, y_{t,2}, \dots, y_{t,24}]^\top$.

A probabilistic forecasting model (such as VI-DDNN) provides a predicted price vector $\hat{\boldsymbol{\mu}}^{(d)}$ and an epistemic/aleatoric uncertainty vector $\hat{\boldsymbol{\sigma}}^{(d)}$ for each hour.

#### Optimal Intraday Scheduling (The Ideal Arbitrage):
A rational battery operator identifies the optimal charging (buying) hour $h_{\text{buy}}$ and discharging (selling) hour $h_{\text{sell}}$ based on predicted prices:

$$h_{\text{buy}} = \arg\min_{h \in \{1,\dots,24\}} \hat{\mu}_h^{(d)}$$

$$h_{\text{sell}} = \arg\max_{h \in \{1,\dots,24\}} \hat{\mu}_h^{(d)}$$

To account for thermodynamic losses, a round-trip storage efficiency factor $\eta \in (0, 1]$ (e.g., $\eta = 0.9$) is applied. The realized financial profit $\Pi^{(d)}$ (in € per MWh) actually captured using the true market prices is:

$$\Pi^{(d)} = y_{\text{true}, h_{\text{sell}}}^{(d)} \cdot \eta - y_{\text{true}, h_{\text{buy}}}^{(d)}$$

---

### 2. Trading Strategies: Trader A vs. Trader B

While both traders select the same operational hours based on predicted price extrema, they differ fundamentally in whether they execute the trade based on model confidence.

#### Strategy A: "Always Bet" (Deterministic Execution)
Trader A ignores predictive variance entirely. Every single day, the trade is executed regardless of how uncertain the model is about price spikes or drops.

$$\text{Action}_A^{(d)} = \text{Execute Trade} \quad \forall d$$

The daily captured profit for Trader A in market regime $r$ is accumulated as:

$$\Pi_A^{(r)} = \sum_{d \in \mathcal{D}^{(r)}} \Pi^{(d)}$$

#### Strategy B: "Bet When Sure" (Uncertainty-Aware Execution)
Trader B uses the predictive standard deviation $\hat{\sigma}_h$ to construct a confidence interval around the expected price gap.

* **Expected Price Gap (Expected Reward):**

$$\Delta \hat{\mu}^{(d)} = \hat{\mu}_{h_{\text{sell}}}^{(d)} - \hat{\mu}_{h_{\text{buy}}}^{(d)}$$

* **Uncertainty Band (Potential Risk/Doubt):**
Using a Gaussian assumption, an $80\%$ confidence bound corresponds to a critical value $z_{0.90} \approx 1.28$. The total combined uncertainty ($\text{Doubt}^{(d)}$) across both trading hours is:

$$\text{Doubt}^{(d)} = 1.28 \cdot \left( \hat{\sigma}_{h_{\text{buy}}}^{(d)} + \hat{\sigma}_{h_{\text{sell}}}^{(d)} \right)$$

#### Binary Decision Rule:
Trader B evaluates a conditional filter. If the expected reward outweighs the potential error boundary, the trade executes; otherwise, the battery remains idle, avoiding catastrophic forecasting errors:

$$\text{Action}_B^{(d)} = \begin{cases} 
\text{Execute Trade} & \text{if } \Delta \hat{\mu}^{(d)} > \text{Doubt}^{(d)} \\ 
\text{Skip Trade ($\Pi^{(d)} = 0$)} & \text{otherwise}     \end{cases}$$

The accumulated profit for Trader B in market regime $r$ is:

$$\Pi_B^{(r)} = \sum_{d \in \mathcal{D}^{(r)}} \mathbb{I}\left(\Delta \hat{\mu}^{(d)} > \text{Doubt}^{(d)}\right) \cdot \Pi^{(d)}$$

where $\mathbb{I}(\cdot)$ is the indicator function.

---

### 3. Macro-Economic Scaling (The 2030 Fleet Valuation)

To translate per-MWh daily performance into an industry-grade funding metric, the framework scales the daily delta across Germany's projected national battery storage capacity for 2030 (target fleet capacity of $C = 10,000 \text{ MWh}$).

The annual monetary value added (in € Millions) by adopting uncertainty-aware trading (Trader B over Trader A) within regime $r$ is calculated as:

$$\text{Value Added}^{(r)} = \left( \bar{\Pi}_B^{(r)} - \bar{\Pi}_A^{(r)} \right) \times 365 \times C \times 10^{-6}$$

where $\bar{\Pi}^{(r)}$ denotes the average daily profit per MWh normalized by the number of active days in regime $r$.

---

### Mathematical Conclusion

During Crisis Regimes ($r=2$), price volatility and model residual errors spike non-linearly. In these windows, $\text{Doubt}^{(d)}$ effectively filters out false-positive arbitrage signals where volatile price spikes are mispredicted. Mathematically:

$$\bar{\Pi}_B^{(2)} \gg \bar{\Pi}_A^{(2)}$$

This proves that quantification and utilization of predictive uncertainty protects storage assets from severe downside risk during market shocks, unlocking tens of millions in enterprise value.

## Mathematical Formulation of Part 12: Multi-Model Battery Arbitrage & Uncertainty Valuation

This mathematical formulation extends the battery profit model to evaluate multiple competing probabilistic and deterministic forecasting models ($\text{DDNN}, \text{EvDNN}, \text{VI-DDNN}, \text{VI+CP}, \text{BSSM}$), assessing how different uncertainty estimation techniques ($\hat{\sigma}$) impact trading profitability.

---

### 1. Model-Specific Predictions and Information Set

Let each model $m \in \mathcal{M}$ (where $\mathcal{M} = \{\text{DDNN}, \text{EvDNN}, \text{VI-DDNN}, \text{VI+CP}, \text{BSSM}\}$) provide a conditional expectation of future prices and a corresponding metric of predictive dispersion over the test set horizon $t$:

$$\hat{\mu}_{m, t} = \mathbb{E}[Y_t \mid \mathcal{I}_t], \quad \hat{\sigma}_{m, t} = \sqrt{\text{Var}(Y_t \mid \mathcal{I}_t)}$$

where $\mathcal{I}_t$ represents the feature information set available at time $t$.

---

### 2. Daily Arbitrage Optimization Problem

For a given model $m$, the battery controller optimizes the intraday dispatch schedule for day $d$ (consisting of 24 hourly steps $t \in \{1, \dots, 24\}$):

$$\hat{h}_{\text{buy}}^{(m, d)} = \arg\min_{t \in \text{day } d} \hat{\mu}_{m, t}$$

$$\hat{h}_{\text{sell}}^{(m, d)} = \arg\max_{t \in \text{day } d} \hat{\mu}_{m, t}$$

The true realized daily profit $\Pi_{\text{true}}^{(d)}$ (in € per MWh) achieved by executing trades at these model-selected hours against the actual wholesale market prices $y_{\text{true}, t}$ with round-trip efficiency $\eta$ is:

$$\Pi_{\text{true}}^{(d)} = y_{\text{true}, \hat{h}_{\text{sell}}^{(m, d)}} \cdot \eta - y_{\text{true}, \hat{h}_{\text{buy}}^{(m, d)}}$$

---

### 3. Model-Dependent Decision Rules (Trader A vs. Trader B)

#### Strategy A: Unconditional Execution (Trader A)
Trader A executes the trade every day $d$ unconditionally, independent of the model's uncertainty estimates:

$$\text{Action}_A^{(m, d)} = 1 \quad \forall d$$

The average daily profit per MWh for model $m$ under Trader A is:

$$\bar{\Pi}_{A}^{(m)} = \frac{1}{D} \sum_{d=1}^{D} \Pi_{\text{true}}^{(d)}$$

where $D$ is the total number of evaluation days.

#### Strategy B: Uncertainty-Filtered Execution (Trader B)
Trader B uses the model-specific uncertainty $\hat{\sigma}_{m, t}$ to filter out unreliable trading signals.

* **Expected Price Spread:**

$$\Delta \hat{\mu}_{m}^{(d)} = \hat{\mu}_{m, \hat{h}_{\text{sell}}^{(m, d)}} - \hat{\mu}_{m, \hat{h}_{\text{buy}}^{(m, d)}}$$

* **Uncertainty Threshold (Risk Boundary):**
Using a confidence coefficient $z_{0.90} \approx 1.28$ (corresponding to an $80\%$ Gaussian interval), the combined risk parameter is:

$$\text{Doubt}_{m}^{(d)} = 1.28 \cdot \left( \hat{\sigma}_{m, \hat{h}_{\text{buy}}^{(m, d)}} + \hat{\sigma}_{m, \hat{h}_{\text{sell}}^{(m, d)}} \right)$$

* **Conditional Filter Indicator:**

$$\mathbb{I}_{\text{trade}}^{(m, d)} = \begin{cases} 
1 & \text{if } \Delta \hat{\mu}_{m}^{(d)} > \text{Doubt}_{m}^{(d)} \\ 
0 & \text{otherwise} 
\end{cases}$$

The average daily profit per MWh under Trader B for model $m$ is:

$$\bar{\Pi}_{B}^{(m)} = \frac{1}{D} \sum_{d=1}^{D} \mathbb{I}_{\text{trade}}^{(m, d)} \cdot \Pi_{\text{true}}^{(d)}$$

---

### 4. Comparative Fleet-Level Economic Valuation

To compare the financial efficacy across different models, the net daily performance delta is scaled across Germany's targeted 2030 national battery storage fleet capacity ($C = 10,000 \text{ MWh}$).

The Annual Value Added ($\text{AVA}_m$) generated specifically by utilizing model $m$'s uncertainty bounds to filter arbitrage trades (Trader B minus Trader A) is quantified as:

$$\text{AVA}_m = \left( \bar{\Pi}_{B}^{(m)} - \bar{\Pi}_{A}^{(m)} \right) \times 365 \times C \times 10^{-6} \quad (\text{in € Million / Year})$$

---

### Mathematical Interpretation

Different models yield varying values of $\text{AVA}_m$ based on the calibration quality of their predictive standard deviation $\hat{\sigma}_{m, t}$:

* **Overconfident models** (such as uncalibrated neural networks with compressed $\hat{\sigma}$) result in $\text{Doubt}_m^{(d)}$ values that are too small, failing to filter out bad trades, making $\bar{\Pi}_B \approx \bar{\Pi}_A$.
* **Well-calibrated Bayesian or Conformal models** (such as $\text{VI-DDNN}$ or $\text{VI+CP}$) yield accurate uncertainty bounds, successfully avoiding loss-making trades during high-volatility hours, which maximizes the positive differential:

$$\bar{\Pi}_B^{(m)} > \bar{\Pi}_A^{(m)} \implies \text{AVA}_m > 0$$

# Update

In [24]:

pio.renderers.default = "iframe"
DOCS_DIR = DATA_DIR
def run_battery(pred, regime_of_hour, eta=0.9):
    """
    pred: dict with mu (predicted price), sigma (uncertainty),
          y_true (real price), per test hour.
    Returns profit per regime for both traders.
    """
    mu, sigma, y_true = pred['mu'], pred['sigma'], pred['y_true']
    n_days = len(mu) // 24

    # accumulate profit per regime for each trader
    prof_A = {0: 0.0, 1: 0.0, 2: 0.0}   # always bet
    prof_B = {0: 0.0, 1: 0.0, 2: 0.0}   # bet when sure
    days   = {0: 0, 1: 0, 2: 0}

    for d in range(n_days):
        sl = slice(d * 24, (d + 1) * 24)
        mu_d, sig_d, true_d = mu[sl], sigma[sl], y_true[sl]
        reg_d = regime_of_hour[sl]
        if len(mu_d) < 24:
            continue
        regime = int(np.bincount(reg_d).argmax())   # day's dominant regime
        days[regime] += 1

        # choose buy (cheapest predicted) and sell (priciest predicted) hours
        buy_h = int(np.argmin(mu_d))
        sell_h = int(np.argmax(mu_d))
        if sell_h <= buy_h:
            buy_h, sell_h = min(buy_h, sell_h), max(buy_h, sell_h)

        # REALISED profit uses TRUE prices (what actually happened)
        realised = true_d[sell_h] * eta - true_d[buy_h]

        # Trader A: always trades
        prof_A[regime] += realised

        # Trader B: only if confident the gap is real
        gap = mu_d[sell_h] - mu_d[buy_h]
        doubt = 1.28 * (sig_d[buy_h] + sig_d[sell_h])
        if gap > doubt:
            prof_B[regime] += realised

    # convert to per-day averages
    names = {0: 'Normal', 1: 'Elevated', 2: 'Crisis'}
    rows = []
    for r in [0, 1, 2]:
        dd = max(days[r], 1)
        a = prof_A[r] / dd
        b = prof_B[r] / dd
        rows.append((names[r], a, b, days[r]))
        print(f"  {names[r]:9s}: Trader A €{a:7.2f}/day  "
              f"Trader B €{b:7.2f}/day  (extra €{b-a:6.2f})  [{days[r]} days]")
    return rows


def visualize_battery_plotly(rows, save_path=os.path.join(DATA_DIR, "part12_battery_profit.html")):
    print("\nVisualizing battery profit with Plotly...")
    regs = [r[0] for r in rows]
    A = [r[1] for r in rows]
    B = [r[2] for r in rows]
    rcol = {'Normal': '#1D9E75', 'Elevated': '#BA7517', 'Crisis': '#D85A30'}

    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=(
            "<b>1. Daily Profit Comparison per MWh</b><br><sup>Uncertainty-aware trading wins most in crisis</sup>",
            "<b>2. Annual Value Unlocked Across Germany's 2030 Fleet</b><br><sup>Target: 10 GW / 10 GWh battery capacity</sup>"
        ),
        horizontal_spacing=0.12
    )

    # ── Left Subplot: Daily Profit Comparison (Trader A vs Trader B) ─────────
    fig.add_trace(
        go.Bar(
            x=regs, y=A,
            name='Trader A (always bet)',
            marker_color='#888780',
            text=[f"€{val:.0f}" for val in A],
            textposition='outside',
            hovertemplate="<b>%{x}</b><br>Trader A Profit: €%{y:.2f}/day<extra></extra>"
        ),
        row=1, col=1
    )

    fig.add_trace(
        go.Bar(
            x=regs, y=B,
            name='Trader B (bet when sure)',
            marker_color=[rcol[r] for r in regs],
            text=[f"€{val:.0f}" for val in B],
            textposition='outside',
            hovertemplate="<b>%{x}</b><br>Trader B Profit: €%{y:.2f}/day<extra></extra>"
        ),
        row=1, col=1
    )

    # ── Right Subplot: Annual Fleet Value (€ Million) ────────────────────────
    fleet = 10000   # MWh, Germany's 2030 target
    fleet_val = [(b - a) * 365 * fleet / 1e6 for _, a, b, _ in rows]

    fig.add_trace(
        go.Bar(
            x=regs, y=fleet_val,
            marker_color=[rcol[r] for r in regs],
            name='Fleet Value (€M)',
            showlegend=False,
            text=[f"€{val:.0f}M" for val in fleet_val],
            textposition='outside',
            hovertemplate="<b>%{x} Regime</b><br>Extra Value: €%{y:.1f}M/year<extra></extra>"
        ),
        row=1, col=2
    )

    # Layout configuration
    fig.update_layout(
        barmode='group',
        title=dict(
            text="<b>Battery Profit Story — Uncertainty Quantification Is Worth Most When the Market Is Volatile</b>",
            font=dict(size=16),
            x=0.5,
            xanchor="center"
        ),
        paper_bgcolor="#f7f6f3",
        plot_bgcolor="#ffffff",
        height=750,
        width=1350,
        legend=dict(
            orientation="h",
            yanchor="bottom",
            y=1.15,
            xanchor="center",
            x=0.5
        )
    )

    fig.update_yaxes(title_text="Battery profit (€ per MWh per day)", row=1, col=1)
    fig.update_yaxes(title_text="Extra value per year (€ million)", row=1, col=2)
    fig.update_xaxes(showgrid=False)
    fig.update_yaxes(showgrid=True, gridwidth=1, gridcolor="rgba(0,0,0,0.1)", range=[0, max(max(B)*1.15, max(fleet_val)*1.15)])

    # Save to HTML and display
    fig.write_html(save_path)
    print(f"  ✓ Saved interactive report to {save_path}")
    try:
        fig.show()   # works in Jupyter/Kaggle
    except Exception:
        pass  # skip display when running as a plain script (HTML already saved)

    total_crisis = fleet_val[2]
    print(f"\n  HEADLINE: in the crisis regime, uncertainty-aware trading unlocks "
          f"€{total_crisis:.0f}M/year across Germany's 2030 battery fleet.")


def make_full_predictions():
    df = pd.read_pickle(os.path.join(DATA_DIR, "data_part7_regimes.pkl"))
    price = df['price'].values
    mu = df['lag_24'].values
    resid = np.abs(price - mu)
    sigma = pd.Series(resid).rolling(168, min_periods=24).mean().bfill().values
    sigma = np.clip(sigma, 5, 200)

    valid = ~np.isnan(mu)
    return {
        'mu': mu[valid], 'sigma': sigma[valid], 'y_true': price[valid],
        'regime': df['regime'].values[valid], 'datetime': df['datetime'].values[valid]
    }


# ── Run this part ──────────────────────────────────────────────────────────
if __name__ == "__main__":
    print("Running battery simulation on full history (all regimes present)...\n")
    pred = make_full_predictions()
    regime_of_hour = pred['regime']

    rows = run_battery(pred, regime_of_hour)
    visualize_battery_plotly(rows, os.path.join(DATA_DIR, "part12_battery_profit.html"))
    print("\n✓ Done — interactive dashboard saved to part12_battery_profit.html")

"""
═══════════════════════════════════════════════════════════════════════════
 PART 12 — BATTERY PROFIT STORY FOR ALL MODELS (PLOTLY DASHBOARD)
═══════════════════════════════════════════════════════════════════════════
 Evaluates uncertainty-aware trading (Trader A vs Trader B) using the REAL
 test-set predictions (mu and sigma) from ALL models saved in Part 8.
═══════════════════════════════════════════════════════════════════════════
"""

def evaluate_model_battery(mu, sigma, y_true, regimes, eta=0.9):
    """
    Run the Trader A vs Trader B battery test for ONE model's predictions.

    Trader A (always bet): trades every day, ignoring uncertainty.
    Trader B (bet when sure): trades only when the predicted price gap
                              exceeds the uncertainty band (doubt).

    Returns (profit_A_per_day, profit_B_per_day) averaged over all days,
    pooled across every regime (a single headline number per trader).
    """
    n_days = len(mu) // 24
    total_A, total_B, n = 0.0, 0.0, 0
    for d in range(n_days):
        sl = slice(d * 24, (d + 1) * 24)
        mu_d, sig_d, true_d = mu[sl], sigma[sl], y_true[sl]
        if len(mu_d) < 24:
            continue
        n += 1
        buy_h = int(np.argmin(mu_d))
        sell_h = int(np.argmax(mu_d))
        if sell_h <= buy_h:
            buy_h, sell_h = min(buy_h, sell_h), max(buy_h, sell_h)
        realised = true_d[sell_h] * eta - true_d[buy_h]
        total_A += realised                                   # always trades
        gap = mu_d[sell_h] - mu_d[buy_h]
        doubt = 1.28 * (sig_d[buy_h] + sig_d[sell_h])
        if gap > doubt:
            total_B += realised                               # only if confident
    dd = max(n, 1)
    return total_A / dd, total_B / dd



def run_all_models_battery():
    data_8 = np.load(os.path.join(DOCS_DIR, 'results_part8.npy'), allow_pickle=True).item()
    
    # Extract selected feature results or fall back to generic results
    results = data_8.get('results_selected', data_8.get('results', {}))
    y_true = data_8['y_test']

    df = pd.read_pickle(os.path.join(DOCS_DIR, 'data_part7_regimes.pkl'))
    test_df = df[df['datetime'] >= '2025-05-01'].copy()
    regimes = test_df['regime'].values

    model_performances = {}

    print("\n  Battery Arbitrage Performance Across All Models (Test Set):")
    print("  " + "─"*55)

    for name, res in results.items():
        mu = res['mu']
        sigma = res['sigma']
        
        min_len = min(len(mu), len(y_true))
        pa, pb = evaluate_model_battery(mu[:min_len], sigma[:min_len], y_true[:min_len], regimes[:min_len])
        
        model_performances[name] = {'Trader_A': pa, 'Trader_B': pb, 'Extra': pb - pa}
        print(f"  {name:<10}: Trader A: €{pa:.2f}/day | Trader B: €{pb:.2f}/day | Extra: €{pb-pa:.2f}/day")

    return model_performances


def visualize_all_models_plotly(performances, save_path=os.path.join(DATA_DIR, "part12_all_models_battery.html")):
    print("\nVisualizing multi-model battery performance with Plotly...")
    models = list(performances.keys())
    trader_a = [performances[m]['Trader_A'] for m in models]
    trader_b = [performances[m]['Trader_B'] for m in models]
    
    # 2030 Fleet scaling (10,000 MWh target)
    fleet = 10000
    annual_extra_val = [(performances[m]['Extra']) * 365 * fleet / 1e6 for m in models]

    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=(
            "<b>1. Daily Profit per MWh (Trader A vs Trader B)</b>",
            "<b>2. Annual Extra Value Unlocked across 2030 Fleet (€M/year)</b>"
        ),
        horizontal_spacing=0.15
    )

    # Subplot 1: Daily Profits
    fig.add_trace(
        go.Bar(
            x=models, y=trader_a,
            name='Trader A (Always Bet)',
            marker_color='#888780',
            text=[f"€{v:.0f}" for v in trader_a],
            textposition='outside'
        ),
        row=1, col=1
    )
    fig.add_trace(
        go.Bar(
            x=models, y=trader_b,
            name='Trader B (Uncertainty-Aware)',
            marker_color='#1D9E75',
            text=[f"€{v:.0f}" for v in trader_b],
            textposition='outside'
        ),
        row=1, col=1
    )

    # Subplot 2: Annual Fleet Value Added
    fig.add_trace(
        go.Bar(
            x=models, y=annual_extra_val,
            marker_color='#D85A30',
            name='Extra Fleet Value (€M)',
            showlegend=False,
            text=[f"€{v:.1f}M" for v in annual_extra_val],
            textposition='outside'
        ),
        row=1, col=2
    )

    fig.update_layout(
        barmode='group',
        title=dict(
            text="<b>Multi-Model Battery Profit & Uncertainty Valuation (Test Set)</b>",
            font=dict(size=16), x=0.5, xanchor="center"
        ),
        paper_bgcolor="#f7f6f3",
        plot_bgcolor="#ffffff",
        height=700, width=1350,
        legend=dict(orientation="h", yanchor="bottom", y=1.12, xanchor="center", x=0.5)
    )

    fig.update_yaxes(title_text="Daily Profit (€ / MWh / day)", row=1, col=1)
    fig.update_yaxes(title_text="Extra Value Generated (€ Million / year)", row=1, col=2)
    fig.update_xaxes(showgrid=False)
    fig.update_yaxes(showgrid=True, gridwidth=1, gridcolor="rgba(0,0,0,0.1)")

    fig.write_html(save_path)
    print(f"  ✓ Saved multi-model dashboard to {save_path}")
    try:
        fig.show()   # works in Jupyter/Kaggle
    except Exception:
        pass  # skip display when running as a plain script (HTML already saved)


if __name__ == "__main__":
    print("Running battery simulation across all trained models from Part 8...\n")
    performances = run_all_models_battery()
    visualize_all_models_plotly(performances)
    print("\n✓ Done — multi-model battery dashboard generated successfully!")

Running battery simulation on full history (all regimes present)...

  Normal   : Trader A €  35.72/day  Trader B €  37.79/day  (extra €  2.07)  [1280 days]
  Elevated : Trader A €  45.24/day  Trader B €  52.00/day  (extra €  6.76)  [800 days]
  Crisis   : Trader A €  72.89/day  Trader B €  81.50/day  (extra €  8.60)  [224 days]

Visualizing battery profit with Plotly...
  ✓ Saved interactive report to part12_battery_profit.html



  HEADLINE: in the crisis regime, uncertainty-aware trading unlocks €31M/year across Germany's 2030 battery fleet.

✓ Done — interactive dashboard saved to part12_battery_profit.html
Running battery simulation across all trained models from Part 8...


  Battery Arbitrage Performance Across All Models (Test Set):
  ───────────────────────────────────────────────────────
  DDNN      : Trader A: €66.48/day | Trader B: €72.59/day | Extra: €6.11/day
  EvDNN     : Trader A: €66.15/day | Trader B: €2.56/day | Extra: €-63.59/day
  VI-DDNN   : Trader A: €70.04/day | Trader B: €9.95/day | Extra: €-60.08/day
  VI+CP     : Trader A: €70.04/day | Trader B: €77.85/day | Extra: €7.82/day
  BSSM      : Trader A: €-8.74/day | Trader B: €1.54/day | Extra: €10.28/day

Visualizing multi-model battery performance with Plotly...
  ✓ Saved multi-model dashboard to part12_all_models_battery.html



✓ Done — multi-model battery dashboard generated successfully!


## Mathematical Formulation of Part 13: Three Model Improvements

Part 13 introduces three targeted improvements that address the weaknesses identified in the baseline models: miscalibrated intervals, the "frozen model" problem, and unprincipled hyperparameter choices.

---

### 1. Signed-Log Conformal Calibration

The baseline converts uncertainty from log-space to euros using the **delta-method**, $\sigma_{\text{eur}} \approx \sigma_z \cdot e^{|z|}$, which inflates the band during high-price hours. Instead, we build the conformal interval **entirely inside the signed-log space** and only then map the bounds to euros.

Let $z_t = \operatorname{slog}(y_t)$ be the signed-log price and $\hat{z}_t$ the model prediction. On the calibration set we compute the nonconformity scores:

$$s_t = |z_t - \hat{z}_t|$$

and take the empirical $(1-\alpha)$ quantile $\hat{q} = \text{Quantile}_{1-\alpha}(\{s_t\})$. The prediction interval in **euro space** is obtained by passing the log-space bounds through the monotonic inverse transform $\operatorname{slog}^{-1}$:

$$\mathcal{C}(x_t) = \left[\,\operatorname{slog}^{-1}(\hat{z}_t - \hat{q}),\; \operatorname{slog}^{-1}(\hat{z}_t + \hat{q})\,\right]$$

Because $\operatorname{slog}^{-1}$ is strictly increasing, the marginal coverage guarantee $\mathbb{P}(y_t \in \mathcal{C}(x_t)) \geq 1-\alpha$ is preserved exactly, without variance inflation.

---

### 2. Rolling-Window Retraining

A model trained once on 2020–2023 becomes stale as the market evolves. We retrain on a **sliding window** of the most recent $W$ days and re-fit every $S$ days. For a forecast origin $\tau$:

$$\mathcal{D}_{\text{train}}(\tau) = \{(\mathbf{x}_t, y_t) : \tau - W \leq t < \tau\}$$

The model $f_\theta$ is refit on $\mathcal{D}_{\text{train}}(\tau)$ and used to predict the next block $[\tau, \tau + S)$. This directly addresses the "frozen model" weakness identified by Lebedev et al. (2026), letting the network adapt to changing volatility and renewable penetration.

---

### 3. Bayesian Hyperparameter Optimization

Rather than hand-picking hyperparameters $\boldsymbol{\lambda}$ (learning rate, depth, epochs), we search them with a lightweight **Gaussian-process surrogate**. Let $g(\boldsymbol{\lambda})$ be the validation score. We model it as a GP:

$$g(\boldsymbol{\lambda}) \sim \mathcal{GP}\big(m(\boldsymbol{\lambda}), k(\boldsymbol{\lambda}, \boldsymbol{\lambda}')\big)$$

and select the next candidate by maximizing an acquisition function (expected improvement):

$$\boldsymbol{\lambda}_{\text{next}} = \arg\max_{\boldsymbol{\lambda}} \; \mathbb{E}\big[\max(0,\; g_{\text{best}} - g(\boldsymbol{\lambda}))\big]$$

This finds strong hyperparameters in far fewer evaluations than grid search.

In [27]:
"""
═══════════════════════════════════════════════════════════════════════════
 PART 13 — THREE MODEL IMPROVEMENTS
═══════════════════════════════════════════════════════════════════════════
 1. SIGNED-LOG CONFORMAL FIX
      Fit conformal prediction directly in signed-log space, then transform
      the interval bounds to euros. This keeps conformal's calibration
      guarantee (instead of inflating sigma via the delta-method).

 2. ROLLING-WINDOW RETRAINING
      Retrain the model every N days on a sliding window of recent data,
      so it always sees the current market regime. Fixes the "frozen model"
      weakness that Lebedev 2026 identified.

 3. BAYESIAN HYPERPARAMETER OPTIMIZATION
      A lightweight Bayesian optimizer (Gaussian-process style surrogate,
      pure numpy/scipy — no optuna needed) that searches hyperparameters
      and picks the set with the best validation score.
═══════════════════════════════════════════════════════════════════════════
"""




# ════════════════════════════════════════════════════════════════════════════
# IMPROVEMENT 1: SIGNED-LOG CONFORMAL PREDICTION
# ════════════════════════════════════════════════════════════════════════════

pio.renderers.default = "iframe"
class SignedLogConformal:
    """
    Conformal prediction done correctly for signed-log targets.
    """
    def __init__(self, coverage=0.90):
        self.coverage = coverage
        self.q_by_hour = {}

    def fit(self, z_true_cal, z_pred_cal, hours_cal):
        """Learn the residual quantile per hour, in signed-log space."""
        resid = np.abs(z_true_cal - z_pred_cal)
        for h in range(24):
            mask = hours_cal == h
            if mask.sum() > 10:
                self.q_by_hour[h] = np.quantile(resid[mask], self.coverage)
            else:
                self.q_by_hour[h] = np.quantile(resid, self.coverage)

    def predict(self, z_pred_test, hours_test):
        """Return euro mean, lower, upper — interval built in log space first."""
        q = np.array([self.q_by_hour.get(int(h), np.median(list(self.q_by_hour.values())))
                      for h in hours_test])
        z_lower = z_pred_test - q
        z_upper = z_pred_test + q
        mu_eur    = inv_signed_log(z_pred_test)
        lower_eur = inv_signed_log(z_lower)
        upper_eur = inv_signed_log(z_upper)
        return mu_eur, lower_eur, upper_eur


def metrics_from_interval(y_true, mu, lower, upper, coverage=0.90):
    """Compute MAE, PICP, MPIW, MAACE directly from an explicit interval."""
    mae  = float(np.mean(np.abs(y_true - mu)))
    rmse = float(np.sqrt(np.mean((y_true - mu)**2)))
    picp = float(np.mean((y_true >= lower) & (y_true <= upper)))
    mpiw = float(np.mean(upper - lower))
    levels = np.arange(0.1, 1.0, 0.1)
    maace = 0.0
    half = (upper - lower) / 2
    z90 = stats.norm.ppf(0.95)
    sigma_equiv = half / z90
    for lv in levels:
        z = stats.norm.ppf((1+lv)/2)
        lo = mu - z*sigma_equiv; hi = mu + z*sigma_equiv
        cov = np.mean((y_true >= lo) & (y_true <= hi))
        maace += abs(cov - lv)
    maace = maace/len(levels)*100
    return {"MAE":mae, "RMSE":rmse, "PICP_90":picp, "MPIW_90":mpiw, "MAACE":maace}


def run_signed_log_conformal(train, val, test, features):
    """Train DDNN in signed-log space and apply the FIXED conformal prediction."""
    print("\n" + "="*70)
    print("  IMPROVEMENT 1: SIGNED-LOG CONFORMAL FIX")
    print("="*70)

    z_tr  = signed_log(train['price'].values)
    z_val = signed_log(val['price'].values)
    y_te  = test['price'].values

    scaler_X = StandardScaler()
    X_tr = scaler_X.fit_transform(train[features].values)
    X_val= scaler_X.transform(val[features].values)
    X_te = scaler_X.transform(test[features].values)

    scaler_z = StandardScaler()
    z_tr_s  = scaler_z.fit_transform(z_tr.reshape(-1,1)).flatten()
    z_val_s = scaler_z.transform(z_val.reshape(-1,1)).flatten()

    print("  Training DDNN in signed-log space...")
    ddnn = DDNN(len(features), hidden=[128,64,32], lr=0.002, epochs=300)
    ddnn.fit(X_tr, z_tr_s, X_val, z_val_s)

    def predict_z(X):
        mu_s, _ = ddnn.predict(X)
        return scaler_z.inverse_transform(mu_s.reshape(-1,1)).flatten()
    z_pred_val  = predict_z(X_val)
    z_pred_test = predict_z(X_te)

    # OLD WAY (delta-method)
    mu_eur = inv_signed_log(z_pred_test)
    sig_z  = np.std(z_val - z_pred_val)
    sig_eur_delta = sig_z * np.exp(np.abs(z_pred_test))
    z90 = stats.norm.ppf(0.95)
    old = metrics_from_interval(y_te, mu_eur,
                                mu_eur - z90*sig_eur_delta,
                                mu_eur + z90*sig_eur_delta)

    # NEW WAY (conformal in log space)
    cp = SignedLogConformal(coverage=0.90)
    cp.fit(z_val, z_pred_val, val['hour'].values)
    mu_new, lo_new, hi_new = cp.predict(z_pred_test, test['hour'].values)
    new = metrics_from_interval(y_te, mu_new, lo_new, hi_new)

    print(f"\n  {'Metric':<10}{'OLD (delta)':>14}{'NEW (log-conformal)':>22}")
    print("  " + "-"*46)
    for k in ['MAE','PICP_90','MPIW_90','MAACE']:
        fmt = (lambda v: f"{v:.1%}") if k=='PICP_90' else (lambda v: f"{v:.2f}")
        print(f"  {k:<10}{fmt(old[k]):>14}{fmt(new[k]):>22}")

    return {'old':old, 'new':new, 'mu':mu_new, 'lower':lo_new, 'upper':hi_new,
            'y_true':y_te}


# ════════════════════════════════════════════════════════════════════════════
# IMPROVEMENT 2: ROLLING-WINDOW RETRAINING
# ════════════════════════════════════════════════════════════════════════════
def run_rolling_window(df, features, window_days=365, step_days=30):
    print("\n" + "="*70)
    print("  IMPROVEMENT 2: ROLLING-WINDOW RETRAINING")
    print("="*70)
    print(f"  Window: {window_days} days, retrain every {step_days} days")

    df = df.sort_values('datetime').reset_index(drop=True)
    test_start = pd.Timestamp('2025-05-01')
    test_idx = df.index[df['datetime'] >= test_start].tolist()

    hours_per_day = 24
    window = window_days * hours_per_day
    step   = step_days * hours_per_day

    preds_rolling = np.full(len(df), np.nan)
    start = test_idx[0]
    print("  Walking forward (retraining each block)...")
    n_blocks = 0
    pos = start
    while pos < len(df):
        tr_lo = max(0, pos - window)
        train_block = df.iloc[tr_lo:pos]
        pred_block  = df.iloc[pos:pos+step]
        if len(pred_block) == 0: break
        if len(train_block) < 24*30:
            pos += step; continue

        sX = StandardScaler()
        Xtr = sX.fit_transform(train_block[features].values)
        ztr = signed_log(train_block['price'].values)
        sz  = StandardScaler(); ztr_s = sz.fit_transform(ztr.reshape(-1,1)).flatten()

        model = DDNN(len(features), hidden=[128,64,32], lr=0.0025, epochs=200)
        model.fit(Xtr, ztr_s, verbose=False)

        Xpr = sX.transform(pred_block[features].values)
        mu_s, _ = model.predict(Xpr)
        z_pred = sz.inverse_transform(mu_s.reshape(-1,1)).flatten()
        preds_rolling[pos:pos+len(pred_block)] = inv_signed_log(z_pred)

        n_blocks += 1
        pos += step
    print(f"  Retrained {n_blocks} times across the test period")

    print("  Training the frozen (train-once) model for comparison...")
    pre = df[df['datetime'] < test_start]
    sX = StandardScaler(); Xtr = sX.fit_transform(pre[features].values)
    ztr = signed_log(pre['price'].values)
    sz = StandardScaler(); ztr_s = sz.fit_transform(ztr.reshape(-1,1)).flatten()
    frozen = DDNN(len(features), hidden=[128,64,32], lr=0.002, epochs=300)
    frozen.fit(Xtr, ztr_s, verbose=False)
    test_df = df[df['datetime'] >= test_start]
    Xte = sX.transform(test_df[features].values)
    mu_s,_ = frozen.predict(Xte)
    frozen_pred = inv_signed_log(sz.inverse_transform(mu_s.reshape(-1,1)).flatten())

    y_te = test_df['price'].values
    roll_pred = preds_rolling[test_df.index]
    valid = ~np.isnan(roll_pred)
    mae_frozen  = np.mean(np.abs(y_te[valid] - frozen_pred[valid]))
    mae_rolling = np.mean(np.abs(y_te[valid] - roll_pred[valid]))

    print(f"\n  {'Strategy':<22}{'MAE (€/MWh)':>14}")
    print("  " + "-"*36)
    print(f"  {'Frozen (train once)':<22}{mae_frozen:>14.2f}")
    print(f"  {'Rolling window':<22}{mae_rolling:>14.2f}")
    improvement = (mae_frozen - mae_rolling)/mae_frozen*100
    print(f"\n  Rolling window improves MAE by {improvement:.1f}%")

    return {'mae_frozen':mae_frozen, 'mae_rolling':mae_rolling,
            'improvement':improvement, 'y_true':y_te[valid],
            'frozen':frozen_pred[valid], 'rolling':roll_pred[valid]}


# ════════════════════════════════════════════════════════════════════════════
# IMPROVEMENT 3: BAYESIAN HYPERPARAMETER OPTIMIZATION
# ════════════════════════════════════════════════════════════════════════════
class SimpleBayesOpt:
    def __init__(self, bounds, n_init=5, n_iter=15):
        self.bounds = bounds
        self.names  = list(bounds)
        self.n_init = n_init
        self.n_iter = n_iter
        self.X = []
        self.y = []

    def _sample_random(self):
        return np.array([np.random.rand() for _ in self.names])

    def _to_real(self, x01):
        out = {}
        for i, n in enumerate(self.names):
            lo, hi = self.bounds[n]
            out[n] = lo + x01[i]*(hi-lo)
        return out

    def _surrogate(self, xq):
        if not self.X:
            return 0.0, 1.0
        X = np.array(self.X)
        d = np.linalg.norm(X - xq, axis=1)
        w = np.exp(-d**2 / 0.1)
        if w.sum() < 1e-9:
            return np.mean(self.y), 1.0
        mu = np.sum(w*np.array(self.y)) / w.sum()
        unc = 1.0 / (1.0 + w.sum())
        return mu, unc

    def _acquisition(self, xq):
        mu, unc = self._surrogate(xq)
        best = min(self.y) if self.y else 0
        return (best - mu) + 1.5*unc

    def optimize(self, objective):
        for _ in range(self.n_init):
            x = self._sample_random()
            score = objective(self._to_real(x))
            self.X.append(x); self.y.append(score)

        for it in range(self.n_iter):
            cands = [self._sample_random() for _ in range(200)]
            acq = [self._acquisition(c) for c in cands]
            x_next = cands[int(np.argmax(acq))]
            score = objective(self._to_real(x_next))
            self.X.append(x_next); self.y.append(score)

        best_i = int(np.argmin(self.y))
        return self._to_real(self.X[best_i]), self.y[best_i]


def run_bayesian_hpo(train, val, features):
    print("\n" + "="*70)
    print("  IMPROVEMENT 3: BAYESIAN HYPERPARAMETER OPTIMIZATION")
    print("="*70)

    z_tr  = signed_log(train['price'].values)
    z_val = signed_log(val['price'].values)
    sX = StandardScaler()
    tr_sub = train.iloc[::3]
    X_tr = sX.fit_transform(tr_sub[features].values)
    X_val= sX.transform(val[features].values)
    sz = StandardScaler()
    z_tr_s = sz.fit_transform(signed_log(tr_sub['price'].values).reshape(-1,1)).flatten()
    z_val_s= sz.transform(z_val.reshape(-1,1)).flatten()
    y_val = val['price'].values

    def objective(hp):
        h1 = int(hp['hidden1']); h2 = int(hp['hidden2']); h3 = max(8, h2//2)
        lr = hp['lr']
        model = DDNN(len(features), hidden=[h1, h2, h3], lr=lr, epochs=80)
        model.fit(X_tr, z_tr_s, verbose=False)
        mu_s, _ = model.predict(X_val)
        z_pred = sz.inverse_transform(mu_s.reshape(-1,1)).flatten()
        mu_eur = inv_signed_log(z_pred)
        return float(np.mean(np.abs(y_val - mu_eur)))

    bounds = {
        'hidden1': (32, 256),
        'hidden2': (16, 128),
        'lr':      (0.0005, 0.01),
    }
    print(f"  Searching: hidden1[32-256], hidden2[16-128], lr[0.0005-0.01]")
    print(f"  Running Bayesian optimization (4 random + 6 guided trials)...")

    opt = SimpleBayesOpt(bounds, n_init=4, n_iter=6)
    best_hp, best_score = opt.optimize(objective)

    baseline_mae = objective({'hidden1':128, 'hidden2':64, 'lr':0.002})
    print(f"\n  Hand-set baseline:  hidden=[128,64], lr=0.002  → val MAE €{baseline_mae:.2f}")
    print(f"  Bayesian-optimized: hidden=[{int(best_hp['hidden1'])},"
          f"{int(best_hp['hidden2'])}], lr={best_hp['lr']:.4f}  → val MAE €{best_score:.2f}")
    improvement = (baseline_mae - best_score)/baseline_mae*100
    print(f"  Improvement: {improvement:+.1f}%")

    return {'best_hp':best_hp, 'best_score':best_score,
            'baseline':baseline_mae, 'history':opt.y}


# ════════════════════════════════════════════════════════════════════════════
# PLOTLY DASHBOARD VISUALIZATION
# ════════════════════════════════════════════════════════════════════════════
def plot_part13_dashboard(r1, r2, r3, save_path=os.path.join(DATA_DIR, "part13_improvements_dashboard.html")):
    print("\nGenerating interactive Part 13 Plotly dashboard...")

    fig = make_subplots(
        rows=2, cols=2,
        subplot_titles=(
            "<b>1. Conformal Prediction: Old (Delta) vs. New (Log-Space)</b>",
            "<b>2. Retraining Strategy: Frozen vs. Rolling Window MAE</b>",
            "<b>3. Bayesian HPO: Validation MAE per Search Iteration</b>",
            "<b>4. Log-Conformal 90% Prediction Intervals (Test Slice)</b>"
        ),
        vertical_spacing=0.18,
        horizontal_spacing=0.10,
        specs=[[{"type": "bar"}, {"type": "bar"}],
               [{"type": "scatter"}, {"type": "scatter"}]]
    )

    metrics = ['MAE', 'PICP_90', 'MPIW_90', 'MAACE']
    old_vals = [r1['old'][k] * 100 if k == 'PICP_90' else r1['old'][k] for k in metrics]
    new_vals = [r1['new'][k] * 100 if k == 'PICP_90' else r1['new'][k] for k in metrics]

    fig.add_trace(go.Bar(x=metrics, y=old_vals, name='Old (Delta Method)', marker_color='#888780'), row=1, col=1)
    fig.add_trace(go.Bar(x=metrics, y=new_vals, name='New (Log-Conformal)', marker_color='#1D9E75'), row=1, col=1)

    strategies = ['Frozen (Train Once)', 'Rolling Window']
    maes = [r2['mae_frozen'], r2['mae_rolling']]
    fig.add_trace(go.Bar(
        x=strategies, y=maes,
        marker_color=['#888780', '#378ADD'],
        text=[f"€{v:.2f}" for v in maes],
        textposition='auto',
        showlegend=False
    ), row=1, col=2)

    history_scores = r3['history']
    iterations = list(range(1, len(history_scores) + 1))
    fig.add_trace(go.Scatter(
        x=iterations, y=history_scores,
        mode='lines+markers',
        name='Surrogate Validation MAE',
        line=dict(color='#D85A30', width=2),
        marker=dict(size=8),
        showlegend=False
    ), row=2, col=1)

    n_slice = min(168, len(r1['y_true']))
    x_idx = list(range(n_slice))
    y_true_slice = r1['y_true'][-n_slice:]
    mu_slice = r1['mu'][-n_slice:]
    lower_slice = r1['lower'][-n_slice:]
    upper_slice = r1['upper'][-n_slice:]

    fig.add_trace(go.Scatter(
        x=x_idx + x_idx[::-1],
        y=np.concatenate([upper_slice, lower_slice[::-1]]),
        fill='toself',
        fillcolor='rgba(29, 158, 117, 0.2)',
        line=dict(color='rgba(255,255,255,0)'),
        name='90% Conformal Interval',
        showlegend=True
    ), row=2, col=2)

    fig.add_trace(go.Scatter(
        x=x_idx, y=y_true_slice,
        mode='lines',
        name='Actual Price',
        line=dict(color='#333333', width=1)
    ), row=2, col=2)

    fig.add_trace(go.Scatter(
        x=x_idx, y=mu_slice,
        mode='lines',
        name='Predicted Mean',
        line=dict(color='#1D9E75', width=1.5)
    ), row=2, col=2)

    fig.update_layout(
        title=dict(
            text="<b>Part 13 — Three Model Improvements Interactive Dashboard</b>",
            font=dict(size=16), x=0.5, xanchor="center"
        ),
        paper_bgcolor="#f7f6f3",
        plot_bgcolor="#ffffff",
        height=850,
        width=1400,
        barmode='group',
        legend=dict(orientation="h", yanchor="bottom", y=1.04, xanchor="center", x=0.5)
    )

    fig.update_yaxes(title_text="Metric Value", row=1, col=1)
    fig.update_yaxes(title_text="MAE (€/MWh)", row=1, col=2)
    fig.update_xaxes(title_text="Optimization Trial", row=2, col=1)
    fig.update_yaxes(title_text="Validation MAE (€)", row=2, col=1)
    fig.update_xaxes(title_text="Test Hours (Slice)", row=2, col=2)
    fig.update_yaxes(title_text="Electricity Price (€/MWh)", row=2, col=2)

    fig.write_html(save_path)
    print(f"  ✓ Saved Part 13 interactive dashboard to {save_path}")
    fig.show()


# ── Execution block ────────────────────────────────────────────────────────
if __name__ == "__main__":
    df = pd.read_pickle(os.path.join(DATA_DIR, "data_part2.pkl"))
    with open(os.path.join(DATA_DIR, "selected_part4.json")) as f:
        features = json.load(f)

    train = df[df['datetime'] <= '2024-04-30 23:00'].copy()
    val   = df[(df['datetime'] >= '2024-05-01') & (df['datetime'] <= '2025-04-30 23:00')].copy()
    test  = df[df['datetime'] >= '2025-05-01'].copy()

    r1 = run_signed_log_conformal(train, val, test, features)
    r3 = run_bayesian_hpo(train, val, features)
    r2 = run_rolling_window(df, features, window_days=365, step_days=30)

    np.save(os.path.join(DATA_DIR, "improvements.npy"),
            {'conformal':r1, 'rolling':r2, 'hpo':r3}, allow_pickle=True)
    
    plot_part13_dashboard(r1, r2, r3)

    print("\n" + "="*70)
    print("  ALL THREE IMPROVEMENTS COMPLETE (WITH PLOTLY DASHBOARD)")
    print("="*70)


  IMPROVEMENT 1: SIGNED-LOG CONFORMAL FIX
  Training DDNN in signed-log space...

done. Final NLL=-1.5841.. 

  Metric       OLD (delta)   NEW (log-conformal)
  ----------------------------------------------
  MAE                17.03                 17.03
  PICP_90            90.8%                 88.0%
  MPIW_90           186.57                171.50
  MAACE              21.73                 16.97

  IMPROVEMENT 3: BAYESIAN HYPERPARAMETER OPTIMIZATION
  Searching: hidden1[32-256], hidden2[16-128], lr[0.0005-0.01]
  Running Bayesian optimization (4 random + 6 guided trials)...

  Hand-set baseline:  hidden=[128,64], lr=0.002  → val MAE €15.62
  Bayesian-optimized: hidden=[228,69], lr=0.0079  → val MAE €15.63
  Improvement: -0.0%

  IMPROVEMENT 2: ROLLING-WINDOW RETRAINING
  Window: 365 days, retrain every 30 days
  Walking forward (retraining each block)...
  Retrained 13 times across the test period
  Training the frozen (train-once) model for comparison...

  Strategy             


  ALL THREE IMPROVEMENTS COMPLETE (WITH PLOTLY DASHBOARD)


## Mathematical Formulation of Part 14: Advanced Upgrades & Full-History Battery

Part 14 unifies the strongest ideas into a fair, leakage-free comparison and introduces a heavy-tailed distribution head. All baselines are recomputed **live** on the same test window (May 2025 – April 2026), and the battery runs over the **full history (2020–2026)** so the 2022 crisis regime is represented.

---

### 1. Log-Space Conformal Mapping

As in Part 13, intervals are constructed in signed-log space and mapped to euros through the monotone inverse, avoiding delta-method inflation.

---

### 2. Walk-Forward Rolling-Window Retraining

The model is retrained on a sliding 365-day window, stepped every 60 days, producing genuinely out-of-sample forecasts across the entire test horizon.

---

### 3. Student-t (Heavy-Tailed) Distribution Head

The Gaussian head assumes thin tails and is repeatedly surprised by price spikes. We replace it with a **Student-t** predictive distribution whose network outputs a location $\mu$, scale $\sigma$, and a learnable degrees-of-freedom $\nu$:

$$(\mu, \log\sigma, \log\nu) = f_\theta(\mathbf{x})$$

The model is trained by minimizing the Student-t negative log-likelihood:

$$\mathcal{L}_t = -\frac{1}{N}\sum_{i=1}^N \left[ \log\Gamma\!\left(\tfrac{\nu+1}{2}\right) - \log\Gamma\!\left(\tfrac{\nu}{2}\right) - \tfrac{1}{2}\log(\nu\pi) - \log\sigma - \tfrac{\nu+1}{2}\log\!\left(1 + \tfrac{z_i^2}{\nu}\right)\right]$$

where $z_i = (y_i - \mu_i)/\sigma_i$. A **low** $\nu$ produces fat tails (rare spikes remain plausible); a **high** $\nu$ recovers the Gaussian. During training, the factor

$$w_i = \frac{\nu + 1}{\nu + z_i^2}$$

acts as a **robust weight** that down-weights outliers, so everyday forecasts are not dragged around by spikes. This single change reduces MAE from roughly €44 to €14. The network is trained with a hand-implemented **Adam** optimizer (momentum + adaptive learning rate) for smooth convergence.

---

### 4. Bayesian Stochastic Battery Optimizer

Instead of a hard threshold rule, the battery solves a daily **expected-utility maximization** over Monte-Carlo price scenarios drawn from the predictive distribution. For candidate buy/sell hours $(b, s)$:

$$U(b, s) = \mathbb{E}[\Pi] - \lambda \cdot \text{Std}[\Pi] - c_{\text{degr}}, \qquad \Pi = y_s \cdot \xi - y_b$$

where $\xi$ is round-trip efficiency, $c_{\text{degr}}$ the degradation cost, and $\lambda$ the risk-aversion coefficient. The trade executes only if $\max_{b<s} U(b,s) > 0$.

In [55]:
"""
═══════════════════════════════════════════════════════════════════════════
 PART 14 (revised) — ADVANCED UPGRADES, FAIR COMPARISON, FULL-HISTORY BATTERY
═══════════════════════════════════════════════════════════════════════════
 Upgrades present:
   1. Log-Space Conformal Mapping
   2. Walk-Forward Rolling-Window Retraining
   3. Student-t (heavy-tailed) distribution head  [Adam-trained]
   4. Bayesian Stochastic Battery Optimizer (expected utility)
   5. Interactive Plotly Dashboards saved to the working directory
═══════════════════════════════════════════════════════════════════════════
"""



pio.renderers.default = "iframe"
DOCS_DIR = DATA_DIR


# ════════════════════════════════════════════════════════════════════════════
# Metric helper (reports MEAN and MEDIAN interval width)
# ════════════════════════════════════════════════════════════════════════════
def interval_metrics(y, mu, lower, upper):
    mae  = float(np.mean(np.abs(y - mu)))
    rmse = float(np.sqrt(np.mean((y - mu)**2)))
    picp = float(np.mean((y >= lower) & (y <= upper)))
    mpiw = float(np.mean(upper - lower))
    mpiw_med = float(np.median(upper - lower))
    sigma = np.maximum((upper - lower)/(2*stats.norm.ppf(0.95)), 1e-6)
    levels = np.arange(0.1, 1.0, 0.1); maace = 0.0
    for lv in levels:
        z = stats.norm.ppf((1+lv)/2)
        cov = np.mean((y >= mu - z*sigma) & (y <= mu + z*sigma))
        maace += abs(cov - lv)
    maace = maace/len(levels)*100
    zc = (y-mu)/sigma
    crps = float(np.mean(sigma*(zc*(2*stats.norm.cdf(zc)-1)
                                + 2*stats.norm.pdf(zc) - 1/np.sqrt(np.pi))))
    return {"MAE":mae,"RMSE":rmse,"CRPS":crps,"PICP_90":picp,
            "MPIW_90":mpiw,"MPIW_90_med":mpiw_med,"MAACE":maace,
            "mu":mu,"lower":lower,"upper":upper,"sigma":sigma}


# ════════════════════════════════════════════════════════════════════════════
# UPGRADE 3 — Student-t head with ADAM optimizer (pure numpy)
# ════════════════════════════════════════════════════════════════════════════
class StudentTDNN_Adam:
    def __init__(self, input_dim, hidden=[64, 32], lr=0.01, epochs=250):
        self.lr, self.epochs = lr, epochs
        h1, h2 = hidden
        self.P = {
            'W1': np.random.randn(input_dim,h1)*np.sqrt(2/input_dim), 'b1': np.zeros(h1),
            'W2': np.random.randn(h1,h2)*np.sqrt(2/h1), 'b2': np.zeros(h2),
            'Wm': np.random.randn(h2,1)*0.01, 'bm': np.zeros(1),
            'Ws': np.random.randn(h2,1)*0.01, 'bs': np.zeros(1),
            'Wn': np.random.randn(h2,1)*0.01, 'bn': np.zeros(1),
        }
        self.m = {k: np.zeros_like(v) for k,v in self.P.items()}
        self.v = {k: np.zeros_like(v) for k,v in self.P.items()}
        self.t = 0
        self.losses = []

    def _relu(self, x): return np.maximum(0, x)

    def _fwd(self, X):
        z1 = X@self.P['W1']+self.P['b1']; h1 = self._relu(z1)
        z2 = h1@self.P['W2']+self.P['b2']; h2 = self._relu(z2)
        mu   = (h2@self.P['Wm']+self.P['bm']).flatten()
        lsig = np.clip((h2@self.P['Ws']+self.P['bs']).flatten(), -4, 4)
        lnu  = np.clip((h2@self.P['Wn']+self.P['bn']).flatten(), 0.3, 3.5)
        return mu, np.exp(lsig)+1e-3, np.exp(lnu)+1.0, (z1,h1,z2,h2)

    def _nll(self, y, mu, sig, nu):
        z = (y-mu)/sig
        ll = (gammaln((nu+1)/2)-gammaln(nu/2)-0.5*np.log(nu*np.pi)
              -np.log(sig)-(nu+1)/2*np.log1p(z**2/nu))
        return -np.mean(ll)

    def _adam_step(self, grads):
        self.t += 1; b1, b2, eps = 0.9, 0.999, 1e-8
        for k in self.P:
            g = grads.get(k, 0)
            self.m[k] = b1*self.m[k] + (1-b1)*g
            self.v[k] = b2*self.v[k] + (1-b2)*(g**2)
            mhat = self.m[k]/(1-b1**self.t)
            vhat = self.v[k]/(1-b2**self.t)
            self.P[k] -= self.lr * mhat/(np.sqrt(vhat)+eps)

    def fit(self, X, y, verbose=False):
        n = len(y)
        for ep in range(self.epochs):
            mu, sig, nu, (z1,h1,z2,h2) = self._fwd(X)
            self.losses.append(self._nll(y, mu, sig, nu))
            z = (y-mu)/sig; w = (nu+1)/(nu+z**2)
            dmu = -(w*z/sig)/n
            dlsig = (1 - w*z**2)/n
            grads = {}
            grads['Wm'] = h2.T@dmu.reshape(-1,1); grads['bm'] = np.array([dmu.sum()])
            grads['Ws'] = h2.T@dlsig.reshape(-1,1); grads['bs'] = np.array([dlsig.sum()])
            grads['Wn'] = np.zeros_like(self.P['Wn']); grads['bn'] = np.zeros_like(self.P['bn'])
            dh2 = (dmu.reshape(-1,1)@self.P['Wm'].T + dlsig.reshape(-1,1)@self.P['Ws'].T)*(z2>0)
            grads['W2'] = h1.T@dh2; grads['b2'] = dh2.sum(axis=0)
            dh1 = (dh2@self.P['W2'].T)*(z1>0)
            grads['W1'] = X.T@dh1; grads['b1'] = dh1.sum(axis=0)
            self._adam_step(grads)
        return self

    def predict(self, X):
        mu, sig, nu, _ = self._fwd(X)
        return mu, sig, nu


# ════════════════════════════════════════════════════════════════════════════
# UPGRADE 1 — Log-space conformal
# ════════════════════════════════════════════════════════════════════════════
class LogSpaceConformal:
    def __init__(self, coverage=0.90): self.coverage=coverage; self.q={}
    def fit(self, z_true, z_pred, hours):
        r = np.abs(z_true-z_pred)
        for h in range(24):
            m = hours==h
            self.q[h] = np.quantile(r[m],self.coverage) if m.sum()>10 else np.quantile(r,self.coverage)
    def predict(self, z_pred, hours):
        q = np.array([self.q.get(int(h),np.median(list(self.q.values()))) for h in hours])
        return inv_signed_log(z_pred), inv_signed_log(z_pred-q), inv_signed_log(z_pred+q)


# ════════════════════════════════════════════════════════════════════════════
# UPGRADE 2 — Walk-forward rolling window
# ════════════════════════════════════════════════════════════════════════════
def walk_forward_rolling(df, features, window_days=365, step_days=60,
                         test_start='2025-05-01'):
    df = df.sort_values('datetime').reset_index(drop=True)
    W = window_days*24; S = step_days*24
    test_idx = df.index[df['datetime'] >= test_start].tolist()
    preds = np.full(len(df), np.nan); pos = test_idx[0]; nb = 0
    while pos < len(df):
        tr = df.iloc[max(0,pos-W):pos]; bl = df.iloc[pos:pos+S]
        if len(bl)==0: break
        if len(tr) < 24*60: pos += S; continue
        sX = StandardScaler(); Xtr = sX.fit_transform(tr[features].values)
        sz = StandardScaler(); ztr = sz.fit_transform(signed_log(tr['price'].values).reshape(-1,1)).flatten()
        m = DDNN(len(features), hidden=[96,48,24], lr=0.0025, epochs=150); m.fit(Xtr, ztr, verbose=False)
        Xb = sX.transform(bl[features].values); mu_s,_ = m.predict(Xb)
        zb = sz.inverse_transform(mu_s.reshape(-1,1)).flatten()
        preds[pos:pos+len(bl)] = inv_signed_log(zb); nb += 1; pos += S
    test = df[df['datetime'] >= test_start]
    return preds[test.index], test, nb


# ════════════════════════════════════════════════════════════════════════════
# UPGRADE 4 — Bayesian Stochastic Battery (over FULL HISTORY, all regimes)
# ════════════════════════════════════════════════════════════════════════════
def battery_full_history(df_full, features, xi=0.90, degr_cost=2.0,
                         risk_aversion=0.02, n_scen=40):
    df = df_full.sort_values('datetime').reset_index(drop=True)
    price = df['price'].values
    hour  = df['hour'].values
    regime= df['regime'].values
    mu = df['lag_24'].values
    resid = np.abs(price - mu)
    sigma = pd.Series(resid).rolling(168, min_periods=24).mean().bfill().values
    sigma = np.clip(sigma, 5, 200)
    valid = ~np.isnan(mu)
    mu, sigma, price, hour, regime = mu[valid], sigma[valid], price[valid], hour[valid], regime[valid]

    n_days = len(mu)//24
    pn = {0:0.,1:0.,2:0.}; pe = {0:0.,1:0.,2:0.}; days={0:0,1:0,2:0}
    rng = np.random.default_rng(0)
    for d in range(n_days):
        sl = slice(d*24,(d+1)*24)
        mu_d, sig_d, true_d = mu[sl], sigma[sl], price[sl]
        reg_d = regime[sl]
        if len(mu_d) < 24: continue
        rg = int(np.bincount(reg_d).argmax()); days[rg]+=1
        b0,s0 = int(np.argmin(mu_d)), int(np.argmax(mu_d))
        if s0>b0: pn[rg] += true_d[s0]*xi - true_d[b0]
        scen = rng.normal(mu_d[None,:], sig_d[None,:], size=(n_scen,24))
        best_u, best = -1e9, None
        for b in np.argsort(mu_d)[:4]:
            for s in np.argsort(mu_d)[-4:]:
                if s<=b: continue
                pr = scen[:,s]*xi - scen[:,b] - degr_cost
                u = pr.mean() - risk_aversion*pr.std()
                if u>best_u: best_u, best = u, (b,s)
        if best and best_u>0:
            b,s = best; pe[rg] += true_d[s]*xi - true_d[b] - degr_cost
    names={0:'Normal',1:'Elevated',2:'Crisis'}; rows=[]
    for r in [0,1,2]:
        dd=max(days[r],1); rows.append((names[r], pn[r]/dd, pe[r]/dd, days[r]))
    return rows


# ════════════════════════════════════════════════════════════════════════════
# PLOTLY VISUALIZATIONS (FIXED MAPPING & SUBPLOT RENDERING)
# ════════════════════════════════════════════════════════════════════════════
def plot_comparison_plotly(table, save_path=os.path.join(DOCS_DIR, "part14_final_comparison.html")):
    names = list(table.keys())
    specs = [
        ('MAE', 'MAE (€/MWh) — lower better', True, None, 1, 1),
        ('PICP_90', 'PICP 90% — closer to 90 better', False, 90, 1, 2),
        ('MPIW_90_med', 'MPIW 90% median — lower (sharper) better', True, None, 2, 1),
        ('MAACE', 'MAACE (%) — lower better', True, None, 2, 2)
    ]

    fig = make_subplots(
        rows=2, cols=2,
        subplot_titles=[s[1] for s in specs],
        vertical_spacing=0.20, horizontal_spacing=0.12
    )

    pal = ['#888780', '#378ADD', '#1D9E75', '#7F77DD', '#BA7517', '#0C447C']

    for key, title, lb, target, r, c in specs:
        vals = [table[m][key]*100 if key=='PICP_90' else table[m][key] for m in names]
        colors = [pal[i % len(pal)] for i in range(len(names))]

        fig.add_trace(
            go.Bar(
                x=names, y=vals,
                marker_color=colors,
                text=[f"{v:.1f}" for v in vals],
                textposition='outside',
                showlegend=False,
                hovertemplate="<b>%{x}</b><br>%{y:.2f}<extra></extra>"
            ),
            row=r, col=c
        )
        if target is not None:
            fig.add_shape(
                type="line", x0=-0.5, x1=len(names)-0.5, y0=target, y1=target,
                line=dict(color="#D85A30", dash="dash", width=2),
                row=r, col=c
            )

    fig.update_layout(
        title=dict(
            text="<b>Part 14 — Fair Comparison (all rows computed live, same test window)</b><br><sup>real SMARD data; test May 2025 – Apr 2026</sup>",
            font=dict(size=15), x=0.5, xanchor="center"
        ),
        paper_bgcolor="#f7f6f3", plot_bgcolor="#ffffff",
        height=900, width=1350, margin=dict(t=150, l=60, r=60, b=60)
    )
    fig.update_xaxes(tickangle=-15)
    
    # Save to HTML file
    fig.write_html(save_path)
    print(f"  ✓ Saved interactive comparison dashboard to {save_path}")
    
    # Render natively in Jupyter Notebook without suppressing errors
    display(fig)



def plot_battery_plotly(rows, save_path=os.path.join(DOCS_DIR, "part14_battery_full_history.html")):
    regs = [r[0] for r in rows]
    A = [r[1] for r in rows]
    B = [r[2] for r in rows]
    rcol = {'Normal': '#1D9E75', 'Elevated': '#BA7517', 'Crisis': '#D85A30'}

    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=(
            "<b>Full-history battery (2020–2026): crisis included</b>",
            "<b>Across Germany's 10 GWh 2030 fleet</b>"
        ),
        horizontal_spacing=0.15
    )

    fig.add_trace(
        go.Bar(name='Naive (always trade)', x=regs, y=A, marker_color='#888780', text=[f"€{v:.0f}" for v in A], textposition='outside'),
        row=1, col=1
    )
    fig.add_trace(
        go.Bar(name='Bayesian utility (Trader C)', x=regs, y=B, marker_color=[rcol[r] for r in regs], text=[f"€{v:.0f}" for v in B], textposition='outside'),
        row=1, col=1
    )

    fleet = 10000
    fv = [(b - a) * 365 * fleet / 1e6 for _, a, b, _ in rows]
    fig.add_trace(
        go.Bar(
            name='Extra Fleet Value', x=regs, y=fv,
            marker_color=[rcol[r] for r in regs],
            text=[f"€{v:.0f}M" for v in fv],
            textposition='outside',
            showlegend=False
        ),
        row=1, col=2
    )

    fig.update_layout(
        title=dict(
            text="<b>Bayesian Stochastic Battery — Value Grows with Volatility</b>",
            font=dict(size=16), x=0.5, xanchor="center"
        ),
        paper_bgcolor="#f7f6f3", plot_bgcolor="#ffffff",
        barmode='group',
        height=650, width=1400,
        margin=dict(t=120, l=60, r=60, b=60),
        legend=dict(orientation="h", yanchor="bottom", y=1.12, xanchor="center", x=0.5)
    )
    fig.update_yaxes(title_text="Battery profit (€ per MWh per day)", row=1, col=1)
    fig.update_yaxes(title_text="Extra value per year (€ million)", row=1, col=2)

    fig.write_html(save_path)
    print(f"  ✓ Saved interactive battery dashboard to {save_path}")
    display(fig)


if __name__ == "__main__":
    df = pd.read_pickle(os.path.join(DOCS_DIR, 'data_part2.pkl'))
    with open(os.path.join(DOCS_DIR, 'selected_part4.json')) as f: features = json.load(f)

    train = df[df['datetime'] <= '2024-04-30 23:00'].copy()
    val   = df[(df['datetime'] >= '2024-05-01') & (df['datetime'] <= '2025-04-30 23:00')].copy()
    test  = df[df['datetime'] >= '2025-05-01'].copy()

    sX = StandardScaler(); X_tr = sX.fit_transform(train[features].values)
    X_val = sX.transform(val[features].values); X_te = sX.transform(test[features].values)
    z_tr = signed_log(train['price'].values); z_val = signed_log(val['price'].values)
    sz = StandardScaler(); z_tr_s = sz.fit_transform(z_tr.reshape(-1,1)).flatten()
    y_te = test['price'].values

    def zpred(model, X):
        mu_s,_ = model.predict(X); return sz.inverse_transform(mu_s.reshape(-1,1)).flatten()

    table = {}

    # ── 1. Gaussian DDNN, delta-method (baseline, LIVE) ──────────────────────
    print("\n[1/4] Baseline DDNN (Gaussian, delta-method)...")
    base = DDNN(len(features), hidden=[128,64,32], lr=0.002, epochs=300)
    base.fit(X_tr, z_tr_s, X_val, sz.transform(z_val.reshape(-1,1)).flatten())
    zp = zpred(base, X_te); mu = inv_signed_log(zp)
    sd = np.std(z_val - zpred(base, X_val)) * np.exp(np.abs(zp))
    z90 = stats.norm.ppf(0.95)
    table['Gaussian (delta)'] = interval_metrics(y_te, mu, mu-z90*sd, mu+z90*sd)

    # ── 2. Gaussian DDNN + log-space conformal (LIVE) ────────────────────────
    print("[2/4] + Log-space conformal...")
    cp = LogSpaceConformal(0.90); cp.fit(z_val, zpred(base, X_val), val['hour'].values)
    mu_c, lo_c, hi_c = cp.predict(zp, test['hour'].values)
    table['Gaussian+LogCP'] = interval_metrics(y_te, mu_c, lo_c, hi_c)

    # ── 3. Student-t (Adam) + log-space conformal (LIVE) ─────────────────────
    print("[3/4] Student-t (Adam) + Log-space conformal...")
    st = StudentTDNN_Adam(len(features), hidden=[64,32], lr=0.01, epochs=250)
    st.fit(X_tr, z_tr_s)
    zt_val = sz.inverse_transform(st.predict(X_val)[0].reshape(-1,1)).flatten()
    zt_te  = sz.inverse_transform(st.predict(X_te)[0].reshape(-1,1)).flatten()
    cp_t = LogSpaceConformal(0.90); cp_t.fit(z_val, zt_val, val['hour'].values)
    mu_t, lo_t, hi_t = cp_t.predict(zt_te, test['hour'].values)
    table['StudentT+LogCP'] = interval_metrics(y_te, mu_t, lo_t, hi_t)

    # ── 4. Rolling window (LIVE) ─────────────────────────────────────────────
    print("[4/4] Walk-forward rolling window...")
    rp_all, test_r, nb = walk_forward_rolling(df, features, 365, 60)
    valid = ~np.isnan(rp_all); rp = rp_all[valid]; rt = test_r['price'].values[valid]
    res = np.abs(signed_log(rt) - signed_log(rp)); q = np.quantile(res, 0.90)
    lo_r = inv_signed_log(signed_log(rp)-q); hi_r = inv_signed_log(signed_log(rp)+q)
    table[f'Rolling ({nb}x)'] = interval_metrics(rt, rp, lo_r, hi_r)

    # ── Print fair comparison table ──────────────────────────────────────────
    print("\n" + "="*82)
    print("  PART 14 — FAIR COMPARISON (all rows live, same test window May'25–Apr'26)")
    print("="*82)
    print(f"  {'Variant':<20}{'MAE':>9}{'PICP_90':>10}{'MPIW(med)':>11}{'MAACE':>9}")
    print("  " + "-"*76)
    for name, m in table.items():
        print(f"  {name:<20}{m['MAE']:>9.2f}{m['PICP_90']:>9.1%}"
              f"{m['MPIW_90_med']:>11.2f}{m['MAACE']:>8.2f}%")
    print("="*82)

    # ── Battery over FULL HISTORY (includes 2022 crisis) ─────────────────────
    print("\n  Battery — FULL HISTORY 2020–2026 (crisis included):")
    dfr = pd.read_pickle(os.path.join(DOCS_DIR, 'data_part7_regimes.pkl'))
    dfr = dfr.sort_values('datetime').reset_index(drop=True)
    if 'lag_24' not in dfr.columns:
        dfr['lag_24'] = dfr['price'].shift(24)
    brows = battery_full_history(dfr, features)
    print(f"  {'Regime':<10}{'Naive €/day':>14}{'BayesOpt €/day':>16}{'extra':>9}")
    for nm, a, b, dd in brows:
        print(f"  {nm:<10}{a:>14.2f}{b:>16.2f}{b-a:>9.2f}   [{dd} days]")

    # ── Save results and interactive Plotly charts ───────────────────────────
    np.save(os.path.join(DOCS_DIR, 'part14_results.npy'), {'table':table, 'battery':brows}, allow_pickle=True)
    plot_comparison_plotly(table)
    plot_battery_plotly(brows)
    print("\n✓ Part 14 complete with interactive Plotly dashboards saved to the working directory")


[1/4] Baseline DDNN (Gaussian, delta-method)...

done. Final NLL=-1.4257.. 
[2/4] + Log-space conformal...
[3/4] Student-t (Adam) + Log-space conformal...
[4/4] Walk-forward rolling window...

  PART 14 — FAIR COMPARISON (all rows live, same test window May'25–Apr'26)
  Variant                   MAE   PICP_90  MPIW(med)    MAACE
  ----------------------------------------------------------------------------
  Gaussian (delta)        16.61    90.5%     184.36   20.49%
  Gaussian+LogCP          16.61    87.2%      93.12   13.40%
  StudentT+LogCP          15.87    88.1%     105.39   21.26%
  Rolling (7x)            17.63    90.0%     209.86   24.66%

  Battery — FULL HISTORY 2020–2026 (crisis included):
  Regime       Naive €/day  BayesOpt €/day    extra
  Normal             42.33           45.38     3.05   [1280 days]
  Elevated           62.80           73.26    10.46   [800 days]
  Crisis            101.85          121.43    19.58   [224 days]
  ✓ Saved interactive comparison dashboard

  ✓ Saved interactive battery dashboard to part14_battery_full_history.html



✓ Part 14 complete with interactive Plotly dashboards saved to 


## Mathematical Formulation of Part 15: Scale-Aware Conformal Calibration

The Student-t model of Part 14 is highly accurate (MAE ≈ €14) but its intervals are **miscalibrated** (MAACE ≈ 20%), because a single conformal quantile is applied uniformly to every hour, ignoring that the model is more confident at some hours than others.

---

### 1. The Problem with Plain Conformal

Plain conformal uses one width everywhere: $\text{band}_i = \hat{q}$. This over-covers easy hours and under-covers hard ones.

---

### 2. Normalized (Scale-Aware) Conformal

We normalize each residual by the model's **own predicted uncertainty** $\hat{\sigma}_i$ before taking the quantile. The nonconformity score becomes:

$$r_i = \frac{|z_i - \hat{z}_i|}{\hat{\sigma}_i}$$

with calibrated quantile $\hat{q} = \text{Quantile}_{1-\alpha}(\{r_i\})$. The per-point half-width is then scaled back:

$$\text{band}_i = \hat{q} \cdot \hat{\sigma}_i$$

so the interval is **wider where the model is genuinely uncertain and tighter where it is confident**. This preserves sharpness on average while dramatically improving calibration.

---

### 3. Calibration Metric (MAACE)

Calibration quality is measured by the Mean Absolute Average Calibration Error across nominal levels $\ell \in \{0.1, \dots, 0.9\}$:

$$\text{MAACE} = \frac{1}{|\mathcal{L}|}\sum_{\ell \in \mathcal{L}} \big|\,\widehat{\text{cov}}(\ell) - \ell\,\big|$$

where $\widehat{\text{cov}}(\ell)$ is the empirical coverage of the nominal-$\ell$ interval. The scale-aware fix reduces MAACE from ≈31% to ≈18%, while keeping MAE unchanged at €14.77.

In [34]:
"""
═══════════════════════════════════════════════════════════════════════════
 PART 15 — IMPROVED, WELL-CALIBRATED STUDENT-T MODEL
═══════════════════════════════════════════════════════════════════════════
 Task 2: fix the model for lower loss AND better calibration.

 The problem from Part 14:
   The Adam-trained Student-t had excellent accuracy (MAE ~€14) but its
   calibration was poor (MAACE ~20%) because we applied ONE shared conformal
   quantile to every hour, ignoring that the model is more certain some hours
   and less certain others.

 The fix — NORMALIZED (scale-aware) CONFORMAL:
   Instead of  band = q  (same width everywhere),
   we use      band = q * sigma_i   (wider where the model itself is unsure).
   We compute the residual RELATIVE to each point's own predicted sigma, take
   the quantile of those normalized residuals, then scale back per point.
   This keeps the sharp average width but puts the width where it's needed,
   which fixes calibration.

 We also combine the Student-t WITH the rolling window (fresh + fat-tailed),
 then feed the best model into the per-regime battery.
═══════════════════════════════════════════════════════════════════════════
"""
"""
═══════════════════════════════════════════════════════════════════════════
 PART 15 — IMPROVED, WELL-CALIBRATED STUDENT-T MODEL (PLOTLY)
═══════════════════════════════════════════════════════════════════════════
 Task 2: fix the model for lower loss AND better calibration using 
 normalized (scale-aware) conformal prediction in signed-log space.
 Replaces static Matplotlib outputs with an interactive Plotly dashboard.
═══════════════════════════════════════════════════════════════════════════
"""



pio.renderers.default = "iframe"

# ════════════════════════════════════════════════════════════════════════════
# Metrics from an explicit interval (mean + median width)
# ════════════════════════════════════════════════════════════════════════════
def interval_metrics(y, mu, lower, upper):
    mae  = float(np.mean(np.abs(y - mu)))
    rmse = float(np.sqrt(np.mean((y - mu)**2)))
    picp = float(np.mean((y >= lower) & (y <= upper)))
    mpiw = float(np.mean(upper - lower))
    mpiw_med = float(np.median(upper - lower))
    sigma = np.maximum((upper - lower)/(2*stats.norm.ppf(0.95)), 1e-6)
    levels = np.arange(0.1, 1.0, 0.1); maace = 0.0
    for lv in levels:
        z = stats.norm.ppf((1+lv)/2)
        cov = np.mean((y >= mu - z*sigma) & (y <= mu + z*sigma))
        maace += abs(cov - lv)
    maace = maace/len(levels)*100
    return {"MAE":mae,"RMSE":rmse,"PICP_90":picp,"MPIW_90":mpiw,
            "MPIW_90_med":mpiw_med,"MAACE":maace,
            "mu":mu,"lower":lower,"upper":upper}


# ════════════════════════════════════════════════════════════════════════════
# Student-t head trained with Adam (with per-point sigma out)
# ════════════════════════════════════════════════════════════════════════════
class StudentTDNN_Adam:
    def __init__(self, input_dim, hidden=[64,32], lr=0.01, epochs=250):
        self.lr, self.epochs = lr, epochs
        h1, h2 = hidden
        self.P = {
            'W1': np.random.randn(input_dim,h1)*np.sqrt(2/input_dim), 'b1': np.zeros(h1),
            'W2': np.random.randn(h1,h2)*np.sqrt(2/h1), 'b2': np.zeros(h2),
            'Wm': np.random.randn(h2,1)*0.01, 'bm': np.zeros(1),
            'Ws': np.random.randn(h2,1)*0.01, 'bs': np.zeros(1),
            'Wn': np.random.randn(h2,1)*0.01, 'bn': np.zeros(1),
        }
        self.m = {k: np.zeros_like(v) for k,v in self.P.items()}
        self.v = {k: np.zeros_like(v) for k,v in self.P.items()}
        self.t = 0
    def _relu(self,x): return np.maximum(0,x)
    def _fwd(self,X):
        z1=X@self.P['W1']+self.P['b1']; h1=self._relu(z1)
        z2=h1@self.P['W2']+self.P['b2']; h2=self._relu(z2)
        mu=(h2@self.P['Wm']+self.P['bm']).flatten()
        lsig=np.clip((h2@self.P['Ws']+self.P['bs']).flatten(),-4,4)
        lnu=np.clip((h2@self.P['Wn']+self.P['bn']).flatten(),0.3,3.5)
        return mu, np.exp(lsig)+1e-3, np.exp(lnu)+1.0, (z1,h1,z2,h2)
    def _adam(self,g):
        self.t+=1; b1,b2,e=0.9,0.999,1e-8
        for k in self.P:
            gr=g.get(k,0)
            self.m[k]=b1*self.m[k]+(1-b1)*gr
            self.v[k]=b2*self.v[k]+(1-b2)*(gr**2)
            mh=self.m[k]/(1-b1**self.t); vh=self.v[k]/(1-b2**self.t)
            self.P[k]-=self.lr*mh/(np.sqrt(vh)+e)
    def fit(self,X,y):
        n=len(y)
        for _ in range(self.epochs):
            mu,sig,nu,(z1,h1,z2,h2)=self._fwd(X)
            z=(y-mu)/sig; w=(nu+1)/(nu+z**2)
            dmu=-(w*z/sig)/n; dls=(1-w*z**2)/n
            g={}
            g['Wm']=h2.T@dmu.reshape(-1,1); g['bm']=np.array([dmu.sum()])
            g['Ws']=h2.T@dls.reshape(-1,1); g['bs']=np.array([dls.sum()])
            g['Wn']=np.zeros_like(self.P['Wn']); g['bn']=np.zeros_like(self.P['bn'])
            dh2=(dmu.reshape(-1,1)@self.P['Wm'].T+dls.reshape(-1,1)@self.P['Ws'].T)*(z2>0)
            g['W2']=h1.T@dh2; g['b2']=dh2.sum(axis=0)
            dh1=(dh2@self.P['W2'].T)*(z1>0)
            g['W1']=X.T@dh1; g['b1']=dh1.sum(axis=0)
            self._adam(g)
        return self
    def predict(self,X):
        mu,sig,nu,_=self._fwd(X); return mu,sig,nu


# ════════════════════════════════════════════════════════════════════════════
# Scale-Aware Normalized Conformal Wrapper
# ════════════════════════════════════════════════════════════════════════════
class NormalizedConformal:
    def __init__(self, coverage=0.90):
        self.coverage = coverage
        self.q = None
    def fit(self, z_true, z_pred, sigma):
        sigma = np.maximum(sigma, 1e-6)
        norm_res = np.abs(z_true - z_pred) / sigma
        self.q = np.quantile(norm_res, self.coverage)
    def predict(self, z_pred, sigma):
        sigma = np.maximum(sigma, 1e-6)
        half = self.q * sigma
        mu  = inv_signed_log(z_pred)
        lo  = inv_signed_log(z_pred - half)
        hi  = inv_signed_log(z_pred + half)
        return mu, lo, hi


# Plain conformal (for comparison)
class PlainConformal:
    def __init__(self, coverage=0.90): self.coverage=coverage; self.q=None
    def fit(self, z_true, z_pred, sigma=None):
        self.q = np.quantile(np.abs(z_true - z_pred), self.coverage)
    def predict(self, z_pred, sigma=None):
        return (inv_signed_log(z_pred),
                inv_signed_log(z_pred - self.q),
                inv_signed_log(z_pred + self.q))


def plot_calibration_fix_plotly(before, after, save_path=os.path.join(DATA_DIR, "part15_calibration_fix.html")):
    print("\nGenerating interactive Part 15 calibration fix Plotly dashboard...")
    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=(
            "<b>1. Reliability Calibration Curves</b>",
            "<b>2. Before vs. After 4-Metric Comparison</b>"
        ),
        horizontal_spacing=0.12
    )

    # Subplot 1: Reliability calibration curves
    levels = np.arange(0.1, 1.0, 0.1) * 100
    for res, name, c in [(before, 'Before (Plain Conformal)', '#888780'),
                         (after, 'After (Scale-Aware Conformal)', '#1D9E75')]:
        mu, lo, hi = res['mu'], res['lower'], res['upper']
        sig = np.maximum((hi - lo) / (2 * stats.norm.ppf(0.95)), 1e-6)
        covs = []
        for lv in np.arange(0.1, 1.0, 0.1):
            z = stats.norm.ppf((1 + lv) / 2)
            covs.append(np.mean((res['_y'] >= mu - z * sig) & (res['_y'] <= mu + z * sig)) * 100)
        
        fig.add_trace(
            go.Scatter(x=levels, y=covs, mode='lines+markers', name=name, line=dict(color=c, width=2)),
            row=1, col=1
        )

    fig.add_trace(
        go.Scatter(x=[10, 90], y=[10, 90], mode='lines', line=dict(color='black', dash='dash', width=1.5), name='Ideal Calibration', showlegend=False),
        row=1, col=1
    )

    # Subplot 2: Metric Comparison Bar Chart
    metrics = ['MAE', 'PICP_90', 'MPIW_90_med', 'MAACE']
    labels = ['MAE (€)', 'PICP (%)', 'MPIW med (€)', 'MAACE (%)']
    bvals = [before['MAE'], before['PICP_90']*100, before['MPIW_90_med'], before['MAACE']]
    avals = [after['MAE'], after['PICP_90']*100, after['MPIW_90_med'], after['MAACE']]

    fig.add_trace(
        go.Bar(x=labels, y=bvals, name='Before', marker_color='#888780', text=[f"{v:.1f}" for v in bvals], textposition='outside'),
        row=1, col=2
    )
    fig.add_trace(
        go.Bar(x=labels, y=avals, name='After', marker_color='#1D9E75', text=[f"{v:.1f}" for v in avals], textposition='outside'),
        row=1, col=2
    )

    fig.update_layout(
        title=dict(
            text="<b>Part 15 — Fixing Student-t Calibration with Scale-Aware Conformal</b>",
            font=dict(size=16), x=0.5, xanchor="center"
        ),
        paper_bgcolor="#f7f6f3", plot_bgcolor="#ffffff",
        barmode='group',
        height=650, width=1400,
        legend=dict(orientation="h", yanchor="bottom", y=1.12, xanchor="center", x=0.5)
    )

    fig.update_xaxes(title_text="Claimed Confidence (%)", row=1, col=1)
    fig.update_yaxes(title_text="Actual Coverage (%)", row=1, col=1)
    fig.update_yaxes(title_text="Metric Score", row=1, col=2)

    fig.write_html(save_path)
    print(f"  ✓ Saved interactive calibration dashboard to {save_path}")
    fig.show()


if __name__ == "__main__":
    df = pd.read_pickle(os.path.join(DATA_DIR, "data_part2.pkl"))
    with open(os.path.join(DATA_DIR, "selected_part4.json")) as f: features = json.load(f)

    train = df[df['datetime'] <= '2024-04-30 23:00'].copy()
    val   = df[(df['datetime'] >= '2024-05-01') & (df['datetime'] <= '2025-04-30 23:00')].copy()
    test  = df[df['datetime'] >= '2025-05-01'].copy()

    sX = StandardScaler(); X_tr = sX.fit_transform(train[features].values)
    X_val = sX.transform(val[features].values); X_te = sX.transform(test[features].values)
    z_tr = signed_log(train['price'].values); z_val = signed_log(val['price'].values)
    sz = StandardScaler(); z_tr_s = sz.fit_transform(z_tr.reshape(-1,1)).flatten()
    y_te = test['price'].values

    print("\nTraining Student-t (Adam)...")
    st = StudentTDNN_Adam(len(features), hidden=[64,32], lr=0.01, epochs=250)
    st.fit(X_tr, z_tr_s)

    def predict_z_sigma(X):
        mu_s, sig_s, nu = st.predict(X)
        mu_z = sz.inverse_transform(mu_s.reshape(-1,1)).flatten()
        sig_z = sig_s * sz.scale_[0]
        return mu_z, sig_z
    
    zt_val, sig_val = predict_z_sigma(X_val)
    zt_te,  sig_te  = predict_z_sigma(X_te)

    # ── BEFORE: plain conformal ───────────────────────────────────────────────
    pc = PlainConformal(0.90); pc.fit(z_val, zt_val)
    mu_b, lo_b, hi_b = pc.predict(zt_te)
    before = interval_metrics(y_te, mu_b, lo_b, hi_b); before['_y'] = y_te

    # ── AFTER: scale-aware conformal ─────────────────────────────────────────
    nc = NormalizedConformal(0.90); nc.fit(z_val, zt_val, sig_val)
    mu_a, lo_a, hi_a = nc.predict(zt_te, sig_te)
    after = interval_metrics(y_te, mu_a, lo_a, hi_a); after['_y'] = y_te

    print("\n" + "="*70)
    print("  CALIBRATION FIX — BEFORE vs AFTER (same Student-t model)")
    print("="*70)
    print(f"  {'Metric':<14}{'BEFORE (plain)':>16}{'AFTER (scale-aware)':>22}")
    print("  " + "-"*54)
    for k, lab in [('MAE','MAE (€)'),('PICP_90','PICP 90%'),
                   ('MPIW_90_med','MPIW median (€)'),('MAACE','MAACE (%)')]:
        fb = (lambda v: f"{v:.1%}") if k=='PICP_90' else (lambda v: f"{v:.2f}")
        print(f"  {lab:<14}{fb(before[k]):>16}{fb(after[k]):>22}")
    print("="*70)
    print("  Target: PICP near 90%, MAACE LOW, MAE unchanged (still sharp).")

    np.save(os.path.join(DATA_DIR, "part15_results.npy"), {'before': before, 'after': after}, allow_pickle=True)
    plot_calibration_fix_plotly(before, after, os.path.join(DATA_DIR, "part15_calibration_fix.html"))
    print("\n✓ Part 15 multi-model scale-aware conformal calibration complete with interactive Plotly dashboard!")


Training Student-t (Adam)...

  CALIBRATION FIX — BEFORE vs AFTER (same Student-t model)
  Metric          BEFORE (plain)   AFTER (scale-aware)
  ------------------------------------------------------
  MAE (€)                  14.51                 14.51
  PICP 90%                 87.7%                 93.0%
  MPIW median (€)          194.56                 74.51
  MAACE (%)                25.01                 14.17
  Target: PICP near 90%, MAACE LOW, MAE unchanged (still sharp).

Generating interactive Part 15 calibration fix Plotly dashboard...
  ✓ Saved interactive calibration dashboard to part15_calibration_fix.html



✓ Part 15 multi-model scale-aware conformal calibration complete with interactive Plotly dashboard!


## Mathematical Formulation of Part 15b: Multi-Level Conformal Calibration

Scale-aware conformal (Part 15) fixes *where* the band is placed but still calibrates a single quantile at the 90% level. Part 15b calibrates a **separate normalized quantile at every confidence level**, flattening the entire calibration curve.

---

### Per-Level Quantile Fitting

For each nominal level $\ell \in \{0.1, 0.2, \dots, 0.9\}$, we fit an independent normalized quantile from the calibration residuals $r_i = |z_i - \hat{z}_i| / \hat{\sigma}_i$:

$$\hat{q}(\ell) = \text{Quantile}_{\ell}\big(\{r_i\}\big)$$

The interval at level $\ell$ for test point $i$ is:

$$\mathcal{C}_\ell(x_i) = \left[\,\operatorname{slog}^{-1}\!\big(\hat{z}_i - \hat{q}(\ell)\,\hat{\sigma}_i\big),\; \operatorname{slog}^{-1}\!\big(\hat{z}_i + \hat{q}(\ell)\,\hat{\sigma}_i\big)\,\right]$$

Because the mapping between claimed and actual coverage is corrected **independently at each level**, the empirical calibration curve becomes nearly diagonal. This reduces MAACE all the way to **≈5%** while retaining the Student-t's sharp intervals and €14.77 MAE — achieving both accuracy and trustworthy uncertainty simultaneously.

In [35]:
"""
═══════════════════════════════════════════════════════════════════════════
 PART 15b — MULTI-LEVEL NORMALISED CONFORMAL CALIBRATION (PLOTLY)
═══════════════════════════════════════════════════════════════════════════
 Pushes calibration lower (flat calibration) by fitting a separate 
 normalised quantile for each confidence level (10% to 90%), combined with 
 the Student-t model. Renders interactive Plotly dashboards.
═══════════════════════════════════════════════════════════════════════════
"""




pio.renderers.default = "iframe"
# ════════════════════════════════════════════════════════════════════════════
# MULTI-LEVEL NORMALISED CONFORMAL CLASS
# ════════════════════════════════════════════════════════════════════════════
class MultiLevelNormalizedConformal:
    """Fit a separate normalised quantile for each confidence level -> flat calibration."""
    def __init__(self):
        self.q = {}
        
    def fit(self, z_true, z_pred, sigma):
        sigma = np.maximum(sigma, 1e-6)
        nr = np.abs(z_true - z_pred) / sigma
        for lv in np.arange(0.1, 1.0, 0.1):
            self.q[round(lv, 1)] = np.quantile(nr, lv)
            
    def band(self, z_pred, sigma, level):
        half = self.q[round(level, 1)] * np.maximum(sigma, 1e-6)
        return inv_signed_log(z_pred - half), inv_signed_log(z_pred + half)


# ════════════════════════════════════════════════════════════════════════════
# PLOTLY MULTI-LEVEL DASHBOARD
# ════════════════════════════════════════════════════════════════════════════
def plot_multilevel_calibration_plotly(mlc, y_te, zt_te, sig_te, m, maace, save_path=os.path.join(DATA_DIR, "part15b_multilevel_calibration.html")):
    print("\nGenerating interactive Multi-Level Conformal Plotly dashboard...")
    
    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=(
            "<b>1. Multi-Level Reliability Calibration Curve (Flat Calibration)</b>",
            "<b>2. Multi-Level Performance Metrics Summary</b>"
        ),
        horizontal_spacing=0.12
    )

    # Subplot 1: Calibration curve across levels (10% to 90%)
    levels = np.arange(0.1, 1.0, 0.1) * 100
    empirical_covs = []
    
    for lv in np.arange(0.1, 1.0, 0.1):
        lo, hi = mlc.band(zt_te, sig_te, lv)
        cov = np.mean((y_te >= lo) & (y_te <= hi)) * 100
        empirical_covs.append(cov)

    fig.add_trace(
        go.Scatter(
            x=levels, y=empirical_covs, mode='lines+markers',
            name='Multi-Level Conformal',
            line=dict(color='#1D9E75', width=2.5),
            marker=dict(size=8)
        ),
        row=1, col=1
    )

    fig.add_trace(
        go.Scatter(
            x=[10, 90], y=[10, 90], mode='lines',
            line=dict(color='black', dash='dash', width=1.5),
            name='Ideal Calibration', showlegend=False
        ),
        row=1, col=1
    )

    # Subplot 2: Metrics Summary Bar Chart
    metrics_labels = ['MAE (€)', 'PICP 90% (%)', 'MPIW Median (€)', 'MAACE (%)']
    metrics_values = [m['MAE'], m['PICP_90'] * 100, m['MPIW_90_med'], maace]

    fig.add_trace(
        go.Bar(
            x=metrics_labels, y=metrics_values,
            marker_color=['#378ADD', '#1D9E75', '#BA7517', '#D85A30'],
            text=[f"{v:.1f}" for v in metrics_values],
            textposition='outside',
            showlegend=False,
            hovertemplate="<b>%{x}</b><br>Value: %{y:.2f}<extra></extra>"
        ),
        row=1, col=2
    )

    fig.update_layout(
        title=dict(
            text="<b>Part 15b — Multi-Level Calibrated Student-t Dashboard</b><br><sup>Achieving flat reliability calibration across confidence intervals</sup>",
            font=dict(size=16), x=0.5, xanchor="center"
        ),
        paper_bgcolor="#f7f6f3", plot_bgcolor="#ffffff",
        height=650, width=1400,
        legend=dict(orientation="h", yanchor="bottom", y=1.12, xanchor="center", x=0.5)
    )

    fig.update_xaxes(title_text="Claimed Confidence (%)", row=1, col=1)
    fig.update_yaxes(title_text="Actual Coverage (%)", row=1, col=1)
    fig.update_yaxes(title_text="Score / Value", row=1, col=2)
    fig.update_yaxes(showgrid=True, gridwidth=1, gridcolor="rgba(0,0,0,0.1)", row=1, col=2)

    fig.write_html(save_path)
    print(f"  ✓ Saved interactive multi-level dashboard to {save_path}")
    fig.show()


if __name__ == "__main__":
    df = pd.read_pickle(os.path.join(DATA_DIR, "data_part2.pkl"))
    with open(os.path.join(DATA_DIR, "selected_part4.json")) as f:
        features = json.load(f)

    train = df[df['datetime'] <= '2024-04-30 23:00'].copy()
    val   = df[(df['datetime'] >= '2024-05-01') & (df['datetime'] <= '2025-04-30 23:00')].copy()
    test  = df[df['datetime'] >= '2025-05-01'].copy()

    sX = StandardScaler()
    X_tr = sX.fit_transform(train[features].values)
    X_val = sX.transform(val[features].values)
    X_te = sX.transform(test[features].values)

    z_tr = signed_log(train['price'].values)
    z_val = signed_log(val['price'].values)
    
    sz = StandardScaler()
    z_tr_s = sz.fit_transform(z_tr.reshape(-1, 1)).flatten()
    y_te = test['price'].values

    print("Training Student-t...")
    st = StudentTDNN_Adam(len(features), hidden=[64, 32], lr=0.01, epochs=250)
    st.fit(X_tr, z_tr_s)

    def pzs(X):
        mu_s, sig_s, nu = st.predict(X)
        return sz.inverse_transform(mu_s.reshape(-1, 1)).flatten(), sig_s * sz.scale_[0]

    zt_val, sig_val = pzs(X_val)
    zt_te, sig_te = pzs(X_te)

    # Fit multi-level normalized conformal wrapper
    mlc = MultiLevelNormalizedConformal()
    mlc.fit(z_val, zt_val, sig_val)
    
    mu = inv_signed_log(zt_te)
    
    # 90% interval for standard headline metrics
    lo90, hi90 = mlc.band(zt_te, sig_te, 0.9)
    m = interval_metrics(y_te, mu, lo90, hi90)
    m['_y'] = y_te

    # Compute proper MAACE using the per-level calibrated bands
    maace = 0.0
    for lv in np.arange(0.1, 1.0, 0.1):
        lo, hi = mlc.band(zt_te, sig_te, lv)
        cov = np.mean((y_te >= lo) & (y_te <= hi))
        maace += abs(cov - lv)
    maace = maace / 9 * 100

    print("\n" + "="*60)
    print("  MULTI-LEVEL CALIBRATED STUDENT-T (final)")
    print("="*60)
    print(f"  MAE           : €{m['MAE']:.2f}")
    print(f"  PICP 90%      : {m['PICP_90']:.1%}")
    print(f"  MPIW median   : €{m['MPIW_90_med']:.2f}")
    print(f"  MAACE (proper): {maace:.2f}%   <- calibrated per level")
    print("="*60)

    # Save results & generate Plotly visualization
    np.save(os.path.join(DATA_DIR, "part15b_results.npy"), {'metrics': m, 'maace': maace}, allow_pickle=True)
    plot_multilevel_calibration_plotly(mlc, y_te, zt_te, sig_te, m, maace, os.path.join(DATA_DIR, "part15b_multilevel_calibration.html"))
    print("\n✓ Part 15b multi-level conformal analysis complete with interactive dashboard saved!")

Training Student-t...

  MULTI-LEVEL CALIBRATED STUDENT-T (final)
  MAE           : €14.93
  PICP 90%      : 92.2%
  MPIW median   : €71.67
  MAACE (proper): 0.88%   <- calibrated per level

Generating interactive Multi-Level Conformal Plotly dashboard...
  ✓ Saved interactive multi-level dashboard to part15b_multilevel_calibration.html



✓ Part 15b multi-level conformal analysis complete with interactive dashboard saved!


## Mathematical Formulation of Part 16: Per-Regime Battery on the Best Model

Part 16 feeds the **calibrated Student-t** predictions (Part 15) into the per-regime battery, run over the **full history (2020–2026)** so that the Normal, Elevated, and Crisis regimes are all represented — unlike the test-only window, which contained no crisis days.

---

### 1. Calibrated Predictive Inputs

For every hour the model provides a euro-space mean $\hat{\mu}_t$ and a scale-aware euro-space uncertainty $\hat{\sigma}_t$ derived from the normalized conformal quantile.

---

### 2. Regime-Conditioned Arbitrage

For each day $d$ with dominant regime $r$, the naive and uncertainty-aware traders select buy/sell hours from the predicted prices. The uncertainty-aware ("smart") trader executes only when the expected price gap exceeds a multiple of the combined uncertainty:

$$\text{trade} \iff \big(\hat{\mu}_s - \hat{\mu}_b\big) > k\,\sqrt{\hat{\sigma}_b^2 + \hat{\sigma}_c^2}$$

Because the Student-t's uncertainty is now **trustworthy** (MAACE ≈5%), the confidence gate makes reliable decisions: it keeps genuinely profitable trades and skips genuinely risky ones.

---

### 3. Regime-Dependent Value

The extra profit of the smart trader over the naive trader is reported per regime and scaled to Germany's planned 10 GWh 2030 fleet. The advantage grows monotonically with volatility — from a few €/MWh/day in Normal conditions to tens of €/MWh/day in Crisis — quantifying the economic value of well-calibrated uncertainty precisely when the grid needs storage most.

In [36]:
"""
═══════════════════════════════════════════════════════════════════════════
 PART 16 — PER-REGIME BATTERY WITH THE IMPROVED STUDENT-T MODEL (PLOTLY)
═══════════════════════════════════════════════════════════════════════════
 Feeds the calibrated Student-t model's predictions and scale-aware sigma 
 into the per-regime battery over full history (2020-2026, including the 2022 crisis).
 Renders interactive Plotly dashboards saved to the working directory.
═══════════════════════════════════════════════════════════════════════════
"""




pio.renderers.default = "iframe"
# ════════════════════════════════════════════════════════════════════════════
# 1. Train Student-t and produce calibrated mu + sigma over FULL history
# ════════════════════════════════════════════════════════════════════════════
def make_calibrated_predictions(df, features):
    """
    Train Student-t on the training period, then predict mu + sigma for EVERY
    hour in the dataset (so all regimes incl. 2022 crisis are covered).
    Sigma is scale-aware calibrated using the validation residuals.
    """
    train = df[df['datetime'] <= '2024-04-30 23:00'].copy()
    val   = df[(df['datetime'] >= '2024-05-01') & (df['datetime'] <= '2025-04-30 23:00')].copy()

    sX = StandardScaler(); X_tr = sX.fit_transform(train[features].values)
    z_tr = signed_log(train['price'].values)
    sz = StandardScaler(); z_tr_s = sz.fit_transform(z_tr.reshape(-1,1)).flatten()

    print("  Training Student-t (Adam) for the battery...")
    st = StudentTDNN_Adam(len(features), hidden=[64,32], lr=0.01, epochs=250)
    st.fit(X_tr, z_tr_s)

    def predict_z_sigma(frame):
        X = sX.transform(frame[features].values)
        mu_s, sig_s, nu = st.predict(X)
        mu_z = sz.inverse_transform(mu_s.reshape(-1,1)).flatten()
        sig_z = sig_s * sz.scale_[0]
        return mu_z, sig_z

    # Calibrate the scale factor on validation (normalized conformal, 90%)
    zt_val, sig_val = predict_z_sigma(val)
    z_val = signed_log(val['price'].values)
    norm_res = np.abs(z_val - zt_val) / np.maximum(sig_val, 1e-6)
    q90 = np.quantile(norm_res, 0.90)    # scale factor for an 80%-style band

    # Predict over the FULL dataframe
    zt_all, sig_all = predict_z_sigma(df)
    mu_eur = inv_signed_log(zt_all)
    
    # euro-space sigma: half the calibrated log band mapped through the transform
    half = q90 * np.maximum(sig_all, 1e-6)
    hi = inv_signed_log(zt_all + half); lo = inv_signed_log(zt_all - half)
    sigma_eur = (hi - lo) / (2 * 1.28)    # convert ~80% band to a sigma
    sigma_eur = np.clip(sigma_eur, 1.0, 400.0)
    return mu_eur, sigma_eur


# ════════════════════════════════════════════════════════════════════════════
# 2. Per-regime battery (naive vs smart) using the calibrated predictions
# ════════════════════════════════════════════════════════════════════════════
def run_battery_per_regime(mu, sigma, price, regime, xi=0.90, k=0.1):
    """
    Naive  : always buy predicted-cheapest, sell predicted-priciest hour.
    Smart  : trade only if (predicted gap) > k*(sigma_buy + sigma_sell).
    """
    n_days = len(mu)//24
    pn={0:0.,1:0.,2:0.}; ps={0:0.,1:0.,2:0.}; days={0:0,1:0,2:0}
    skipped={0:0,1:0,2:0}
    for d in range(n_days):
        sl = slice(d*24,(d+1)*24)
        mu_d, sig_d, true_d = mu[sl], sigma[sl], price[sl]
        reg_d = regime[sl]
        if len(mu_d) < 24: continue
        rg = int(np.bincount(reg_d).argmax()); days[rg]+=1
        b = int(np.argmin(mu_d)); s = int(np.argmax(mu_d))
        if s <= b: b,s = min(b,s),max(b,s)
        realised = true_d[s]*xi - true_d[b]
        pn[rg] += realised                                      # naive: always
        gap = mu_d[s] - mu_d[b]
        doubt = k * np.sqrt(sig_d[b]**2 + sig_d[s]**2)      # combined uncertainty
        if gap > doubt:
            ps[rg] += realised                                  # smart: only if sure
        else:
            skipped[rg] += 1
    names={0:'Normal',1:'Elevated',2:'Crisis'}; rows=[]
    for r in [0,1,2]:
        dd=max(days[r],1)
        rows.append((names[r], pn[r]/dd, ps[r]/dd, days[r], skipped[r]))
    return rows


# ════════════════════════════════════════════════════════════════════════════
# INTERACTIVE PLOTLY DASHBOARD
# ════════════════════════════════════════════════════════════════════════════
def plot_battery_plotly(rows, save_path=os.path.join(DATA_DIR, "part16_battery_calibrated.html")):
    print("\nGenerating interactive Part 16 battery Plotly dashboard...")
    regs = [r[0] for r in rows]
    A = [r[1] for r in rows]
    B = [r[2] for r in rows]
    rcol = {'Normal': '#1D9E75', 'Elevated': '#BA7517', 'Crisis': '#D85A30'}

    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=(
            "<b>Per-Regime Battery Profit (€ per MWh per day)</b>",
            "<b>Annual Extra Value Across Germany's 10 GWh Fleet</b>"
        ),
        horizontal_spacing=0.12
    )

    # Subplot 1: Daily Profit per MWh
    fig.add_trace(
        go.Bar(
            name='Naive (always trade)', x=regs, y=A,
            marker_color='#888780',
            text=[f"€{v:.0f}" for v in A], textposition='outside'
        ),
        row=1, col=1
    )
    fig.add_trace(
        go.Bar(
            name='Smart (trade when sure)', x=regs, y=B,
            marker_color=[rcol[r] for r in regs],
            text=[f"€{v:.0f}" for v in B], textposition='outside'
        ),
        row=1, col=1
    )

    # Subplot 2: Annual Fleet Value Added
    fleet = 10000
    fv = [(b - a) * 365 * fleet / 1e6 for _, a, b, _, _ in rows]
    max_fv = max(fv) if fv else 10
    
    fig.add_trace(
        go.Bar(
            name='Extra Fleet Value', x=regs, y=fv,
            marker_color=[rcol[r] for r in regs],
            text=[f"€{v:.0f}M" for v in fv],
            textposition='outside',
            showlegend=False,
            hovertemplate="<b>%{x}</b><br>Extra Value: €%{y:.1f}M/year<extra></extra>"
        ),
        row=1, col=2
    )

    fig.update_layout(
        title=dict(
            text="<b>Part 16 — Money Story on the Best Model: Calibrated Student-t + Per-Regime Battery</b>",
            font=dict(size=16), x=0.5, xanchor="center"
        ),
        paper_bgcolor="#f7f6f3", plot_bgcolor="#ffffff",
        barmode='group',
        height=650, width=1400,
        legend=dict(orientation="h", yanchor="bottom", y=1.12, xanchor="center", x=0.5)
    )

    fig.update_yaxes(title_text="Battery Profit (€ / MWh / day)", row=1, col=1)
    fig.update_yaxes(title_text="Extra Value per Year (€ Million)", row=1, col=2, range=[0, max_fv * 1.25])

    fig.write_html(save_path)
    print(f"  ✓ Saved interactive Part 16 dashboard to {save_path}")
    fig.show()


if __name__ == "__main__":
    df = pd.read_pickle(os.path.join(DATA_DIR, "data_part2.pkl"))
    with open(os.path.join(DATA_DIR, "selected_part4.json")) as f:
        features = json.load(f)

    # Bring in regimes (aligned by datetime)
    dfr = pd.read_pickle(os.path.join(DATA_DIR, "data_part7_regimes.pkl"))[['datetime','regime']].drop_duplicates('datetime')
    df = df.merge(dfr, on='datetime', how='left')
    df['regime'] = df['regime'].fillna(0).astype(int)
    df = df.sort_values('datetime').reset_index(drop=True)

    print("\nGenerating calibrated predictions over full history...")
    mu, sigma = make_calibrated_predictions(df, features)
    price = df['price'].values
    regime = df['regime'].values

    print("\nRunning per-regime battery (naive vs smart)...")
    rows = run_battery_per_regime(mu, sigma, price, regime)

    print("\n" + "="*74)
    print("  PART 16 — PER-REGIME BATTERY ON THE CALIBRATED STUDENT-T")
    print("="*74)
    print(f"  {'Regime':<10}{'Naive €/day':>13}{'Smart €/day':>13}{'Extra':>9}{'Days':>7}{'Skipped':>9}")
    print("  " + "-"*68)
    for nm, a, b, dd, sk in rows:
        print(f"  {nm:<10}{a:>13.2f}{b:>13.2f}{b-a:>9.2f}{dd:>7}{sk:>9}")
    print("="*74)
    
    fleet = 10000
    print("\n  ANNUAL FLEET VALUE (Germany 10 GWh by 2030):")
    for nm, a, b, dd, sk in rows:
        print(f"    {nm:<10}: +€{b-a:5.2f}/MWh/day → €{(b-a)*365*fleet/1e6:6.1f}M/yr extra")

    np.save(os.path.join(DATA_DIR, "part16_results.npy"), {'rows': rows}, allow_pickle=True)
    plot_battery_plotly(rows, os.path.join(DATA_DIR, "part16_battery_calibrated.html"))
    print("\n✓ Task 3 complete — money story now runs on the best model with interactive Plotly dashboard saved!")


Generating calibrated predictions over full history...
  Training Student-t (Adam) for the battery...

Running per-regime battery (naive vs smart)...

  PART 16 — PER-REGIME BATTERY ON THE CALIBRATED STUDENT-T
  Regime      Naive €/day  Smart €/day    Extra   Days  Skipped
  --------------------------------------------------------------------
  Normal            41.04        47.52     6.47   1280      233
  Elevated          54.03        76.32    22.29    800      178
  Crisis            87.68       128.66    40.98    224       52

  ANNUAL FLEET VALUE (Germany 10 GWh by 2030):
    Normal    : +€ 6.47/MWh/day → €  23.6M/yr extra
    Elevated  : +€22.29/MWh/day → €  81.4M/yr extra
    Crisis    : +€40.98/MWh/day → € 149.6M/yr extra

Generating interactive Part 16 battery Plotly dashboard...
  ✓ Saved interactive Part 16 dashboard to part16_battery_calibrated.html



✓ Task 3 complete — money story now runs on the best model with interactive Plotly dashboard saved!


## Mathematical Formulation of Part 17: Variance-Stabilizing Transform Study

Electricity prices are heavy-tailed and can be negative, so they are not modelled directly. Part 17 compares **variance-stabilizing transformations (VSTs)** across all model architectures to determine which scaling yields the best accuracy.

---

### 1. Candidate Transforms

Let $p$ be the raw price. We compare:

- **Signed-log then standardize:** $z = \dfrac{\operatorname{sign}(p)\log(1+|p|) - \mu_a}{\sigma_a}$
- **Standardize then asinh:** $z = \operatorname{asinh}\!\left(\dfrac{p - a}{b}\right) = \log\!\left(y + \sqrt{y^2+1}\right),\quad y=\tfrac{p-a}{b}$
- **Standardize (robust) then asinh:** same as above with $a = \text{median}(p)$ and $b = 1.4826\cdot\text{MAD}(p)$.

The inverse hyperbolic sine is the field-standard VST (Uniejewski, Weron & Ziel, 2018): it behaves like a logarithm for large magnitudes but remains defined for negative and zero prices.

---

### 2. Fair Evaluation Protocol

Every transform $T$ is passed through the **same** model on the **same** data. Predictions are made in transformed space and mapped back with the exact inverse $T^{-1}$:

$$\hat{p} = T^{-1}(\hat{z}), \qquad \text{MAE} = \frac{1}{N}\sum_i |p_i - \hat{p}_i|$$

Because each $T$ is invertible and (for the VSTs) monotone, conformal coverage is preserved under the mapping. Comparing MAE and RMSE across the transform × model grid identifies the scaling that most improves forecast accuracy.

In [37]:
"""
═══════════════════════════════════════════════════════════════════════════
 PART 17 — TRANSFORM × MODEL COMPARISON (PLOTLY)
═══════════════════════════════════════════════════════════════════════════
 Evaluates variance-stabilizing price transformations across ALL model 
 architectures (DDNN, EvDNN, VI-DDNN, and Student-t). Renders interactive 
 Plotly dashboards and opens them automatically.
═══════════════════════════════════════════════════════════════════════════
"""


pio.renderers.default = "iframe"


# ── FORCE JUPYTER NATIVE RENDERER ─────────────────────────────────────────────
# Use "notebook" for classic Jupyter Notebook, or "jupyterlab" if you use JupyterLab

DOCS_DIR = DATA_DIR


# ════════════════════════════════════════════════════════════════════════════
# Student-t head with Adam optimizer — FULL backprop
# ════════════════════════════════════════════════════════════════════════════
class StudentTDNN_Adam:
    """Student-t output network trained with Adam (full backprop, all layers)."""
    def __init__(self, input_dim, hidden=[64,32], lr=0.01, epochs=250):
        self.lr, self.epochs = lr, epochs
        h1, h2 = hidden
        self.P = {
            'W1': np.random.randn(input_dim,h1)*np.sqrt(2/input_dim), 'b1': np.zeros(h1),
            'W2': np.random.randn(h1,h2)*np.sqrt(2/h1), 'b2': np.zeros(h2),
            'Wm': np.random.randn(h2,1)*0.01, 'bm': np.zeros(1),
            'Ws': np.random.randn(h2,1)*0.01, 'bs': np.zeros(1),
            'Wn': np.random.randn(h2,1)*0.01, 'bn': np.zeros(1),
        }
        self.m = {k: np.zeros_like(v) for k,v in self.P.items()}
        self.v = {k: np.zeros_like(v) for k,v in self.P.items()}
        self.t = 0
    def _relu(self,x): return np.maximum(0,x)
    def _fwd(self,X):
        z1=X@self.P['W1']+self.P['b1']; h1=self._relu(z1)
        z2=h1@self.P['W2']+self.P['b2']; h2=self._relu(z2)
        mu=(h2@self.P['Wm']+self.P['bm']).flatten()
        lsig=np.clip((h2@self.P['Ws']+self.P['bs']).flatten(),-4,4)
        lnu=np.clip((h2@self.P['Wn']+self.P['bn']).flatten(),0.3,3.5)
        return mu, np.exp(lsig)+1e-3, np.exp(lnu)+1.0, (z1,h1,z2,h2)
    def _adam(self,g):
        self.t+=1; b1,b2,e=0.9,0.999,1e-8
        for k in self.P:
            gr=g.get(k,0)
            self.m[k]=b1*self.m[k]+(1-b1)*gr
            self.v[k]=b2*self.v[k]+(1-b2)*(gr**2)
            mh=self.m[k]/(1-b1**self.t); vh=self.v[k]/(1-b2**self.t)
            self.P[k]-=self.lr*mh/(np.sqrt(vh)+e)
    def fit(self,X,y):
        n=len(y)
        for _ in range(self.epochs):
            mu,sig,nu,(z1,h1,z2,h2)=self._fwd(X)
            z=(y-mu)/sig; w=(nu+1)/(nu+z**2)
            dmu=-(w*z/sig)/n; dls=(1-w*z**2)/n
            g={}
            g['Wm']=h2.T@dmu.reshape(-1,1); g['bm']=np.array([dmu.sum()])
            g['Ws']=h2.T@dls.reshape(-1,1); g['bs']=np.array([dls.sum()])
            g['Wn']=np.zeros_like(self.P['Wn']); g['bn']=np.zeros_like(self.P['bn'])
            dh2=(dmu.reshape(-1,1)@self.P['Wm'].T+dls.reshape(-1,1)@self.P['Ws'].T)*(z2>0)
            g['W2']=h1.T@dh2; g['b2']=dh2.sum(axis=0)
            dh1=(dh2@self.P['W2'].T)*(z1>0)
            g['W1']=X.T@dh1; g['b1']=dh1.sum(0)
            self._adam(g)
        return self
    def predict(self,X):
        mu,sig,nu,_=self._fwd(X); return mu,sig,nu


# ════════════════════════════════════════════════════════════════════════════
# The four transforms
# ════════════════════════════════════════════════════════════════════════════
class TransformSignedLogThenStd:
    name = "B. signed-log then std"
    def _slog(self, p): return np.sign(p) * np.log1p(np.abs(p))
    def _islog(self, z): return np.sign(z) * np.expm1(np.abs(z))
    def fit(self, price):
        a = self._slog(price); self.mu = a.mean(); self.sd = a.std()
    def forward(self, price): return (self._slog(price) - self.mu) / self.sd
    def inverse(self, z): return self._islog(z * self.sd + self.mu)

class TransformStdThenAsinh:
    name = "C. std then asinh"
    def fit(self, price): self.a = price.mean(); self.b = price.std()
    def forward(self, price): return np.arcsinh((price - self.a) / self.b)
    def inverse(self, z): return np.sinh(z) * self.b + self.a

class TransformStdThenAsinhRobust:
    name = "D. std(robust) then asinh"
    def fit(self, price):
        self.a = np.median(price)
        mad = np.median(np.abs(price - self.a))
        self.b = mad * 1.4826 + 1e-6
    def forward(self, price): return np.arcsinh((price - self.a) / self.b)
    def inverse(self, z): return np.sinh(z) * self.b + self.a


# ════════════════════════════════════════════════════════════════════════════
# Evaluate one (transform, model) pair
# ════════════════════════════════════════════════════════════════════════════
def evaluate_transform_model(T, model_name, model_fn, train, val, test, features):
    T.fit(train['price'].values)
    z_tr  = T.forward(train['price'].values)
    z_val = T.forward(val['price'].values)
    y_te  = test['price'].values

    sX = StandardScaler()
    X_tr = sX.fit_transform(train[features].values)
    X_val= sX.transform(val[features].values)
    X_te = sX.transform(test[features].values)

    sz = StandardScaler()
    z_tr_s  = sz.fit_transform(z_tr.reshape(-1,1)).flatten()
    z_val_s = sz.transform(z_val.reshape(-1,1)).flatten()

    model = model_fn(X_tr.shape[1])
    if model_name == "StudentT":
        model.fit(X_tr, z_tr_s)
    else:
        model.fit(X_tr, z_tr_s, X_val, z_val_s, verbose=False)

    def to_euros(z_scaled):
        z = sz.inverse_transform(z_scaled.reshape(-1,1)).flatten()
        return T.inverse(z), z

    if model_name == "StudentT":
        z_pred_s = model.predict(X_te)[0]
    else:
        z_pred_s, _ = model.predict(X_te)
    mu_eur, z_pred = to_euros(z_pred_s)
    mu_eur = np.clip(mu_eur, -600.0, 1000.0)

    if model_name == "StudentT":
        z_val_pred_s = model.predict(X_val)[0]
    else:
        z_val_pred_s, _ = model.predict(X_val)
    _, z_val_pred = to_euros(z_val_pred_s)
    q = np.quantile(np.abs(z_val - z_val_pred), 0.90)
    lo_eur = T.inverse(z_pred - q); hi_eur = T.inverse(z_pred + q)
    lo2 = np.clip(np.minimum(lo_eur, hi_eur), -600.0, 1000.0)
    hi2 = np.clip(np.maximum(lo_eur, hi_eur), -600.0, 1000.0)

    return {"transform": T.name, "model": model_name,
            "MAE": float(np.mean(np.abs(y_te - mu_eur))),
            "RMSE": float(np.sqrt(np.mean((y_te - mu_eur)**2))),
            "PICP_90": float(np.mean((y_te >= lo2) & (y_te <= hi2))),
            "MPIW_med": float(np.median(hi2 - lo2))}


def plot_multi_model_transforms(results, save_path=os.path.join(DOCS_DIR, "part17_all_models_transforms.html")):
    print("\nGenerating interactive multi-model transform comparison Plotly dashboard...")
    models = sorted(set(r['model'] for r in results))
    colors = {'DDNN':'#378ADD','EvDNN':'#D85A30','VI-DDNN':'#7F77DD','StudentT':'#1D9E75'}

    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=("<b>MAE (€/MWh) Across Transforms & Models</b>",
                        "<b>RMSE (€/MWh) Across Transforms & Models</b>"),
        horizontal_spacing=0.12
    )

    for mdl in models:
        m_res = [r for r in results if r['model'] == mdl]
        m_res = sorted(m_res, key=lambda r: r['transform'])
        fig.add_trace(
            go.Bar(x=[r['transform'] for r in m_res], y=[r['MAE'] for r in m_res],
                   name=mdl, marker_color=colors.get(mdl, '#888780'),
                   text=[f"€{v:.1f}" for v in [r['MAE'] for r in m_res]], textposition='outside',
                   hovertemplate="<b>%{x}</b><br>"+mdl+" MAE: €%{y:.2f}<extra></extra>"),
            row=1, col=1)
        fig.add_trace(
            go.Bar(x=[r['transform'] for r in m_res], y=[r['RMSE'] for r in m_res],
                   name=mdl, marker_color=colors.get(mdl, '#888780'), showlegend=False,
                   text=[f"€{v:.1f}" for v in [r['RMSE'] for r in m_res]], textposition='outside',
                   hovertemplate="<b>%{x}</b><br>"+mdl+" RMSE: €%{y:.2f}<extra></extra>"),
            row=1, col=2)

    fig.update_layout(
        title=dict(text="<b>Part 17 — Multi-Model Price Transform Head-to-Head Comparison</b>",
                   font=dict(size=16), x=0.5, xanchor="center"),
        paper_bgcolor="#f7f6f3", plot_bgcolor="#ffffff",
        barmode='group', height=650, width=1250,
        margin=dict(t=100, l=50, r=50, b=50),
        legend=dict(orientation="h", yanchor="bottom", y=1.12, xanchor="center", x=0.5))

    good_mae = [r['MAE'] for r in results if r['MAE'] < 500]
    good_rmse = [r['RMSE'] for r in results if r['RMSE'] < 500]
    mae_cap = (max(good_mae)*1.25) if good_mae else 100
    rmse_cap = (max(good_rmse)*1.25) if good_rmse else 100
    fig.update_yaxes(title_text="MAE (€/MWh)", range=[0, mae_cap], row=1, col=1)
    fig.update_yaxes(title_text="RMSE (€/MWh)", range=[0, rmse_cap], row=1, col=2)
    fig.update_xaxes(tickangle=-15)

    # Save backup HTML file
    fig.write_html(save_path)
    print(f"  ✓ Saved interactive dashboard copy to {save_path}")
    
    # Render natively inside the Jupyter cell
    fig.show()


if __name__ == "__main__":
    df = pd.read_pickle(os.path.join(DOCS_DIR, 'data_part2.pkl'))
    with open(os.path.join(DOCS_DIR, 'selected_part4.json')) as f: features = json.load(f)

    train = df[df['datetime'] <= '2024-04-30 23:00'].copy()
    val   = df[(df['datetime'] >= '2024-05-01') & (df['datetime'] <= '2025-04-30 23:00')].copy()
    test  = df[df['datetime'] >= '2025-05-01'].copy()

    transforms = [TransformSignedLogThenStd(),
                  TransformStdThenAsinh(), TransformStdThenAsinhRobust()]
    models_dict = {
        "DDNN":    lambda dim: DDNN(dim, hidden=[128,64,32], lr=0.002, epochs=120),
        "EvDNN":   lambda dim: EvDNN(dim, hidden=[128,64,32], lr=0.001, epochs=120, lam=0.02),
        "VI-DDNN": lambda dim: VIDDNN(dim, hidden=[128,64], lr=0.002, epochs=120, n_samples=15),
        "StudentT":lambda dim: StudentTDNN_Adam(dim, hidden=[64,32], lr=0.01, epochs=150),
    }

    results = []
    for T in transforms:
        for model_name, model_fn in models_dict.items():
            print(f"Evaluating Transform: {T.name} | Model: {model_name} ...")
            results.append(evaluate_transform_model(T, model_name, model_fn, train, val, test, features))

    print("\n" + "="*74)
    print("  PART 17 — TRANSFORM × MODEL COMPARISON (test May'25–Apr'26)")
    print("="*74)
    print(f"  {'Transform':<26}{'Model':<10}{'MAE':>9}{'RMSE':>9}{'PICP':>8}")
    print("  " + "-"*68)
    for r in sorted(results, key=lambda r:(r['transform'], r['MAE'])):
        print(f"  {r['transform']:<26}{r['model']:<10}{r['MAE']:>9.2f}{r['RMSE']:>9.2f}{r['PICP_90']:>8.1%}")
    print("="*74)
    best = min(results, key=lambda r:r['MAE'])
    print(f"\n  Best overall: {best['model']} with {best['transform']} (MAE €{best['MAE']:.2f})")

    np.save(os.path.join(DOCS_DIR, 'part17_all_models_results.npy'), {'results':results}, allow_pickle=True)
    plot_multi_model_transforms(results)
    print("\n✓ Part 17 complete")

Evaluating Transform: B. signed-log then std | Model: DDNN ...
Evaluating Transform: B. signed-log then std | Model: EvDNN ...
Evaluating Transform: B. signed-log then std | Model: VI-DDNN ...
Evaluating Transform: B. signed-log then std | Model: StudentT ...
Evaluating Transform: C. std then asinh | Model: DDNN ...
Evaluating Transform: C. std then asinh | Model: EvDNN ...
Evaluating Transform: C. std then asinh | Model: VI-DDNN ...
Evaluating Transform: C. std then asinh | Model: StudentT ...
Evaluating Transform: D. std(robust) then asinh | Model: DDNN ...
Evaluating Transform: D. std(robust) then asinh | Model: EvDNN ...
Evaluating Transform: D. std(robust) then asinh | Model: VI-DDNN ...
Evaluating Transform: D. std(robust) then asinh | Model: StudentT ...

  PART 17 — TRANSFORM × MODEL COMPARISON (test May'25–Apr'26)
  Transform                 Model           MAE     RMSE    PICP
  --------------------------------------------------------------------
  B. signed-log then std    S


✓ Part 17 complete


### Part 17 (continued) — Transform *Order* Study

The cell below complements the multi-model comparison above by isolating the **order** of operations: applying signed-log *then* standardizing versus standardizing *then* applying asinh. This confirms that the field-standard order (standardize → asinh) yields the lowest error on our data.

In [38]:
"""
═══════════════════════════════════════════════════════════════════════════
 PART 17 — TRANSFORM COMPARISON (signed-log vs asinh, and the ORDER) (PLOTLY)
═══════════════════════════════════════════════════════════════════════════
 Runs a fair head-to-head of four ways to transform the price target, all
 through the SAME model on the SAME data, and reports which gives lower error.
 Renders interactive Plotly dashboards saved to the working directory.
═══════════════════════════════════════════════════════════════════════════
"""



pio.renderers.default = "iframe"
DOCS_DIR = DATA_DIR


# ════════════════════════════════════════════════════════════════════════════
# The transforms — each is a pair: forward(price)->z  and  inverse(z)->price
# ════════════════════════════════════════════════════════════════════════════

class TransformNone:
    """A. Raw price, only standardized. Baseline (no VST)."""
    name = "A. None (raw+std)"
    def fit(self, price):
        self.mu = price.mean(); self.sd = price.std()
    def forward(self, price):
        return (price - self.mu) / self.sd
    def inverse(self, z):
        return z * self.sd + self.mu


class TransformSignedLogThenStd:
    """B. YOUR order: signed-log first, then standardize the result."""
    name = "B. signed-log then std"
    def _slog(self, p): return np.sign(p) * np.log1p(np.abs(p))
    def _islog(self, z): return np.sign(z) * np.expm1(np.abs(z))
    def fit(self, price):
        a = self._slog(price)
        self.mu = a.mean(); self.sd = a.std()
    def forward(self, price):
        return (self._slog(price) - self.mu) / self.sd
    def inverse(self, z):
        return self._islog(z * self.sd + self.mu)


class TransformStdThenAsinh:
    """C. FIELD order: standardize first (mean/std), then asinh."""
    name = "C. std then asinh"
    def fit(self, price):
        self.a = price.mean(); self.b = price.std()
    def forward(self, price):
        return np.arcsinh((price - self.a) / self.b)
    def inverse(self, z):
        return np.sinh(z) * self.b + self.a


class TransformStdThenAsinhRobust:
    """D. FIELD order with ROBUST scaling: median + MAD, then asinh."""
    name = "D. std(robust) then asinh"
    def fit(self, price):
        self.a = np.median(price)
        mad = np.median(np.abs(price - self.a))
        self.b = mad * 1.4826 + 1e-6       # 1.4826 makes MAD comparable to std
    def forward(self, price):
        return np.arcsinh((price - self.a) / self.b)
    def inverse(self, z):
        return np.sinh(z) * self.b + self.a


# ════════════════════════════════════════════════════════════════════════════
# Run one transform through the model, return euro-space metrics
# ════════════════════════════════════════════════════════════════════════════
def evaluate_transform(T, train, val, test, features):
    """Train a DDNN on the transformed target, predict, invert, score in euros."""
    T.fit(train['price'].values)
    z_tr  = T.forward(train['price'].values)
    z_val = T.forward(val['price'].values)
    y_te  = test['price'].values

    sX = StandardScaler()
    X_tr = sX.fit_transform(train[features].values)
    X_val= sX.transform(val[features].values)
    X_te = sX.transform(test[features].values)

    sz = StandardScaler()
    z_tr_s  = sz.fit_transform(z_tr.reshape(-1,1)).flatten()
    z_val_s = sz.transform(z_val.reshape(-1,1)).flatten()

    model = DDNN(len(features), hidden=[128,64,32], lr=0.002, epochs=300)
    model.fit(X_tr, z_tr_s, X_val, z_val_s)

    mu_s, sig_s = model.predict(X_te)
    z_pred = sz.inverse_transform(mu_s.reshape(-1,1)).flatten()
    mu_eur = T.inverse(z_pred)

    z_val_pred = sz.inverse_transform(model.predict(X_val)[0].reshape(-1,1)).flatten()
    q = np.quantile(np.abs(z_val - z_val_pred), 0.90)
    lo_eur = T.inverse(z_pred - q)
    hi_eur = T.inverse(z_pred + q)
    lo2 = np.minimum(lo_eur, hi_eur); hi2 = np.maximum(lo_eur, hi_eur)

    mae  = float(np.mean(np.abs(y_te - mu_eur)))
    rmse = float(np.sqrt(np.mean((y_te - mu_eur)**2)))
    picp = float(np.mean((y_te >= lo2) & (y_te <= hi2)))
    mpiw_med = float(np.median(hi2 - lo2))
    return {"name": T.name, "MAE": mae, "RMSE": rmse, "PICP_90": picp, "MPIW_med": mpiw_med}


# ════════════════════════════════════════════════════════════════════════════
# PLOTLY INTERACTIVE DASHBOARD
# ════════════════════════════════════════════════════════════════════════════
def plot_comparison_plotly(results, save_path=os.path.join(DOCS_DIR, "part17_transform_comparison.html")):
    print("\nGenerating interactive Part 17 transform comparison Plotly dashboard...")
    names = [r['name'] for r in results]
    cols = ['#888780', '#1D9E75', '#0C447C', '#BA7517']

    fig = make_subplots(
        rows=1, cols=3,
        subplot_titles=(
            "<b>MAE (€/MWh) — lower better</b>",
            "<b>RMSE (€/MWh) — lower better</b>",
            "<b>PICP 90% — target 90%</b>"
        ),
        horizontal_spacing=0.10
    )

    # 1. MAE (Values displayed on top of bars)
    mae_vals = [r['MAE'] for r in results]
    fig.add_trace(go.Bar(
        x=[n.split('.')[0] for n in names], y=mae_vals,
        marker_color=cols, 
        text=[f"€{v:.1f}" for v in mae_vals], 
        textposition='outside',
        textfont=dict(size=12),
        showlegend=False, hovertemplate="<b>%{x}</b><br>MAE: €%{y:.2f}<extra></extra>"
    ), row=1, col=1)

    # 2. RMSE (Values displayed on top of bars)
    rmse_vals = [r['RMSE'] for r in results]
    fig.add_trace(go.Bar(
        x=[n.split('.')[0] for n in names], y=rmse_vals,
        marker_color=cols, 
        text=[f"€{v:.1f}" for v in rmse_vals], 
        textposition='outside',
        textfont=dict(size=12),
        showlegend=False, hovertemplate="<b>%{x}</b><br>RMSE: €%{y:.2f}<extra></extra>"
    ), row=1, col=2)

    # 3. PICP_90 (Values displayed on top of bars)
    picp_vals = [r['PICP_90'] * 100 for r in results]
    fig.add_trace(go.Bar(
        x=[n.split('.')[0] for n in names], y=picp_vals,
        marker_color=cols, 
        text=[f"{v:.1f}%" for v in picp_vals], 
        textposition='outside',
        textfont=dict(size=12),
        showlegend=False, hovertemplate="<b>%{x}</b><br>PICP: %{y:.1f}%<extra></extra>"
    ), row=1, col=3)

    fig.add_shape(
        type="line", x0=-0.5, x1=len(names)-0.5, y0=90, y1=90,
        line=dict(color="#D85A30", dash="dash", width=2),
        row=1, col=3
    )

    legend_text = "    ".join([f"<b>{n.split('.')[0]}</b>: {n.split('.', 1)[1].strip()}" for n in names])

    fig.update_layout(
        title=dict(
            text=f"<b>Part 17 — Price Transform Head-to-Head Comparison</b><br><sup>{legend_text}</sup>",
            font=dict(size=14), x=0.5, xanchor="center"
        ),
        paper_bgcolor="#f7f6f3", plot_bgcolor="#ffffff",
        height=600, width=1400,
        margin=dict(t=120, l=60, r=60, b=60)
    )

    fig.update_yaxes(title_text="MAE (€/MWh)", row=1, col=1, range=[0, max(mae_vals)*1.2])
    fig.update_yaxes(title_text="RMSE (€/MWh)", row=1, col=2, range=[0, max(rmse_vals)*1.2])
    fig.update_yaxes(title_text="PICP (%)", row=1, col=3, range=[0, 115])
    fig.update_xaxes(tickangle=-15)

    fig.write_html(save_path)
    print(f"  ✓ Saved interactive transform comparison dashboard to {save_path}")
    
    try:
        fig.show()
    except Exception:
        pass


if __name__ == "__main__":
    df = pd.read_pickle(os.path.join(DOCS_DIR, 'data_part2.pkl'))
    with open(os.path.join(DOCS_DIR, 'selected_part4.json')) as f: features = json.load(f)

    train = df[df['datetime'] <= '2024-04-30 23:00'].copy()
    val   = df[(df['datetime'] >= '2024-05-01') & (df['datetime'] <= '2025-04-30 23:00')].copy()
    test  = df[df['datetime'] >= '2025-05-01'].copy()

    transforms = [TransformNone(), TransformSignedLogThenStd(),
                  TransformStdThenAsinh(), TransformStdThenAsinhRobust()]

    results = []
    for T in transforms:
        print(f"Evaluating {T.name} ...")
        results.append(evaluate_transform(T, train, val, test, features))

    print("\n" + "="*74)
    print("  PART 17 — TRANSFORM COMPARISON (test May 2025 – Apr 2026)")
    print("="*74)
    print(f"  {'Transform':<28}{'MAE':>9}{'RMSE':>9}{'PICP_90':>10}{'MPIW_med':>11}")
    print("  " + "-"*68)
    best_mae = min(r['MAE'] for r in results)
    for r in results:
        star = "  <- best MAE" if r['MAE'] == best_mae else ""
        print(f"  {r['name']:<28}{r['MAE']:>9.2f}{r['RMSE']:>9.2f}"
              f"{r['PICP_90']:>9.1%}{r['MPIW_med']:>11.2f}{star}")
    print("="*74)

    winner = min(results, key=lambda r: r['MAE'])
    print(f"\n  Winner on accuracy: {winner['name']} (MAE €{winner['MAE']:.2f})")
    print("  Note: run 3-5 times and average before drawing a firm conclusion.")

    np.save(os.path.join(DOCS_DIR, 'part17_results.npy'), {'results': results}, allow_pickle=True)
    plot_comparison_plotly(results, os.path.join(DOCS_DIR, 'part17_transform_comparison.html'))
    print("\n✓ Part 17 complete with interactive Plotly dashboard saved!")

Evaluating A. None (raw+std) ...

done. Final NLL=-1.5683.. 
Evaluating B. signed-log then std ...

done. Final NLL=-1.7049.. 
Evaluating C. std then asinh ...

done. Final NLL=-1.4573.. 
Evaluating D. std(robust) then asinh ...

done. Final NLL=-1.4496.. 

  PART 17 — TRANSFORM COMPARISON (test May 2025 – Apr 2026)
  Transform                         MAE     RMSE   PICP_90   MPIW_med
  --------------------------------------------------------------------
  A. None (raw+std)               16.07    22.70    88.4%      64.24
  B. signed-log then std          36.23   823.03    88.1%     197.66
  C. std then asinh               15.23    22.49    88.7%      57.44
  D. std(robust) then asinh       14.45    21.71    89.4%      54.03  <- best MAE

  Winner on accuracy: D. std(robust) then asinh (MAE €14.45)
  Note: run 3-5 times and average before drawing a firm conclusion.

Generating interactive Part 17 transform comparison Plotly dashboard...
  ✓ Saved interactive transform comparison dashbo


✓ Part 17 complete with interactive Plotly dashboard saved!


## Part 18: Unified Model Comparison

Part 18 consolidates every model trained across the notebook into a single ranked table and interactive dashboard. Because all parts share the **same test window** (May 2025 – April 2026) and the same metrics, the numbers are directly comparable.

The comparison spans:

- **Classical / baseline models** (Part 8): DDNN, EvDNN, VI-DDNN, VI+CP, BSSM
- **Advanced variants** (Part 14): Gaussian + Log-CP, Student-t + Log-CP, Rolling window
- **Calibrated Student-t** (Parts 15 / 15b): scale-aware and multi-level conformal

Ranking by MAE, PICP, MPIW, and MAACE reveals the full progression — from the classical baselines down to the calibrated Student-t, which achieves the best accuracy **and** the best calibration simultaneously.

In [44]:
"""
═══════════════════════════════════════════════════════════════════════════
 PART 18 — UNIFIED COMPARISON (all models in ONE ranked table) (PLOTLY)
═══════════════════════════════════════════════════════════════════════════
 Pulls together every model you trained, across parts, into a single table
 and an interactive Plotly dashboard.
 
 Reads (whichever exist):
   • results_part8.npy    → classical models (Selected & All Feature sets)
   • part14_results.npy   → Gaussian, Gaussian+LogCP, StudentT+LogCP, Rolling
   • part15_results.npy   → Student-t before/after scale-aware conformal
   • part15b_results.npy  → Student-t with multi-level conformal (final)
═══════════════════════════════════════════════════════════════════════════
"""


pio.renderers.default = "iframe"
DOCS_DIR = DATA_DIR



def load_if_exists(filename):
    """Checks user documents directory first, then local directory."""
    paths = [os.path.join(DATA_DIR, f"{filename}"), filename]
    for p in paths:
        if os.path.exists(p):
            return np.load(p, allow_pickle=True).item()
    return None


def collect_rows():
    """Gather (source, name, MAE, PICP, MPIW_med, MAACE) from every saved file."""
    rows = []

    # ── Part 8/9: classical models (supports both old and new Part 8 structures) ───
    d8 = load_if_exists('results_part8.npy')
    if d8 is not None:
        if 'results_selected' in d8:
            for name, r in d8['results_selected'].items():
                rows.append(('Part 8 (Selected)', name, r.get('MAE'), r.get('PICP_90'),
                             r.get('MPIW_90'), r.get('MAACE')))
        if 'results_all' in d8:
            for name, r in d8['results_all'].items():
                rows.append(('Part 8 (All)', name, r.get('MAE'), r.get('PICP_90'),
                             r.get('MPIW_90'), r.get('MAACE')))
        if 'results' in d8:
            for name, r in d8['results'].items():
                rows.append(('Part 8', name, r.get('MAE'), r.get('PICP_90'),
                             r.get('MPIW_90'), r.get('MAACE')))

    # ── Part 14: advanced variants ───────────────────────────────────────────
    d14 = load_if_exists('part14_results.npy')
    if d14 is not None:
        for name, m in d14['table'].items():
            rows.append(('Part 14', name, m.get('MAE'), m.get('PICP_90'),
                         m.get('MPIW_90_med', m.get('MPIW_90')), m.get('MAACE')))

    # ── Part 15: Student-t before/after scale-aware conformal ────────────────
    d15 = load_if_exists('part15_results.npy')
    if d15 is not None:
        for key, label in [('before', 'StudentT plain-CP (P15)'),
                           ('after', 'StudentT scale-aware (P15)')]:
            if key in d15:
                m = d15[key]
                rows.append(('Part 15', label, m.get('MAE'), m.get('PICP_90'),
                             m.get('MPIW_90_med', m.get('MPIW_90')), m.get('MAACE')))

    # ── Part 15b: multi-level calibrated (the final best) ────────────────────
    d15b = load_if_exists('part15b_results.npy')
    if d15b is not None:
        m = d15b['metrics']
        rows.append(('Part 15b', 'StudentT multi-level (FINAL)', m.get('MAE'),
                     m.get('PICP_90'), m.get('MPIW_90_med', m.get('MPIW_90')),
                     d15b.get('maace')))

    return rows


def print_table(rows):
    print("\n" + "="*95)
    print("  UNIFIED MODEL COMPARISON  —  same test window (May 2025 – Apr 2026)")
    print("="*95)
    print(f"  {'Source':<20}{'Model':<30}{'MAE':>8}{'PICP_90':>10}{'MPIW':>9}{'MAACE':>9}")
    print("  " + "-"*91)
    
    rows_sorted = sorted(rows, key=lambda r: (r[2] is None, r[2] if r[2] is not None else 1e9))
    best_mae = min((r[2] for r in rows if r[2] is not None), default=None)
    
    for src, name, mae, picp, mpiw, maace in rows_sorted:
        mae_s   = f"{mae:.2f}"   if mae   is not None else "  —"
        picp_s  = f"{picp:.1%}"  if picp  is not None else "  —"
        mpiw_s  = f"{mpiw:.1f}"  if mpiw  is not None else "  —"
        maace_s = f"{maace:.2f}%" if maace is not None else "  —"
        star = "  <= best MAE" if (mae is not None and mae == best_mae) else ""
        print(f"  {src:<20}{name:<30}{mae_s:>8}{picp_s:>10}{mpiw_s:>9}{maace_s:>9}{star}")
    
    print("="*95)
    print("  Lower MAE = more accurate | PICP closest to 90% = best coverage")
    print("  Lower MPIW = sharper        | Lower MAACE = better calibrated")


def plot_unified_plotly(rows, save_path=os.path.join(DOCS_DIR, "part18_unified_comparison.html")):
    print("\nGenerating interactive Plotly unified comparison dashboard...")
    rows = [r for r in rows if r[2] is not None and r[5] is not None]
    rows_sorted = sorted(rows, key=lambda r: r[2])  # Ascending MAE
    
    names = [r[1] for r in rows_sorted]
    sources = [r[0] for r in rows_sorted]
    mae = [r[2] for r in rows_sorted]
    maace = [r[5] for r in rows_sorted]
    picp = [(r[3]*100 if r[3] is not None else np.nan) for r in rows_sorted]

    srccol = {
        'Part 8': '#888780', 
        'Part 8 (Selected)': '#7F77DD', 
        'Part 8 (All)': '#BA7517', 
        'Part 14': '#378ADD', 
        'Part 15': '#1D9E75', 
        'Part 15b': '#0C447C'
    }
    cols = [srccol.get(s, '#888780') for s in sources]

    fig = make_subplots(
        rows=1, cols=3,
        subplot_titles=(
            "<b>MAE (€/MWh) — lower better</b>",
            "<b>MAACE (%) — lower = better calibrated</b>",
            "<b>PICP 90% — target 90%</b>"
        ),
        horizontal_spacing=0.10,
        shared_yaxes=True
    )

    # 1. MAE
    fig.add_trace(go.Bar(
        y=names, x=mae, orientation='h',
        marker_color=cols, text=[f"{v:.1f}" for v in mae], textposition='inside',
        showlegend=False, hovertemplate="<b>%{y}</b><br>MAE: €%{x:.2f}<extra></extra>"
    ), row=1, col=1)

    # 2. MAACE
    fig.add_trace(go.Bar(
        y=names, x=maace, orientation='h',
        marker_color=cols, text=[f"{v:.1f}%" for v in maace], textposition='inside',
        showlegend=False, hovertemplate="<b>%{y}</b><br>MAACE: %{x:.2f}%<extra></extra>"
    ), row=1, col=2)

    # 3. PICP 90%
    fig.add_trace(go.Bar(
        y=names, x=picp, orientation='h',
        marker_color=cols, text=[f"{v:.0f}%" for v in picp], textposition='inside',
        showlegend=False, hovertemplate="<b>%{y}</b><br>PICP: %{x:.1f}%<extra></extra>"
    ), row=1, col=3)

    fig.add_shape(
        type="line", x0=90, x1=90, y0=-0.5, y1=len(names)-0.5,
        line=dict(color="#D85A30", dash="dash", width=2),
        row=1, col=3
    )

    fig.update_layout(
        title=dict(
            text="<b>Part 18 — All Models Compared on the Same Test Window</b><br><sup>from classical (Part 8) to the calibrated Student-t (Part 15b)</sup>",
            font=dict(size=15), x=0.5, xanchor="center"
        ),
        paper_bgcolor="#f7f6f3", plot_bgcolor="#ffffff",
        height=max(600, len(names) * 35), width=1500,
        yaxis=dict(autorange="reversed")
    )

    fig.update_xaxes(title_text="MAE (€/MWh)", row=1, col=1)
    fig.update_xaxes(title_text="MAACE Error (%)", row=1, col=2)
    fig.update_xaxes(title_text="PICP Coverage (%)", row=1, col=3)

    fig.write_html(save_path)
    print(f"  ✓ Saved interactive unified comparison dashboard to {save_path}")
    fig.show(renderer="iframe")


if __name__ == "__main__":
    rows = collect_rows()
    if not rows:
        print("No result files found. Run parts 8/9, 14, and 15 first, then re-run this.")
    else:
        found = sorted(set(r[0] for r in rows))
        print(f"\nFound results from sources: {', '.join(found)}")
        print_table(rows)
        plot_unified_plotly(rows)
        print("\n✓ Part 18 complete — unified ranked table + interactive Plotly dashboard saved!")


Found results from sources: Part 14, Part 15, Part 15b, Part 8 (All), Part 8 (Selected)

  UNIFIED MODEL COMPARISON  —  same test window (May 2025 – Apr 2026)
  Source              Model                              MAE   PICP_90     MPIW    MAACE
  -------------------------------------------------------------------------------------------
  Part 8 (All)        EvDNN                             8.56    100.0%    391.5   43.52%  <= best MAE
  Part 8 (All)        DDNN                             14.18     75.5%     80.2    7.31%
  Part 15             StudentT plain-CP (P15)          14.51     87.7%    194.6   25.01%
  Part 15             StudentT scale-aware (P15)       14.51     93.0%     74.5   14.17%
  Part 14             StudentT+LogCP                   14.66     88.4%    101.3   18.44%
  Part 15b            StudentT multi-level (FINAL)     14.93     92.2%     71.7    0.88%
  Part 8 (Selected)   DDNN                             15.81     78.9%     61.1    5.46%
  Part 8 (All)       


✓ Part 18 complete — unified ranked table + interactive Plotly dashboard saved!


### Mathematical summary of the final model

**Transform (standardize → asinh):**

$$ z = \operatorname{asinh}\!\left(\frac{p - a}{b}\right), \qquad p = b\,\sinh(z) + a $$

**Distribution head (Student-t):** the network outputs $(\mu, \sigma, \nu)$ trained by minimizing the Student-t NLL, giving fat-tailed, spike-robust predictions.

**Calibration (multi-level normalized conformal):** a separate normalized quantile $\hat q(\ell)$ per confidence level $\ell$ yields a near-diagonal calibration curve.

The cell below trains this combined model on **both** feature sets, prints a comparison table (like Part 9), identifies the winning feature set by MAE, plots the winner's forecast, and runs the per-regime battery (like Parts 12/16) on the winning model — all with interactive Plotly dashboards.

In [56]:
# ═══════════════════════════════════════════════════════════════════════════
#  PART 19 — FINAL MODEL (Student-t + asinh + multi-level conformal)
#  • BOTH feature sets (Selected & All)   • forecast plots use DATES + WHOLE test set
#  • ALL 4 MODELS reported on MAE / RMSE / PICP / MPIW(med) / MAACE
#  • Multi-Model Battery Valuation across the 4 feature-based models (asinh scaling)
# ═══════════════════════════════════════════════════════════════════════════

# ---- BEST SCALING: standardize → asinh --------------------------------------


# ═══════════════════════════════════════════════════════════════════════════
#  PART 19 — FINAL MODEL (Student-t + asinh + multi-level conformal)
#  • BOTH feature sets (Selected & All)   • forecast plots use DATES + WHOLE test set
#  • ALL 4 MODELS reported on MAE / RMSE / PICP / MPIW(med) / MAACE
#  • Multi-Model Battery Valuation across the 4 feature-based models (asinh scaling)
# ═══════════════════════════════════════════════════════════════════════════

# ---- BEST SCALING: standardize → asinh -------------------------------------\
from IPython.display import display
class AsinhTransform:
    def fit(self, p): self.a=float(np.mean(p)); self.b=float(np.std(p)+1e-8)
    def forward(self, p): return np.arcsinh((p-self.a)/self.b)
    def inverse(self, z): return np.sinh(z)*self.b+self.a

# ---- BEST DISTRIBUTION: Student-t head with Adam ----------------------------
class StudentTAdam:
    def __init__(self, d, hidden=(64,32), lr=0.01, epochs=250):
        h1,h2=hidden; self.lr,self.epochs=lr,epochs
        self.P={'W1':np.random.randn(d,h1)*np.sqrt(2/d),'b1':np.zeros(h1),
                'W2':np.random.randn(h1,h2)*np.sqrt(2/h1),'b2':np.zeros(h2),
                'Wm':np.random.randn(h2,1)*0.01,'bm':np.zeros(1),
                'Ws':np.random.randn(h2,1)*0.01,'bs':np.zeros(1),
                'Wn':np.random.randn(h2,1)*0.01,'bn':np.zeros(1)}
        self.m={k:np.zeros_like(v) for k,v in self.P.items()}
        self.v={k:np.zeros_like(v) for k,v in self.P.items()}; self.t=0
    def _relu(self,x): return np.maximum(0,x)
    def _fwd(self,X):
        z1=X@self.P['W1']+self.P['b1']; h1=self._relu(z1)
        z2=h1@self.P['W2']+self.P['b2']; h2=self._relu(z2)
        mu=(h2@self.P['Wm']+self.P['bm']).flatten()
        lsig=np.clip((h2@self.P['Ws']+self.P['bs']).flatten(),-4,4)
        lnu=np.clip((h2@self.P['Wn']+self.P['bn']).flatten(),0.3,3.5)
        return mu,np.exp(lsig)+1e-3,np.exp(lnu)+1.0,(z1,h1,z2,h2)
    def _adam(self,g):
        self.t+=1; b1,b2,e=0.9,0.999,1e-8
        for k in self.P:
            gr=g.get(k,0)
            self.m[k]=b1*self.m[k]+(1-b1)*gr; self.v[k]=b2*self.v[k]+(1-b2)*gr**2
            mh=self.m[k]/(1-b1**self.t); vh=self.v[k]/(1-b2**self.t)
            self.P[k]-=self.lr*mh/(np.sqrt(vh)+e)
    def fit(self,X,y):
        n=len(y)
        for _ in range(self.epochs):
            mu,sig,nu,(z1,h1,z2,h2)=self._fwd(X)
            z=(y-mu)/sig; w=(nu+1)/(nu+z**2)
            dmu=-(w*z/sig)/n; dls=(1-w*z**2)/n
            g={'Wm':h2.T@dmu.reshape(-1,1),'bm':np.array([dmu.sum()]),
               'Ws':h2.T@dls.reshape(-1,1),'bs':np.array([dls.sum()]),
               'Wn':np.zeros_like(self.P['Wn']),'bn':np.zeros_like(self.P['bn'])}
            dh2=(dmu.reshape(-1,1)@self.P['Wm'].T+dls.reshape(-1,1)@self.P['Ws'].T)*(z2>0)
            g['W2']=h1.T@dh2; g['b2']=dh2.sum(0)
            dh1=(dh2@self.P['W2'].T)*(z1>0)
            g['W1']=X.T@dh1; g['b1']=dh1.sum(0)
            self._adam(g)
        return self
    def predict(self,X):
        mu,sig,nu,_=self._fwd(X); return mu,sig,nu

# ---- BEST CALIBRATION: multi-level normalized conformal ---------------------
class MultiLevelConformal:
    def __init__(self): self.q={}
    def fit(self,z_true,z_pred,sigma):
        r=np.abs(z_true-z_pred)/np.maximum(sigma,1e-6)
        for lv in np.arange(0.1,1.0,0.1): self.q[round(lv,1)]=np.quantile(r,lv)
    def band(self,z_pred,sigma,level,T):
        half=self.q[round(level,1)]*np.maximum(sigma,1e-6)
        return T.inverse(z_pred-half), T.inverse(z_pred+half)

# ---- Load data + BOTH feature sets ------------------------------------------
DOCS_DIR = DATA_DIR
def _p(fname):
    cand = os.path.join(DOCS_DIR, fname)
    return cand if os.path.exists(cand) else fname
df = pd.read_pickle(_p("data_part2.pkl"))
with open(_p("selected_part4.json")) as f: selected_features = json.load(f)
with open(_p("features_part2.json"))  as f: all_features = json.load(f)
print(f"Selected feature set: {len(selected_features)} features")
print(f"All-candidate set:    {len(all_features)} features\n")

# Reuse the Optuna-tuned hyperparameters that Part 8 found (falls back to defaults).
DEFAULTS = {"DDNN":{"lr":0.005,"epochs":250,"batch_size":128},
            "EvDNN":{"lr":0.005,"epochs":250,"batch_size":128,"lam":0.02},
            "VI-DDNN":{"lr":0.002,"epochs":250,"batch_size":128,"prior_sigma":1.0}}
try:
    with open(_p("results_part8.json")) as f:
        loaded = json.load(f)
    BP = {m: {**DEFAULTS[m], **loaded.get(m, {})} for m in DEFAULTS}
    print(f"Loaded per-model Optuna hyperparameters from Part 8:\n  {BP}\n")
except FileNotFoundError:
    BP = DEFAULTS
    print("No results_part8.json found - using per-model defaults.\n")
STUDENTT_EPOCHS = 250

train = df[df["datetime"] <= "2024-04-30 23:00"].copy()
val   = df[(df["datetime"] >= "2024-05-01") & (df["datetime"] <= "2025-04-30 23:00")].copy()
test  = df[df["datetime"] >= "2025-05-01"].copy()
test_dates = test["datetime"].values
dfr = pd.read_pickle(_p("data_part7_regimes.pkl"))[["datetime","regime"]].drop_duplicates("datetime")
full = df.merge(dfr, on="datetime", how="left")
full["regime"] = full["regime"].fillna(0).astype(int)
full = full.sort_values("datetime").reset_index(drop=True)

# ═══════════════════════════════════════════════════════════════════════════
#  A) BEST MODEL on BOTH feature sets  (Student-t + asinh + multi-level CP)
# ═══════════════════════════════════════════════════════════════════════════
def run_best_model(features, tag):
    T = AsinhTransform(); T.fit(train["price"].values)
    sX = StandardScaler()
    X_tr = sX.fit_transform(train[features].values)
    X_val = sX.transform(val[features].values)
    X_te = sX.transform(test[features].values)
    z_tr = T.forward(train["price"].values); z_val = T.forward(val["price"].values)
    sz = StandardScaler(); z_tr_s = sz.fit_transform(z_tr.reshape(-1,1)).flatten()
    y_te = test["price"].values
    print(f"  Training best model on {tag} feature set...")
    model = StudentTAdam(len(features), hidden=(64,32), lr=0.01, epochs=STUDENTT_EPOCHS).fit(X_tr, z_tr_s)
    def pzs(X):
        mu_s, sig_s, nu = model.predict(X)
        return sz.inverse_transform(mu_s.reshape(-1,1)).flatten(), sig_s*sz.scale_[0]
    zt_val, sig_val = pzs(X_val); zt_te, sig_te = pzs(X_te)
    cal = MultiLevelConformal(); cal.fit(z_val, zt_val, sig_val)
    mu_eur = np.clip(T.inverse(zt_te), -600.0, 1000.0); lo90, hi90 = cal.band(zt_te, sig_te, 0.9, T)
    lo90 = np.clip(lo90, -600.0, 1000.0); hi90 = np.clip(hi90, -600.0, 1000.0)
    mae=float(np.mean(np.abs(y_te-mu_eur))); rmse=float(np.sqrt(np.mean((y_te-mu_eur)**2)))
    picp=float(np.mean((y_te>=np.minimum(lo90,hi90))&(y_te<=np.maximum(lo90,hi90))))
    mpiw=float(np.median(np.abs(hi90-lo90)))
    maace=0.0
    for lv in np.arange(0.1,1.0,0.1):
        lo,hi=cal.band(zt_te,sig_te,lv,T)
        maace+=abs(np.mean((y_te>=np.minimum(lo,hi))&(y_te<=np.maximum(lo,hi)))-lv)
    maace=maace/9*100
    return {"tag":tag,"MAE":mae,"RMSE":rmse,"PICP":picp,"MPIW":mpiw,"MAACE":maace,
            "mu":mu_eur,"lo":lo90,"hi":hi90,"y":y_te}

res_sel = run_best_model(selected_features, f"Selected ({len(selected_features)})")
res_all = run_best_model(all_features,      f"All ({len(all_features)})")

print("\n"+"="*72)
print("  FINAL MODEL — BOTH FEATURE SETS (Student-t + asinh + multi-level CP)")
print("="*72)
print(f"  {'Feature set':<18}{'MAE':>9}{'RMSE':>9}{'PICP':>8}{'MPIW(med)':>11}{'MAACE':>9}")
print("  "+"-"*66)
for r in [res_sel, res_all]:
    print(f"  {r['tag']:<18}{r['MAE']:>9.2f}{r['RMSE']:>9.2f}{r['PICP']:>8.1%}{r['MPIW']:>11.2f}{r['MAACE']:>8.2f}%")
print("="*72)
winner = min([res_sel,res_all], key=lambda r:r['MAE'])
print(f"  Winner: {winner['tag']} feature set (MAE €{winner['MAE']:.2f})")

# ---- PLOT 1: metric bars with values on top --------------------------------
metrics=["MAE","RMSE","MPIW","MAACE"]
figm=make_subplots(rows=1,cols=4,subplot_titles=[f"<b>{m}</b>" for m in metrics])
for j,m in enumerate(metrics,start=1):
    figm.add_trace(go.Bar(x=[res_sel['tag'],res_all['tag']],y=[res_sel[m],res_all[m]],
                    marker_color=["#1D9E75","#378ADD"],showlegend=False,
                    text=[f"{res_sel[m]:.2f}",f"{res_all[m]:.2f}"],
                    textposition="outside",textfont=dict(size=13)),row=1,col=j)
    figm.update_yaxes(range=[0,max(res_sel[m],res_all[m])*1.25],row=1,col=j)
figm.update_layout(title="<b>Part 19 — Best Model: Selected vs All Features</b>",
    paper_bgcolor="#f7f6f3",plot_bgcolor="#ffffff",height=440,width=1300)

# Save Plot 1 to HTML
path_figm = os.path.join(DOCS_DIR, "part19_best_model_metrics.html")
figm.write_html(path_figm)
print(f"  ✓ Saved Best Model metrics plot to {path_figm}")
display(figm)

# ---- PLOT 2: forecast for BOTH sets, DATES x-axis, WHOLE test set -----------
def forecast_fig(r, dates):
    x = pd.to_datetime(dates)
    fig=go.Figure()
    fig.add_trace(go.Scatter(x=x,y=np.maximum(r['lo'],r['hi']),mode="lines",
                  line=dict(width=0),showlegend=False,hoverinfo="skip"))
    fig.add_trace(go.Scatter(x=x,y=np.minimum(r['lo'],r['hi']),mode="lines",
                  line=dict(width=0),fill="tonexty",fillcolor="rgba(29,158,117,0.2)",
                  name="90% band",hoverinfo="skip"))
    fig.add_trace(go.Scatter(x=x,y=r['y'],mode="lines",
                  line=dict(color="#222",width=0.7),name="Actual"))
    fig.add_trace(go.Scatter(x=x,y=r['mu'],mode="lines",
                  line=dict(color="#1D9E75",width=1.0),name="Forecast"))
    fig.update_layout(
        title=f"<b>Forecast Accuracy (WHOLE test set) — {r['tag']} features "
              f"(MAE €{r['MAE']:.1f}, MAACE {r['MAACE']:.1f}%)</b>",
        paper_bgcolor="#f7f6f3",plot_bgcolor="#ffffff",height=430,width=1350,
        legend=dict(orientation="h",y=1.06,x=0.5,xanchor="center"))
    fig.update_yaxes(title_text="Price (€/MWh)")
    fig.update_xaxes(title_text="Date",tickformat="%b %d\n%Y",rangeslider=dict(visible=True))
    return fig

fig_sel = forecast_fig(res_sel, test_dates)
path_sel = os.path.join(DOCS_DIR, "part19_forecast_selected.html")
fig_sel.write_html(path_sel)
print(f"  ✓ Saved Selected Features forecast plot to {path_sel}")
display(fig_sel)

fig_all = forecast_fig(res_all, test_dates)
path_all = os.path.join(DOCS_DIR, "part19_forecast_all.html")
fig_all.write_html(path_all)
print(f"  ✓ Saved All Features forecast plot to {path_all}")
display(fig_all)

# ═══════════════════════════════════════════════════════════════════════════
#  B) MULTI-MODEL — full metrics on BOTH feature sets (asinh + conformal)
#     4 models × 2 feature sets = 8 rows. Battery uses the overall winner.
# ═══════════════════════════════════════════════════════════════════════════
CLIP=(-600.0,1000.0); order=["DDNN","EvDNN","VI-DDNN","StudentT"]

def train_all_models(feat, fset_tag):
    print(f"\nTraining 4 models on {fset_tag} feature set ({len(feat)} features)...")
    T=AsinhTransform(); T.fit(train["price"].values)
    sX=StandardScaler(); Xtr=sX.fit_transform(train[feat].values)
    Xval=sX.transform(val[feat].values); Xte=sX.transform(test[feat].values)
    ztr=T.forward(train["price"].values); zval=T.forward(val["price"].values)
    sz=StandardScaler(); ztr_s=sz.fit_transform(ztr.reshape(-1,1)).flatten()
    zval_s=sz.transform(zval.reshape(-1,1)).flatten()
    yte=test["price"].values
    def unscale(mu_s,sig_s): return sz.inverse_transform(mu_s.reshape(-1,1)).flatten(), np.abs(sig_s)*sz.scale_[0]
    def predict_z(m,X):
        out=m.predict(X); return unscale(out[0],out[1])
    builders = {
        "DDNN":    lambda: DDNN(len(feat),hidden=[128,64,32],lr=BP["DDNN"]["lr"],epochs=BP["DDNN"]["epochs"],batch_size=BP["DDNN"]["batch_size"]),
        "EvDNN":   lambda: EvDNN(len(feat),hidden=[128,64,32],lr=BP["EvDNN"]["lr"],epochs=BP["EvDNN"]["epochs"],lam=BP["EvDNN"].get("lam",0.02),batch_size=BP["EvDNN"]["batch_size"]),
        "VI-DDNN": lambda: VIDDNN(len(feat),hidden=[128,64],lr=BP["VI-DDNN"]["lr"],epochs=BP["VI-DDNN"]["epochs"],n_samples=15,prior_sigma=BP["VI-DDNN"].get("prior_sigma",1.0),batch_size=BP["VI-DDNN"]["batch_size"]),
        "StudentT":lambda: StudentTAdam(len(feat),hidden=(64,32),lr=0.01,epochs=STUDENTT_EPOCHS),
    }
    preds={}; metrics={}
    for name in order:
        print(f"  Training {name}...")
        m=builders[name]()
        if name=="StudentT": m.fit(Xtr,ztr_s)
        else:                m.fit(Xtr,ztr_s,Xval,zval_s,verbose=False)
        mu_z_val,sig_z_val = predict_z(m,Xval)
        mu_z_te ,sig_z_te  = predict_z(m,Xte)
        cal=MultiLevelConformal(); cal.fit(zval,mu_z_val,sig_z_val)
        mu_e=np.clip(T.inverse(mu_z_te),*CLIP)
        lo90,hi90=cal.band(mu_z_te,sig_z_te,0.9,T)
        lo90=np.clip(lo90,*CLIP); hi90=np.clip(hi90,*CLIP)
        lo=np.minimum(lo90,hi90); hi=np.maximum(lo90,hi90)
        mae=float(np.mean(np.abs(yte-mu_e))); rmse=float(np.sqrt(np.mean((yte-mu_e)**2)))
        picp=float(np.mean((yte>=lo)&(yte<=hi))); mpiw=float(np.median(hi-lo))
        maace=0.0
        for lv in np.arange(0.1,1.0,0.1):
            l,h=cal.band(mu_z_te,sig_z_te,lv,T); l,h=np.clip(l,*CLIP),np.clip(h,*CLIP)
            lo_,hi_=np.minimum(l,h),np.maximum(l,h)
            maace+=abs(np.mean((yte>=lo_)&(yte<=hi_))-lv)
        maace=maace/9*100
        sig_e=np.clip((hi-lo)/(2*1.2816),1.0,400.0)
        preds[name]={"mu":mu_e,"sigma":sig_e,"lo":lo,"hi":hi,"MAE":mae}
        metrics[name]={"MAE":mae,"RMSE":rmse,"PICP":picp,"MPIW":mpiw,"MAACE":maace}
    return preds, metrics

# Train on BOTH feature sets
preds_sel, metrics_sel = train_all_models(selected_features, f"Selected ({len(selected_features)})")
preds_all, metrics_all = train_all_models(all_features,      f"All ({len(all_features)})")

# ---- metric table: ALL 4 MODELS × BOTH FEATURE SETS (8 rows) ----------------
print("\n"+"="*84)
print("  ALL 4 MODELS × BOTH FEATURE SETS — TEST-SET METRICS (asinh + multi-level CP @90%)")
print("="*84)
print(f"  {'Feature set':<16}{'Model':<12}{'MAE':>9}{'RMSE':>9}{'PICP':>8}{'MPIW(med)':>12}{'MAACE':>9}")
print("  "+"-"*80)
for fset_tag, metrics in [(f"Selected ({len(selected_features)})", metrics_sel),
                         (f"All ({len(all_features)})", metrics_all)]:
    for nm in order:
        r=metrics[nm]
        print(f"  {fset_tag:<16}{nm:<12}{r['MAE']:>9.2f}{r['RMSE']:>9.2f}{r['PICP']:>8.1%}{r['MPIW']:>12.2f}{r['MAACE']:>8.2f}%")
    print("  "+"-"*80)
print("="*84)

# Overall best (across both feature sets) drives the battery + forecast plots
combined = {f"{fs}|{nm}":(preds, metrics[nm]) 
            for fs,preds,metrics in [("Selected",preds_sel,metrics_sel),("All",preds_all,metrics_all)]
            for nm in order}
best_key = min(combined, key=lambda k: combined[k][1]['MAE'])
best_fset = best_key.split('|')[0]
print(f"  Best overall: {best_key.replace('|',' / ')} (MAE €{combined[best_key][1]['MAE']:.2f})")

model_preds   = preds_all   if best_fset=="All" else preds_sel
model_metrics = metrics_all if best_fset=="All" else metrics_sel
yte = test["price"].values   

# ---- PLOT 3: 4-model metric bars (values on top) ---------------------------
figmm=make_subplots(rows=1,cols=5,subplot_titles=[f"<b>{m}</b>" for m in ["MAE","RMSE","PICP","MPIW","MAACE"]])
mcolor={"DDNN":"#378ADD","EvDNN":"#D85A30","VI-DDNN":"#7F77DD","StudentT":"#1D9E75"}
for j,m in enumerate(["MAE","RMSE","PICP","MPIW","MAACE"],start=1):
    vals=[model_metrics[nm][m]*100 if m=="PICP" else model_metrics[nm][m] for nm in order]
    figmm.add_trace(go.Bar(x=order,y=vals,marker_color=[mcolor[n] for n in order],showlegend=False,
                    text=[f"{v:.1f}" for v in vals],textposition="outside",textfont=dict(size=11)),row=1,col=j)
    figmm.update_yaxes(range=[0,max(vals)*1.25],row=1,col=j)
figmm.update_layout(title="<b>All 4 Models — Test-Set Metrics (values on bars)</b>",
    paper_bgcolor="#f7f6f3",plot_bgcolor="#ffffff",height=420,width=1500)

path_figmm = os.path.join(DOCS_DIR, "part19_all_models_metrics.html")
figmm.write_html(path_figmm)
print(f"  ✓ Saved All Models metrics plot to {path_figmm}")
display(figmm)

# ---- Battery valuation across the 4 models ---------------------------------
def battery_daily(mu,sigma,price,xi=0.9,k=0.3):
    nd=len(mu)//24; A=0.0; B=0.0; days=0
    for d in range(nd):
        sl=slice(d*24,(d+1)*24); md_,sd_,td_=mu[sl],sigma[sl],price[sl]
        if len(md_)<24: continue
        days+=1; b=int(np.argmin(md_)); s=int(np.argmax(md_))
        if s<=b: b,s=min(b,s),max(b,s)
        real=td_[s]*xi-td_[b]; A+=real
        if (md_[s]-md_[b])>k*np.sqrt(sd_[b]**2+sd_[s]**2): B+=real
    dd=max(days,1); return A/dd, B/dd
perf={}
print("\n  Multi-Model Battery Valuation (Test Set):")
print("  "+"-"*55)
for name in order:
    pr=model_preds[name]; a,b=battery_daily(pr["mu"],pr["sigma"],yte)
    perf[name]={"A":a,"B":b,"extra":b-a}
    print(f"  {name:<10}: Naive €{a:6.2f}/day | Smart €{b:6.2f}/day | Extra €{b-a:6.2f}/day")

# ---- PLOT 5: Multi-Model Battery Profit & Uncertainty Valuation ------------
models=list(perf.keys()); A=[perf[m]["A"] for m in models]; B=[perf[m]["B"] for m in models]
fleet=10000; annual=[(perf[m]["extra"])*365*fleet/1e6 for m in models]
figv=make_subplots(rows=1,cols=2,
    subplot_titles=("<b>1. Daily Profit per MWh (Naive vs Smart)</b>",
                    "<b>2. Annual Extra Value across 2030 Fleet (€M/yr)</b>"),horizontal_spacing=0.15)
figv.add_trace(go.Bar(x=models,y=A,name="Naive (Always Bet)",marker_color="#888780",
              text=[f"€{v:.0f}" for v in A],textposition="outside"),row=1,col=1)
figv.add_trace(go.Bar(x=models,y=B,name="Smart (Uncertainty-Aware)",marker_color="#1D9E75",
              text=[f"€{v:.0f}" for v in B],textposition="outside"),row=1,col=1)
figv.add_trace(go.Bar(x=models,y=annual,marker_color="#D85A30",showlegend=False,
              text=[f"€{v:.1f}M" for v in annual],textposition="outside"),row=1,col=2)
figv.update_layout(barmode="group",
    title="<b>Multi-Model Battery Profit & Uncertainty Valuation (Test Set) — asinh scaling</b>",
    paper_bgcolor="#f7f6f3",plot_bgcolor="#ffffff",height=560,width=1350,
    legend=dict(orientation="h",yanchor="bottom",y=1.12,xanchor="center",x=0.5))
figv.update_yaxes(title_text="Daily Profit (€/MWh/day)",row=1,col=1)
figv.update_yaxes(title_text="Extra Value (€M/year)",row=1,col=2)

path_figv = os.path.join(DOCS_DIR, "part19_battery_valuation.html")
figv.write_html(path_figv)
print(f"  ✓ Saved Battery Valuation plot to {path_figv}")
display(figv)

print("\n✓ Part 19 complete — saved all interactive Plotly plots to HTML files successfully!")

Selected feature set: 15 features
All-candidate set:    35 features

Loaded per-model Optuna hyperparameters from Part 8:
  {'DDNN': {'lr': 0.0008274803923368282, 'epochs': 100, 'batch_size': 128}, 'EvDNN': {'lr': 0.002006906273275508, 'epochs': 250, 'batch_size': 128, 'lam': 0.003851460995733125}, 'VI-DDNN': {'lr': 0.001696657436973614, 'epochs': 300, 'batch_size': 256, 'prior_sigma': 1.9114387304106248}}

  Training best model on Selected (15) feature set...
  Training best model on All (35) feature set...

  FINAL MODEL — BOTH FEATURE SETS (Student-t + asinh + multi-level CP)
  Feature set             MAE     RMSE    PICP  MPIW(med)    MAACE
  ------------------------------------------------------------------
  Selected (15)         15.05    22.36   91.4%      61.51    0.34%
  All (35)               7.86    14.79   92.1%      31.67    3.77%
  Winner: All (35) feature set (MAE €7.86)
  ✓ Saved Best Model metrics plot to part19_best_model_metrics.html


  ✓ Saved Selected Features forecast plot to part19_forecast_selected.html


  ✓ Saved All Features forecast plot to part19_forecast_all.html



Training 4 models on Selected (15) feature set (15 features)...
  Training DDNN...
  Training EvDNN...
  Training VI-DDNN...
  Training StudentT...

Training 4 models on All (35) feature set (35 features)...
  Training DDNN...
  Training EvDNN...
  Training VI-DDNN...
  Training StudentT...

  ALL 4 MODELS × BOTH FEATURE SETS — TEST-SET METRICS (asinh + multi-level CP @90%)
  Feature set     Model             MAE     RMSE    PICP   MPIW(med)    MAACE
  --------------------------------------------------------------------------------
  Selected (15)   DDNN            16.28    23.61   89.3%       60.34    1.39%
  Selected (15)   EvDNN           14.62    22.46   91.3%       61.34    1.11%
  Selected (15)   VI-DDNN         17.01    24.60   89.4%       61.14    0.73%
  Selected (15)   StudentT        15.44    22.39   90.3%       60.92    1.01%
  --------------------------------------------------------------------------------
  All (35)        DDNN            10.64    17.68   92.7%       40.


  Multi-Model Battery Valuation (Test Set):
  -------------------------------------------------------
  DDNN      : Naive € 70.47/day | Smart € 83.14/day | Extra € 12.67/day
  EvDNN     : Naive € 74.40/day | Smart € 88.39/day | Extra € 13.99/day
  VI-DDNN   : Naive € 71.29/day | Smart € 83.53/day | Extra € 12.24/day
  StudentT  : Naive € 74.81/day | Smart € 86.15/day | Extra € 11.34/day
  ✓ Saved Battery Valuation plot to part19_battery_valuation.html



✓ Part 19 complete — saved all interactive Plotly plots to HTML files successfully!


In [60]:
# ---- PLOT 4: forecast accuracy of ALL 4 MODELS (stacked, dates, WebGL optimized) ----
xd = pd.to_datetime(test_dates)
fig4 = make_subplots(
    rows=4, cols=1, vertical_spacing=0.05,
    subplot_titles=[f"<b>{nm} — MAE €{model_preds[nm]['MAE']:.1f}</b>" for nm in order]
)

for i, nm in enumerate(order, start=1):
    pr = model_preds[nm]
    sl = (i == 1)
    
    # Use Scattergl (WebGL) instead of Scatter to handle 8,760+ hourly steps smoothly
    fig4.add_trace(go.Scattergl(
        x=xd, y=pr["hi"], mode="lines", line=dict(width=0),
        showlegend=False, hoverinfo="skip"
    ), row=i, col=1)
    
    fig4.add_trace(go.Scattergl(
        x=xd, y=pr["lo"], mode="lines", line=dict(width=0),
        fill="tonexty", fillcolor="rgba(120,120,120,0.15)",
        name="90% band", showlegend=sl, hoverinfo="skip"
    ), row=i, col=1)
    
    fig4.add_trace(go.Scattergl(
        x=xd, y=yte, mode="lines", line=dict(color="#222", width=0.6),
        name="Actual", showlegend=sl
    ), row=i, col=1)
    
    fig4.add_trace(go.Scattergl(
        x=xd, y=pr["mu"], mode="lines", line=dict(color=mcolor[nm], width=1.0),
        name="Forecast", showlegend=sl
    ), row=i, col=1)
    
    fig4.update_yaxes(title_text="€/MWh", row=i, col=1)

fig4.update_xaxes(title_text="Date", tickformat="%b %d\n%Y", row=4, col=1)
fig4.update_layout(
    title="<b>Forecast Accuracy — All 4 Models (whole test set, with all features, asinh scaling)</b>",
    paper_bgcolor="#f7f6f3", plot_bgcolor="#ffffff", height=1200, width=1350,
    legend=dict(orientation="h", y=1.05, x=0.5, xanchor="center")
)

# Save Plot 4 to HTML
path_fig4 = os.path.join(DOCS_DIR, "part19_all_models_forecasts.html")
fig4.write_html(path_fig4)
print(f"  ✓ Saved All Models Forecasts plot to {path_fig4}")

# Render natively in Jupyter Notebook
display(fig4)

  ✓ Saved All Models Forecasts plot to part19_all_models_forecasts.html
